# Basic Item Extraction with DuckDB SQL

This notebook extracts variables from **base items** (not prebuilt ratios) using DuckDB SQL:
- `CED` from `proper_vn_results` (`S2_CARBON_DISC`)
- `ISO` from `proper_vn_results` (`S2_ISO14001`)
- `BOARDSIZE`, `WOMAN` from `governance_results` (`GOV_DIRECTORY`)
- `AUD` from `bctc_audit_results`
- `AGE` from `company_history_summary`
- `SIZE`, `ROA`, `LOA` from raw `financial_statements` item codes

In [26]:
from pathlib import Path
import duckdb
import pandas as pd

DB_PATH = Path('db.db')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH.resolve()}')

con = duckdb.connect(str(DB_PATH), read_only=True)
print('Connected to', DB_PATH.resolve())

Connected to /media/nvme0n1/dev/annual_report/db.db


In [27]:
# Configure scope
TICKERS = []  # e.g. ['VNM', 'AAA']; keep [] to include all
START_YEAR = 2015
END_YEAR = 2025

# Base item codes used to derive financial variables from raw statement rows
# You can adjust these if your target industry uses different codes.
ITEM_CODES = {
    'total_assets': '12700.0',
    'profit_after_tax': '23003.0',
    'pretax_profit': '23800.0',
    'short_term_debt': '13110.0',
    'long_term_debt': '13340.0',
    'current_assets': '11000.0',
    'long_term_assets': '12000.0',
    'short_term_liabilities': '13100.0',
    'short_term_financial_investments': '11200.0',
    'net_revenue': '21001.0',
    'total_liabilities': '13000.0',
    'owners_equity': '14000.0',
    'cash_and_equivalents': '11100.0',
    'tangible_fixed_assets': '12210.0',
    'depreciation_expense': '22230.0',
    'net_cf_operating': '32000.0',
    'taxes_payable_state': '13140.0',
    'current_corporate_income_tax_expense': '22051.0',
}

print('Tickers:', TICKERS if TICKERS else 'ALL')
print('Year range:', START_YEAR, 'to', END_YEAR)
print('Item codes:', ITEM_CODES)

Tickers: ALL
Year range: 2015 to 2025
Item codes: {'total_assets': '12700.0', 'profit_after_tax': '23003.0', 'pretax_profit': '23800.0', 'short_term_debt': '13110.0', 'long_term_debt': '13340.0', 'current_assets': '11000.0', 'long_term_assets': '12000.0', 'short_term_liabilities': '13100.0', 'short_term_financial_investments': '11200.0', 'net_revenue': '21001.0', 'total_liabilities': '13000.0', 'owners_equity': '14000.0', 'cash_and_equivalents': '11100.0', 'tangible_fixed_assets': '12210.0', 'depreciation_expense': '22230.0', 'net_cf_operating': '32000.0', 'taxes_payable_state': '13140.0', 'current_corporate_income_tax_expense': '22051.0'}


In [ ]:
# Optional: inspect candidate item codes by keyword before running final extraction
keyword_sql = '''
SELECT
  item_code,
  MAX(item_en_name) AS item_en_name,
  MAX(item_vn_name) AS item_vn_name,
  COUNT(*) AS occurrences
FROM financial_models
WHERE
  LOWER(COALESCE(item_en_name, '')) LIKE '%total assets%'
  OR LOWER(COALESCE(item_en_name, '')) LIKE '%profit after tax%'
  OR LOWER(COALESCE(item_en_name, '')) LIKE '%profit before tax%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%tài sản ngắn hạn%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%tài sản dài hạn%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%nợ phải trả%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%nợ ngắn hạn%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%vay và nợ ngắn hạn%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%đầu tư tài chính ngắn hạn%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%vốn chủ sở hữu%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%tiền và các khoản tương đương tiền%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%tài sản cố định hữu hình%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%khấu hao tài sản cố định%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%lưu chuyển tiền thuần từ hoạt động kinh doanh%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%thuế và các khoản phải nộp nhà nước%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%thuế tndn hiện hành%'
  OR LOWER(COALESCE(item_vn_name, '')) LIKE '%doanh thu thuần%'
GROUP BY item_code
ORDER BY occurrences DESC, item_code
LIMIT 80
'''
pd.set_option('display.max_colwidth', 120)
con.execute(keyword_sql).fetchdf()

,item_code,item_en_name,item_vn_name,occurrences
0,23000.0,Net Profit After Tax of Parent Company,Lợi nhuận sau thuế của Công ty mẹ,12
1,12700.0,TOTAL ASSETS,TỔNG CỘNG TÀI SẢN,9
2,21001.0,Net Sales,Doanh thu thuần,9
3,13340.0,Loans and Long term Liabilities,Vay và nợ dài hạn,8
4,23003.0,Net Profit After Tax,Lợi nhuận sau thuế thu nhập doanh nghiệp,8
5,11000.0,Current Assets,Tài sản ngắn hạn,7
6,13110.0,Short term Loan and Liabilities,Vay và nợ ngắn hạn,7
7,14200.0,Undistributed Profit after Tax,Lợi nhuận sau thuế chưa phân phối,7
8,11500.0,Other Current Assets,Tài sản ngắn hạn khác,6
9,11580.0,Other current assets,Tài sản ngắn hạn khác,6


In [28]:
def _in_clause(values):
    if not values:
        return ''
    cleaned = [v.strip().upper() for v in values if v and v.strip()]
    if not cleaned:
        return ''
    literal = ', '.join(["'" + v.replace("'", "''") + "'" for v in cleaned])
    return f'AND b.ticker IN ({literal})'

ticker_filter_sql = _in_clause(TICKERS)

CED_BASE_CODES = [
    'CC1', 'CC2',
    'GHG1', 'GHG2', 'GHG3', 'GHG4', 'GHG5', 'GHG6', 'GHG7',
    'EC1', 'EC2', 'EC3',
    'RC1', 'RC2', 'RC3', 'RC4',
    'ACC1', 'ACC2',
]

CED_CODES = []
for code in CED_BASE_CODES:
    CED_CODES.extend([code, f'{code}_alt', f'{code}_alt_two'])

ced_select_sql = ',\n'.join([
    f"    MAX(CASE WHEN category_code = '{c}' THEN CAST(is_valid AS INTEGER) END) AS {c.lower()}"
    for c in CED_CODES
])
ced_category_sql = ', '.join([f"'{c}'" for c in CED_CODES])
ced_projection_sql = ',\n'.join([f"  i.{c.lower()}" for c in CED_CODES])

audit_table_exists = bool(
    con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'main' AND table_name = 'bctc_audit_results'
        """
    ).fetchone()[0]
    > 0
)
if audit_table_exists:
    audit_vars_sql = f'''
audit_vars AS (
  SELECT
    ticker,
    year,
    MAX(CAST(found AS INTEGER)) AS aud
  FROM bctc_audit_results
  WHERE year BETWEEN {START_YEAR} AND {END_YEAR}
  GROUP BY ticker, year
) ,
'''
else:
    audit_vars_sql = '''
audit_vars AS (
  SELECT
    ticker,
    year,
    CAST(NULL AS INTEGER) AS aud
  FROM (
    SELECT DISTINCT ticker, year FROM financial_vars
    UNION
    SELECT DISTINCT ticker, year FROM proper_flags
    UNION
    SELECT DISTINCT ticker, year FROM inference_ced
    UNION
    SELECT DISTINCT ticker, year FROM board_vars
  ) audit_base
) ,
'''

sql = f'''
WITH fs_base AS (
  SELECT
    code AS ticker,
    TRY_CAST(SUBSTR(fiscal_date, 1, 4) AS INTEGER) AS year,
    item_code,
    numeric_value,
    fiscal_date,
    ROW_NUMBER() OVER (
      PARTITION BY code, TRY_CAST(SUBSTR(fiscal_date, 1, 4) AS INTEGER), item_code
      ORDER BY fiscal_date DESC
    ) AS rn
  FROM financial_statements
  WHERE TRY_CAST(SUBSTR(fiscal_date, 1, 4) AS INTEGER) BETWEEN {START_YEAR} AND {END_YEAR}
) ,
fs_latest AS (
  SELECT ticker, year, item_code, numeric_value
  FROM fs_base
  WHERE rn = 1
) ,
financial_inputs AS (
  SELECT
    ticker,
    year,
    MAX(CASE WHEN item_code = '{ITEM_CODES['total_assets']}' THEN numeric_value END) AS total_assets,
    MAX(CASE WHEN item_code = '{ITEM_CODES['profit_after_tax']}' THEN numeric_value END) AS profit_after_tax,
    MAX(CASE WHEN item_code = '{ITEM_CODES['pretax_profit']}' THEN numeric_value END) AS pretax_profit,
    MAX(CASE WHEN item_code = '{ITEM_CODES['short_term_debt']}' THEN numeric_value END) AS short_term_debt,
    MAX(CASE WHEN item_code = '{ITEM_CODES['long_term_debt']}' THEN numeric_value END) AS long_term_debt,
    MAX(CASE WHEN item_code = '{ITEM_CODES['current_assets']}' THEN numeric_value END) AS current_assets,
    MAX(CASE WHEN item_code = '{ITEM_CODES['long_term_assets']}' THEN numeric_value END) AS long_term_assets,
    MAX(CASE WHEN item_code = '{ITEM_CODES['short_term_liabilities']}' THEN numeric_value END) AS short_term_liabilities,
    MAX(CASE WHEN item_code = '{ITEM_CODES['short_term_financial_investments']}' THEN numeric_value END) AS short_term_financial_investments,
    MAX(CASE WHEN item_code = '{ITEM_CODES['net_revenue']}' THEN numeric_value END) AS net_revenue,
    MAX(CASE WHEN item_code = '{ITEM_CODES['total_liabilities']}' THEN numeric_value END) AS total_liabilities,
    MAX(CASE WHEN item_code = '{ITEM_CODES['owners_equity']}' THEN numeric_value END) AS owners_equity,
    MAX(CASE WHEN item_code = '{ITEM_CODES['cash_and_equivalents']}' THEN numeric_value END) AS cash_and_equivalents,
    MAX(CASE WHEN item_code = '{ITEM_CODES['tangible_fixed_assets']}' THEN numeric_value END) AS tangible_fixed_assets,
    MAX(CASE WHEN item_code = '{ITEM_CODES['depreciation_expense']}' THEN numeric_value END) AS depreciation_expense,
    MAX(CASE WHEN item_code = '{ITEM_CODES['net_cf_operating']}' THEN numeric_value END) AS net_cf_operating,
    MAX(CASE WHEN item_code = '{ITEM_CODES['taxes_payable_state']}' THEN numeric_value END) AS taxes_payable_state,
    MAX(CASE WHEN item_code = '{ITEM_CODES['current_corporate_income_tax_expense']}' THEN numeric_value END) AS current_corporate_income_tax_expense
  FROM fs_latest
  GROUP BY ticker, year
) ,
financial_vars AS (
  SELECT
    ticker,
    year,
    CASE WHEN total_assets > 0 THEN LN(total_assets) END AS size,
    CASE WHEN total_assets > 0 THEN profit_after_tax / total_assets END AS roa,
    CASE WHEN total_assets > 0 THEN (COALESCE(short_term_debt, 0) + COALESCE(long_term_debt, 0)) / total_assets END AS loa
  FROM financial_inputs
) ,
proper_flags AS (
  SELECT
    ticker,
    year,
    MAX(CASE WHEN indicator_code = 'S2_CARBON_DISC' THEN CAST(is_present AS INTEGER) END) AS ced,
    MAX(CASE WHEN indicator_code = 'S2_ISO14001' THEN CAST(is_present AS INTEGER) END) AS iso
  FROM proper_vn_results
  WHERE year BETWEEN {START_YEAR} AND {END_YEAR}
  GROUP BY ticker, year
) ,
inference_ced AS (
  SELECT
    ticker,
    year,
{ced_select_sql}
  FROM inference_results
  WHERE year BETWEEN {START_YEAR} AND {END_YEAR}
    AND category_code IN ({ced_category_sql})
  GROUP BY ticker, year
) ,
board_vars AS (
  SELECT
    ticker,
    year,
    MAX(TRY_CAST(json_extract_string(value_json, '$.total_members') AS INTEGER)) AS boardsize,
    MAX(TRY_CAST(json_extract_string(value_json, '$.women_count') AS INTEGER)) AS woman
  FROM governance_results
  WHERE item_code = 'GOV_DIRECTORY'
    AND year BETWEEN {START_YEAR} AND {END_YEAR}
  GROUP BY ticker, year
) ,
{audit_vars_sql}
age_vars AS (
  SELECT
    y.ticker,
    y.year,
    h.first_event_year,
    CASE
      WHEN h.first_event_year IS NOT NULL THEN y.year - h.first_event_year + 1
      ELSE NULL
    END AS age
  FROM (
    SELECT DISTINCT ticker, year FROM financial_vars
    UNION
    SELECT DISTINCT ticker, year FROM proper_flags
    UNION
    SELECT DISTINCT ticker, year FROM inference_ced
    UNION
    SELECT DISTINCT ticker, year FROM board_vars
    UNION
    SELECT DISTINCT ticker, year FROM audit_vars
  ) y
  LEFT JOIN company_history_summary h
    ON h.ticker = y.ticker
) ,
base AS (
  SELECT DISTINCT ticker, year
  FROM age_vars
)
SELECT
  b.ticker,
  b.year,
  p.ced,
{ced_projection_sql},
  f.size,
  bd.boardsize,
  a.age,
  f.roa,
  f.loa,
  p.iso,
  bd.woman,
  au.aud,
  fi.total_assets AS raw_total_assets,
  fi.current_assets AS raw_current_assets,
  fi.long_term_assets AS raw_long_term_assets,
  fi.total_liabilities AS raw_total_liabilities,
  fi.short_term_liabilities AS raw_short_term_liabilities,
  fi.short_term_debt AS raw_short_term_debt,
  fi.short_term_financial_investments AS raw_short_term_financial_investments,
  fi.owners_equity AS raw_owners_equity,
  fi.cash_and_equivalents AS raw_cash_and_equivalents,
  fi.tangible_fixed_assets AS raw_tangible_fixed_assets,
  fi.depreciation_expense AS raw_depreciation_expense,
  fi.net_cf_operating AS raw_net_cf_operating,
  fi.taxes_payable_state AS raw_taxes_payable_state,
  fi.current_corporate_income_tax_expense AS raw_current_corporate_income_tax_expense,
  fi.net_revenue AS raw_net_revenue,
  fi.profit_after_tax AS raw_profit_after_tax,
  fi.pretax_profit AS raw_pretax_profit,
  fi.long_term_debt AS raw_long_term_debt,
  a.first_event_year AS raw_first_event_year
FROM base b
LEFT JOIN financial_vars f ON f.ticker = b.ticker AND f.year = b.year
LEFT JOIN financial_inputs fi ON fi.ticker = b.ticker AND fi.year = b.year
LEFT JOIN proper_flags p ON p.ticker = b.ticker AND p.year = b.year
LEFT JOIN inference_ced i ON i.ticker = b.ticker AND i.year = b.year
LEFT JOIN board_vars bd ON bd.ticker = b.ticker AND bd.year = b.year
LEFT JOIN age_vars a ON a.ticker = b.ticker AND a.year = b.year
LEFT JOIN audit_vars au ON au.ticker = b.ticker AND au.year = b.year
WHERE b.year BETWEEN {START_YEAR} AND {END_YEAR}
{ticker_filter_sql}
ORDER BY b.ticker, b.year
'''

df_all = con.execute(sql).fetchdf()

ced_columns = [c.lower() for c in CED_CODES]
metric_columns = [
    'ticker', 'year', 'ced', *ced_columns, 'size', 'boardsize', 'age', 'roa', 'loa', 'iso', 'woman', 'aud'
]
raw_columns = [
    'ticker', 'year',
    'raw_total_assets', 'raw_current_assets', 'raw_long_term_assets',
    'raw_total_liabilities', 'raw_short_term_liabilities', 'raw_short_term_debt',
    'raw_short_term_financial_investments',
    'raw_owners_equity', 'raw_cash_and_equivalents', 'raw_tangible_fixed_assets',
    'raw_depreciation_expense', 'raw_net_cf_operating',
    'raw_taxes_payable_state', 'raw_current_corporate_income_tax_expense',
    'raw_net_revenue', 'raw_profit_after_tax', 'raw_pretax_profit',
    'raw_long_term_debt',
    'raw_first_event_year',
    'ced', *ced_columns, 'iso', 'boardsize', 'woman', 'aud'
]

df = df_all[metric_columns].copy()
df_raw_inputs = df_all[raw_columns].copy()

print('Derived metrics df shape:', df.shape)
print('Raw inputs df shape:', df_raw_inputs.shape)
print('Total CED columns:', len(ced_columns))
df_raw_inputs.head(50)

Derived metrics df shape: (949, 65)
Raw inputs df shape: (949, 80)
Total CED columns: 54


,ticker,year,raw_total_assets,raw_current_assets,raw_long_term_assets,raw_total_liabilities,raw_short_term_liabilities,raw_short_term_debt,raw_short_term_financial_investments,raw_owners_equity,...,acc1,acc1_alt,acc1_alt_two,acc2,acc2_alt,acc2_alt_two,iso,boardsize,woman,aud
0,AAA,2015,1.954765e+12,1.071561e+12,8.832037e+11,1.135279e+12,6.670792e+11,4.387699e+11,0.000000e+00,8.194853e+11,...,0,0,1,0,0,0,0,5,1,<NA>
1,AAA,2016,3.077616e+12,1.361646e+12,1.715970e+12,2.122864e+12,1.140285e+12,8.007948e+11,0.000000e+00,9.547521e+11,...,0,0,1,0,0,0,0,5,1,<NA>
2,AAA,2017,4.576157e+12,2.142717e+12,2.433441e+12,2.951187e+12,1.990804e+12,1.417686e+12,5.000000e+10,1.624970e+12,...,0,1,1,0,0,0,1,5,1,<NA>
3,AAA,2018,7.529167e+12,3.989369e+12,3.539797e+12,4.548917e+12,3.206103e+12,2.492407e+12,7.209065e+11,2.980250e+12,...,0,1,1,0,1,1,1,5,1,<NA>
4,AAA,2019,7.987454e+12,4.971364e+12,3.016091e+12,4.732216e+12,3.236646e+12,2.400087e+12,1.251822e+12,3.255238e+12,...,0,1,1,0,1,0,1,5,1,<NA>
5,AAA,2020,8.569414e+12,4.496051e+12,4.073364e+12,4.545452e+12,3.772835e+12,2.943359e+12,7.586000e+11,4.023962e+12,...,0,1,1,0,1,0,1,5,1,<NA>
6,AAA,2021,1.000953e+13,5.354611e+12,4.654916e+12,4.555145e+12,3.282339e+12,2.183181e+12,4.361560e+11,5.454382e+12,...,0,1,1,0,1,0,1,7,0,<NA>
7,AAA,2022,1.079583e+13,5.658759e+12,5.137073e+12,4.624647e+12,3.206483e+12,1.887821e+12,4.486560e+11,6.171185e+12,...,0,1,1,0,1,0,1,7,3,<NA>
8,AAA,2023,1.158345e+13,5.681580e+12,5.901865e+12,5.619575e+12,3.737041e+12,2.625493e+12,1.079610e+12,5.963871e+12,...,0,1,1,0,1,0,1,5,3,<NA>
9,AAA,2024,1.376822e+13,6.426369e+12,7.341846e+12,7.531942e+12,4.132594e+12,2.554855e+12,7.186394e+11,6.236274e+12,...,0,1,1,0,1,0,1,5,2,<NA>


In [30]:
# Summary checks
print('Rows:', len(df))
print('Tickers:', df['ticker'].nunique() if not df.empty else 0)
print('Year min/max:', (df['year'].min(), df['year'].max()) if not df.empty else (None, None))

summary_cols = [c for c in df.columns if c not in ['ticker', 'year']]
missing = df[summary_cols].isna().mean().sort_values(ascending=False)
missing

Rows: 949
Tickers: 87
Year min/max: (np.int32(2015), np.int32(2025))


aud            0.969442
boardsize      0.024236
ced            0.023182
cc1_alt_two    0.023182
cc2            0.023182
                 ...   
woman          0.023182
roa            0.000000
age            0.000000
size           0.000000
loa            0.000000
Length: 63, dtype: float64

In [31]:
# Correlation table for all numeric variables using stats_duck, with SQL fallback
numeric_vars = [
    c for c in df.columns
    if c not in ['ticker', 'year'] and pd.api.types.is_numeric_dtype(df[c])
]

df_corr_input = df[numeric_vars].copy()

use_stats_duck = True
stats_duck_error = None

try:
    # Use a writable in-memory connection because corr_matrix expects a real table name.
    con_stats = duckdb.connect()
    try:
        con_stats.execute("INSTALL stats_duck")
    except Exception:
        con_stats.execute("INSTALL stats_duck FROM community")
    con_stats.execute("LOAD stats_duck")
    con_stats.register('df_corr_input', df_corr_input)
    con_stats.execute("CREATE OR REPLACE TABLE corr_input AS SELECT * FROM df_corr_input")

    var_list_sql = '[' + ', '.join("'" + c.replace("'", "''") + "'" for c in numeric_vars) + ']'
    corr_sql = f"""
    SELECT *
    FROM corr_matrix('corr_input', method := 'pearson', variables := {var_list_sql})
    """
    df_corr_long = con_stats.execute(corr_sql).fetchdf()
except Exception as e:
    use_stats_duck = False
    stats_duck_error = str(e)

if not use_stats_duck:
    # Fallback: build pairwise Pearson correlation table with DuckDB built-ins
    con.register('df_corr_input', df_corr_input)
    pairs = [
        (numeric_vars[i], numeric_vars[j])
        for i in range(len(numeric_vars))
        for j in range(i, len(numeric_vars))
    ]
    if not pairs:
        raise ValueError('No numeric variables found for correlation.')

    union_parts = []
    for v1, v2 in pairs:
        q1 = v1.replace('"', '""')
        q2 = v2.replace('"', '""')
        union_parts.append(
            f"SELECT '{v1}' AS variable_1, '{v2}' AS variable_2, corr(\"{q1}\", \"{q2}\") AS correlation FROM df_corr_input"
        )
    corr_sql = "\nUNION ALL\n".join(union_parts)
    df_corr_long = con.execute(corr_sql).fetchdf()

cols = list(df_corr_long.columns)
corr_col = next((c for c in cols if c.lower() in {'correlation', 'corr', 'r', 'value', 'coef'}), cols[-1])
var_cols = [c for c in cols if c != corr_col][:2]

if len(var_cols) < 2:
    raise ValueError('Unexpected correlation output schema; cannot identify variable columns.')

df_corr_table = df_corr_long.pivot(index=var_cols[0], columns=var_cols[1], values=corr_col)
df_corr_table = df_corr_table.combine_first(df_corr_table.T)
for v in df_corr_table.index.intersection(df_corr_table.columns):
    df_corr_table.loc[v, v] = 1.0

print('method:', 'stats_duck.corr_matrix' if use_stats_duck else 'duckdb_sql_fallback')
if stats_duck_error:
    print('stats_duck unavailable:', stats_duck_error.splitlines()[0])
print('corr rows:', len(df_corr_long))
print('corr columns:', list(df_corr_long.columns))

df_corr_long.head(20), df_corr_table

method: stats_duck.corr_matrix
corr rows: 3969
corr columns: ['row_var', 'col_var', 'coef', 'p_value', 'n']


(   row_var       col_var      coef       p_value    n
 0      ced           ced  1.000000  0.000000e+00  927
 1      ced           cc1  0.164729  4.578465e-07  927
 2      ced       cc1_alt  0.196376  1.642426e-09  927
 3      ced   cc1_alt_two  0.227172  2.577494e-12  927
 4      ced           cc2  0.015403  6.395180e-01  927
 5      ced       cc2_alt  0.332489  0.000000e+00  927
 6      ced   cc2_alt_two  0.301545  0.000000e+00  927
 7      ced          ghg1  0.205655  2.602811e-10  927
 8      ced      ghg1_alt  0.285756  0.000000e+00  927
 9      ced  ghg1_alt_two  0.377575  0.000000e+00  927
 10     ced          ghg2  0.135396  3.536472e-05  927
 11     ced      ghg2_alt  0.228552  1.885825e-12  927
 12     ced  ghg2_alt_two  0.080574  1.413164e-02  927
 13     ced          ghg3  0.061672  6.052459e-02  927
 14     ced      ghg3_alt  0.596538  0.000000e+00  927
 15     ced  ghg3_alt_two  0.581070  0.000000e+00  927
 16     ced          ghg4  0.157647  1.410358e-06  927
 17     ce

In [32]:
# Regression-oriented diagnostics from the correlation table
import numpy as np

# Planned model (from your equation image): CED ~ controls
y_var = 'ced'
candidate_x = ['size', 'boardsize', 'age', 'roa', 'loa', 'iso', 'woman', 'aud']
x_vars = [c for c in candidate_x if c in df.columns]

if 'df_corr_table' in globals() and not df_corr_table.empty:
    corr_source = df_corr_table.copy()
else:
    corr_source = df[x_vars + [y_var]].corr(numeric_only=True)

# 1) Pairwise correlation among X variables
x_corr = corr_source.reindex(index=x_vars, columns=x_vars)
x_corr_abs = x_corr.abs().where(~np.eye(len(x_vars), dtype=bool))

high_corr_pairs = []
for i, c1 in enumerate(x_vars):
    for c2 in x_vars[i+1:]:
        v = x_corr.loc[c1, c2]
        if pd.notna(v) and abs(v) >= 0.7:
            high_corr_pairs.append((c1, c2, float(v)))

# 2) Correlation of Y with each X
yx_corr = corr_source.reindex(index=[y_var], columns=x_vars).T.rename(columns={y_var: 'corr_with_ced'})

# 3) Simple VIF from pandas for additional multicollinearity signal
vif_rows = []
x_num = df[x_vars].copy()
x_num = x_num.apply(pd.to_numeric, errors='coerce').dropna()
if len(x_num) > 20 and len(x_vars) >= 2:
    for target in x_vars:
        others = [c for c in x_vars if c != target]
        if not others:
            continue
        y_t = x_num[target].astype(float).to_numpy()
        X_o = x_num[others].astype(float).copy()
        X_o['_const'] = 1.0
        X_mat = X_o.to_numpy(dtype=float)
        beta, *_ = np.linalg.lstsq(X_mat, y_t, rcond=None)
        y_hat = X_mat @ beta
        ss_res = np.sum((y_t - y_hat) ** 2)
        ss_tot = np.sum((y_t - y_t.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        vif = 1 / (1 - r2) if pd.notna(r2) and r2 < 1 else np.inf
        vif_rows.append((target, float(vif)))

vif_df = pd.DataFrame(vif_rows, columns=['variable', 'vif']).sort_values('vif', ascending=False) if vif_rows else pd.DataFrame(columns=['variable', 'vif'])

print('Planned X variables found:', x_vars)
print('High-correlation X pairs (|r| >= 0.7):', len(high_corr_pairs))
if high_corr_pairs:
    for p in high_corr_pairs:
        print(f'  {p[0]} vs {p[1]}: r={p[2]:.3f}')
else:
    print('  None')

print('\nSuggestion for regression-ready table:')
print("- Keep only: ['ticker', 'year', 'ced'] + planned X variables")
print('- Do NOT include CC/GHG/EC/RC/ACC columns when ced is dependent variable (concept overlap).')
print('- If any VIF > 10, consider dropping or transforming that variable.')

regression_table = df[['ticker', 'year', y_var] + x_vars].copy()
regression_table.head(20), x_corr, yx_corr, vif_df

Planned X variables found: ['size', 'boardsize', 'age', 'roa', 'loa', 'iso', 'woman', 'aud']
High-correlation X pairs (|r| >= 0.7): 0
  None

Suggestion for regression-ready table:
- Keep only: ['ticker', 'year', 'ced'] + planned X variables
- Do NOT include CC/GHG/EC/RC/ACC columns when ced is dependent variable (concept overlap).
- If any VIF > 10, consider dropping or transforming that variable.


(   ticker  year  ced       size  boardsize  age       roa       loa  iso  \
 0     AAA  2015    0  28.301291          5   14  0.020744  0.312075    0   
 1     AAA  2016    0  28.755176          5   15  0.046440  0.579466    0   
 2     AAA  2017    0  29.151881          5   16  0.057551  0.519565    1   
 3     AAA  2018    0  29.649805          5   17  0.028177  0.509380    1   
 4     AAA  2019    0  29.708893          5   18  0.061475  0.472190    1   
 5     AAA  2020    0  29.779221          5   19  0.033045  0.415490    1   
 6     AAA  2021    0  29.934558          7   20  0.032383  0.335598    1   
 7     AAA  2022    1  30.010181          7   21  0.010864  0.289944    1   
 8     AAA  2023    1  30.080598          5   22  0.026693  0.257639    1   
 9     AAA  2024    1  30.253384          5   23  0.023226  0.282860    1   
 10    AAA  2025    1  30.187597          5   24  0.032884  0.235118    1   
 11    ACB  2015    0  32.936597         11   23  0.005104  0.000000    0   

In [24]:
# stats_duck linear regression model selection for CED-related targets (without aud_filled)
alpha = 0.05

# Build model frame from full df so all targets exist
ced_targets = ['ced']
if 'ced_columns' in globals():
    ced_targets.extend([c for c in ced_columns if c in df.columns])
ced_targets = sorted(set([c for c in ced_targets if c in df.columns]))

reg_df = df[['ticker', 'year'] + [c for c in df.columns if c in ced_targets or c in ['size', 'boardsize', 'age', 'roa', 'loa', 'iso', 'woman', 'aud']]].copy()

# Add extra candidate controls discovered from DuckDB tables
key_df = reg_df[['ticker', 'year']].drop_duplicates().copy()
con.register('reg_keys', key_df)
extra_sql = """
WITH proper_extra AS (
  SELECT
    k.ticker, k.year,
    MAX(CASE WHEN p.indicator_code = 'S1_COMPLIANCE' THEN CAST(p.is_present AS INTEGER) END) AS s1_compliance,
    MAX(CASE WHEN p.indicator_code = 'S1_VIOLATION' THEN CAST(p.is_present AS INTEGER) END) AS s1_violation,
    MAX(CASE WHEN p.indicator_code = 'S1_MINOR_NC' THEN CAST(p.is_present AS INTEGER) END) AS s1_minor_nc,
    MAX(CASE WHEN p.indicator_code = 'S2_REDUCTION' THEN CAST(p.is_present AS INTEGER) END) AS s2_reduction,
    MAX(CASE WHEN p.indicator_code = 'S2_EFFICIENCY' THEN CAST(p.is_present AS INTEGER) END) AS s2_efficiency
  FROM reg_keys k
  LEFT JOIN proper_vn_results p ON p.ticker = k.ticker AND p.year = k.year
  GROUP BY k.ticker, k.year
) ,
gov_dir AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.foreign_count') AS DOUBLE)) AS board_foreign_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.independent_count') AS DOUBLE)) AS board_independent_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_members') AS DOUBLE)) AS board_total_members
  FROM reg_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_DIRECTORY'
  GROUP BY k.ticker, k.year
) ,
gov_sup AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.women_count') AS DOUBLE)) AS sup_women_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_members') AS DOUBLE)) AS sup_total_members
  FROM reg_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_SUPERVISORY'
  GROUP BY k.ticker, k.year
) ,
gov_share AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.foreign_ownership_pct') AS DOUBLE)) AS foreign_ownership_pct,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.state_ownership_pct') AS DOUBLE)) AS state_ownership_pct,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.institutional_shares') AS DOUBLE)) AS institutional_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.individual_shares') AS DOUBLE)) AS individual_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_outstanding_shares') AS DOUBLE)) AS total_outstanding_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_treasury_shares') AS DOUBLE)) AS total_treasury_shares
  FROM reg_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_SHAREHOLDERS'
  GROUP BY k.ticker, k.year
)
SELECT
  k.ticker,
  k.year,
  p.s1_compliance, p.s1_violation, p.s1_minor_nc, p.s2_reduction, p.s2_efficiency,
  CASE WHEN d.board_total_members > 0 THEN d.board_foreign_count / d.board_total_members END AS board_foreign_ratio,
  CASE WHEN d.board_total_members > 0 THEN d.board_independent_count / d.board_total_members END AS board_independent_ratio,
  CASE WHEN s.sup_total_members > 0 THEN s.sup_women_count / s.sup_total_members END AS sup_women_ratio,
  h.foreign_ownership_pct,
  h.state_ownership_pct,
  CASE
    WHEN COALESCE(h.institutional_shares, 0) + COALESCE(h.individual_shares, 0) > 0
    THEN h.institutional_shares / (h.institutional_shares + h.individual_shares)
  END AS institutional_ownership_ratio,
  CASE WHEN h.total_outstanding_shares > 0 THEN h.total_treasury_shares / h.total_outstanding_shares END AS treasury_share_ratio
FROM reg_keys k
LEFT JOIN proper_extra p ON p.ticker = k.ticker AND p.year = k.year
LEFT JOIN gov_dir d ON d.ticker = k.ticker AND d.year = k.year
LEFT JOIN gov_sup s ON s.ticker = k.ticker AND s.year = k.year
LEFT JOIN gov_share h ON h.ticker = k.ticker AND h.year = k.year
"""
extra_df = con.execute(extra_sql).fetchdf()
reg_df = reg_df.merge(extra_df, on=['ticker', 'year'], how='left')

# Feature engineering
if 'woman' in reg_df.columns and 'boardsize' in reg_df.columns:
    reg_df['woman_ratio'] = reg_df['woman'] / reg_df['boardsize'].where(reg_df['boardsize'] > 0)

# Candidate predictors (drop aud_filled; aud raw can remain as-is)
candidate_controls = [
    'size', 'boardsize', 'age', 'roa', 'loa', 'iso', 'aud', 'woman_ratio',
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency',
    'board_foreign_ratio', 'board_independent_ratio', 'sup_women_ratio',
    'foreign_ownership_pct', 'state_ownership_pct', 'institutional_ownership_ratio', 'treasury_share_ratio',
]
candidate_controls = [c for c in candidate_controls if c in reg_df.columns]

# Numeric cleanup and control filtering for stable estimation
for col in reg_df.columns:
    if col != 'ticker':
        reg_df[col] = pd.to_numeric(reg_df[col], errors='coerce')

x_vars_lm_final = []
for c in candidate_controls:
    s = reg_df[c]
    coverage = s.notna().mean()
    uniq = s.dropna().nunique()
    if coverage >= 0.6 and uniq > 1:
        x_vars_lm_final.append(c)

con_lm = duckdb.connect()
try:
    try:
        con_lm.execute("INSTALL stats_duck")
    except Exception:
        con_lm.execute("INSTALL stats_duck FROM community")
    con_lm.execute("LOAD stats_duck")
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for lm modeling: {str(e).splitlines()[0]}')

required_cols = sorted(set(['ticker', 'year'] + x_vars_lm_final + ced_targets))
con_lm.register('reg_df_view', reg_df[required_cols])
con_lm.execute('CREATE OR REPLACE TABLE reg_lm_data AS SELECT * FROM reg_df_view')

x_list_sql = '[' + ', '.join("'" + c + "'" for c in x_vars_lm_final) + ']'

model_rows = []
coef_tables = {}
for y_var in ced_targets:
    q_summary = f"""
    SELECT *
    FROM lm_summary('reg_lm_data', y := '{y_var}', x := {x_list_sql})
    """
    q_coef = f"""
    SELECT *
    FROM lm('reg_lm_data', y := '{y_var}', x := {x_list_sql})
    """
    try:
        s_df = con_lm.execute(q_summary).fetchdf()
        c_df = con_lm.execute(q_coef).fetchdf()

        non_intercept = c_df[c_df['term'] != '(Intercept)'].copy()
        sig_df = non_intercept[non_intercept['p_value'] < alpha]
        sig_count = int(len(sig_df))
        min_p = float(non_intercept['p_value'].min()) if not non_intercept.empty else None
        mean_abs_t = float(non_intercept['t_statistic'].abs().mean()) if not non_intercept.empty else None

        row = {
            'target': y_var,
            'n_significant': sig_count,
            'min_p_value': min_p,
            'mean_abs_t': mean_abs_t,
            'r_squared': float(s_df.loc[0, 'r_squared']) if not s_df.empty else None,
            'adj_r_squared': float(s_df.loc[0, 'adj_r_squared']) if not s_df.empty else None,
            'f_p_value': float(s_df.loc[0, 'f_p_value']) if not s_df.empty else None,
            'n': int(s_df.loc[0, 'n']) if not s_df.empty else None,
        }
        model_rows.append(row)
        coef_tables[y_var] = c_df.sort_values('p_value')
    except Exception:
        continue

model_ranking = pd.DataFrame(model_rows)
if model_ranking.empty:
    raise RuntimeError('No stats_duck lm model could be estimated for CED targets.')

model_ranking = model_ranking.sort_values(
    by=['n_significant', 'adj_r_squared', 'mean_abs_t'], ascending=[False, False, False]
).reset_index(drop=True)

best_target = model_ranking.loc[0, 'target']
best_model_coefs = coef_tables[best_target].copy()
best_model_sig = best_model_coefs[
    (best_model_coefs['term'] != '(Intercept)') & (best_model_coefs['p_value'] < alpha)
] .copy()

print('Predictors used:', x_vars_lm_final)
print('Best target by significance + fit:', best_target)
print('Top model summary row:')
print(model_ranking.head(1).to_string(index=False))

print('\nSignificant variables in best model (p < 0.05):')
if best_model_sig.empty:
    print('  None under current specification.')
else:
    print(best_model_sig[['term', 'estimate', 't_statistic', 'p_value']].to_string(index=False))

model_ranking.head(15), best_model_coefs, best_model_sig

CatalogException: Catalog Error: Table with name proper_vn_results does not exist!
Did you mean "reg_keys"?

LINE 11:   LEFT JOIN proper_vn_results p ON p.ticker = k.ticker AND p.year =...
                     ^

In [25]:
# stats_duck linear models on composite CED targets + explanatory docstrings (without aud_filled)
from itertools import combinations

alpha = 0.05

# Build modeling frame
combo_df = df.copy()
if 'woman' in combo_df.columns and 'boardsize' in combo_df.columns:
    combo_df['woman_ratio'] = combo_df['woman'] / combo_df['boardsize'].where(combo_df['boardsize'] > 0)

# Add extra candidate controls discovered from DuckDB tables
key_df = combo_df[['ticker', 'year']].drop_duplicates().copy()
con.register('combo_keys', key_df)
extra_sql = """
WITH proper_extra AS (
  SELECT
    k.ticker, k.year,
    MAX(CASE WHEN p.indicator_code = 'S1_COMPLIANCE' THEN CAST(p.is_present AS INTEGER) END) AS s1_compliance,
    MAX(CASE WHEN p.indicator_code = 'S1_VIOLATION' THEN CAST(p.is_present AS INTEGER) END) AS s1_violation,
    MAX(CASE WHEN p.indicator_code = 'S1_MINOR_NC' THEN CAST(p.is_present AS INTEGER) END) AS s1_minor_nc,
    MAX(CASE WHEN p.indicator_code = 'S2_REDUCTION' THEN CAST(p.is_present AS INTEGER) END) AS s2_reduction,
    MAX(CASE WHEN p.indicator_code = 'S2_EFFICIENCY' THEN CAST(p.is_present AS INTEGER) END) AS s2_efficiency
  FROM combo_keys k
  LEFT JOIN proper_vn_results p ON p.ticker = k.ticker AND p.year = k.year
  GROUP BY k.ticker, k.year
) ,
gov_dir AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.foreign_count') AS DOUBLE)) AS board_foreign_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.independent_count') AS DOUBLE)) AS board_independent_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_members') AS DOUBLE)) AS board_total_members
  FROM combo_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_DIRECTORY'
  GROUP BY k.ticker, k.year
) ,
gov_sup AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.women_count') AS DOUBLE)) AS sup_women_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_members') AS DOUBLE)) AS sup_total_members
  FROM combo_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_SUPERVISORY'
  GROUP BY k.ticker, k.year
) ,
gov_share AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.foreign_ownership_pct') AS DOUBLE)) AS foreign_ownership_pct,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.state_ownership_pct') AS DOUBLE)) AS state_ownership_pct,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.institutional_shares') AS DOUBLE)) AS institutional_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.individual_shares') AS DOUBLE)) AS individual_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_outstanding_shares') AS DOUBLE)) AS total_outstanding_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_treasury_shares') AS DOUBLE)) AS total_treasury_shares
  FROM combo_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_SHAREHOLDERS'
  GROUP BY k.ticker, k.year
)
SELECT
  k.ticker,
  k.year,
  p.s1_compliance, p.s1_violation, p.s1_minor_nc, p.s2_reduction, p.s2_efficiency,
  CASE WHEN d.board_total_members > 0 THEN d.board_foreign_count / d.board_total_members END AS board_foreign_ratio,
  CASE WHEN d.board_total_members > 0 THEN d.board_independent_count / d.board_total_members END AS board_independent_ratio,
  CASE WHEN s.sup_total_members > 0 THEN s.sup_women_count / s.sup_total_members END AS sup_women_ratio,
  h.foreign_ownership_pct,
  h.state_ownership_pct,
  CASE
    WHEN COALESCE(h.institutional_shares, 0) + COALESCE(h.individual_shares, 0) > 0
    THEN h.institutional_shares / (h.institutional_shares + h.individual_shares)
  END AS institutional_ownership_ratio,
  CASE WHEN h.total_outstanding_shares > 0 THEN h.total_treasury_shares / h.total_outstanding_shares END AS treasury_share_ratio
FROM combo_keys k
LEFT JOIN proper_extra p ON p.ticker = k.ticker AND p.year = k.year
LEFT JOIN gov_dir d ON d.ticker = k.ticker AND d.year = k.year
LEFT JOIN gov_sup s ON s.ticker = k.ticker AND s.year = k.year
LEFT JOIN gov_share h ON h.ticker = k.ticker AND h.year = k.year
"""
extra_df = con.execute(extra_sql).fetchdf()
combo_df = combo_df.merge(extra_df, on=['ticker', 'year'], how='left')

# CED component columns
ced_components = [c for c in ced_columns if c in combo_df.columns] if 'ced_columns' in globals() else []
if not ced_components:
    raise RuntimeError('No CED component columns found to build combinations.')

# Keep stable/usable component variables
usable_components = []
for c in ced_components:
    s = pd.to_numeric(combo_df[c], errors='coerce')
    cov = s.notna().mean()
    uniq = s.dropna().nunique()
    if cov >= 0.6 and uniq > 1:
        usable_components.append(c)

if len(usable_components) < 4:
    raise RuntimeError('Too few usable CED components for combination modeling.')

# 1) Theory-based composites by families
families = {
    'cc_index': [c for c in usable_components if c.startswith('cc')],
    'ghg_index': [c for c in usable_components if c.startswith('ghg')],
    'ec_index': [c for c in usable_components if c.startswith('ec')],
    'rc_index': [c for c in usable_components if c.startswith('rc')],
    'acc_index': [c for c in usable_components if c.startswith('acc')],
}

target_defs = {}
for name, cols in families.items():
    if cols:
        combo_df[name] = combo_df[cols].mean(axis=1, skipna=True)
        target_defs[name] = f"Mean of {len(cols)} variables: {', '.join(cols)}"

# Cross-family composites
env_cols = [c for c in usable_components if c.startswith('ghg') or c.startswith('ec') or c.startswith('cc')]
gov_cols = [c for c in usable_components if c.startswith('rc') or c.startswith('acc')]
if env_cols:
    combo_df['env_disclosure_index'] = combo_df[env_cols].mean(axis=1, skipna=True)
    target_defs['env_disclosure_index'] = f"Mean of environmental CED variables ({len(env_cols)} cols)"
if gov_cols:
    combo_df['gov_disclosure_index'] = combo_df[gov_cols].mean(axis=1, skipna=True)
    target_defs['gov_disclosure_index'] = f"Mean of governance CED variables ({len(gov_cols)} cols)"
combo_df['ced_composite_all'] = combo_df[usable_components].mean(axis=1, skipna=True)
target_defs['ced_composite_all'] = f"Mean of all usable CED components ({len(usable_components)} cols)"

# 2) Data-driven combinations: top components by |corr with ced|
corr_with_ced = []
for c in usable_components:
    cc = combo_df[['ced', c]].corr(numeric_only=True).iloc[0, 1]
    if pd.notna(cc):
        corr_with_ced.append((c, abs(float(cc))))
corr_with_ced = sorted(corr_with_ced, key=lambda x: x[1], reverse=True)
top_pool = [c for c, _ in corr_with_ced[:8]]

combo_targets = []
for k in [2, 3, 4]:
    for cols in combinations(top_pool, k):
        name = 'ced_combo_' + '_'.join(cols)
        combo_df[name] = combo_df[list(cols)].mean(axis=1, skipna=True)
        target_defs[name] = f"Mean of top-correlation components: {', '.join(cols)}"
        combo_targets.append(name)

# Candidate targets to test
candidate_targets = ['ced'] + list(target_defs.keys())

# Candidate controls and filtering (drop aud_filled)
candidate_controls = [
    'size', 'boardsize', 'age', 'roa', 'loa', 'iso', 'aud', 'woman_ratio',
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency',
    'board_foreign_ratio', 'board_independent_ratio', 'sup_women_ratio',
    'foreign_ownership_pct', 'state_ownership_pct', 'institutional_ownership_ratio', 'treasury_share_ratio',
]
candidate_controls = [c for c in candidate_controls if c in combo_df.columns]

for c in set(candidate_controls + candidate_targets):
    if c in combo_df.columns and c != 'ticker':
        combo_df[c] = pd.to_numeric(combo_df[c], errors='coerce')

x_controls = []
for c in candidate_controls:
    s = combo_df[c]
    coverage = s.notna().mean()
    uniq = s.dropna().nunique()
    if coverage >= 0.6 and uniq > 1:
        x_controls.append(c)

# stats_duck connection
con_combo = duckdb.connect()
try:
    try:
        con_combo.execute("INSTALL stats_duck")
    except Exception:
        con_combo.execute("INSTALL stats_duck FROM community")
    con_combo.execute("LOAD stats_duck")
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for combo modeling: {str(e).splitlines()[0]}')

# Register model table
needed_cols = sorted(set(['ticker', 'year'] + x_controls + candidate_targets))
con_combo.register('combo_df_view', combo_df[needed_cols])
con_combo.execute('CREATE OR REPLACE TABLE combo_model_data AS SELECT * FROM combo_df_view')

x_list_sql = '[' + ', '.join("'" + c + "'" for c in x_controls) + ']'

results = []
coef_map = {}
for y in candidate_targets:
    q_s = f"SELECT * FROM lm_summary('combo_model_data', y := '{y}', x := {x_list_sql})"
    q_c = f"SELECT * FROM lm('combo_model_data', y := '{y}', x := {x_list_sql})"
    try:
        s_df = con_combo.execute(q_s).fetchdf()
        c_df = con_combo.execute(q_c).fetchdf()
        if s_df.empty or c_df.empty:
            continue

        non_intercept = c_df[c_df['term'] != '(Intercept)'].copy()
        sig = non_intercept[non_intercept['p_value'] < alpha].copy()
        n_sig = int(len(sig))
        adj_r2 = float(s_df.loc[0, 'adj_r_squared'])
        r2 = float(s_df.loc[0, 'r_squared'])
        f_p = float(s_df.loc[0, 'f_p_value'])
        n_obs = int(s_df.loc[0, 'n'])
        min_p = float(non_intercept['p_value'].min()) if not non_intercept.empty else None

        score = n_sig * 10.0 + adj_r2 * 5.0 - (0.0 if f_p < alpha else 2.0)
        results.append({
            'target': y,
            'kind': 'base' if y == 'ced' else ('composite' if y in target_defs else 'other'),
            'n_significant': n_sig,
            'adj_r_squared': adj_r2,
            'r_squared': r2,
            'f_p_value': f_p,
            'min_p_value': min_p,
            'n': n_obs,
            'score': score,
            'definition': target_defs.get(y, 'Original CED target'),
        })
        coef_map[y] = c_df.sort_values('p_value').copy()
    except Exception:
        continue

combo_model_ranking = pd.DataFrame(results)
if combo_model_ranking.empty:
    raise RuntimeError('No combination models were estimable.')

combo_model_ranking = combo_model_ranking.sort_values(
    by=['score', 'n_significant', 'adj_r_squared'], ascending=[False, False, False]
).reset_index(drop=True)

top_n = min(10, len(combo_model_ranking))
top_models = combo_model_ranking.head(top_n).copy()

# Create docstrings for top outcomes
doc_rows = []
for _, row in top_models.iterrows():
    y = row['target']
    coefs = coef_map[y]
    sig = coefs[(coefs['term'] != '(Intercept)') & (coefs['p_value'] < alpha)].copy()
    if sig.empty:
        sig_text = 'No control variable is significant at p < 0.05.'
    else:
        pieces = []
        for _, rr in sig.iterrows():
            direction = 'positive' if rr['estimate'] >= 0 else 'negative'
            pieces.append(f"{rr['term']} ({direction}, p={rr['p_value']:.4g})")
        sig_text = '; '.join(pieces)

    doc = (
        f"""Model target: {y}
Definition: {row['definition']}
Sample size (n): {int(row['n'])}
Controls used: {', '.join(x_controls)}
Fit: R^2={row['r_squared']:.4f}, Adj.R^2={row['adj_r_squared']:.4f}, F-test p-value={row['f_p_value']:.4g}
Significant controls (alpha={alpha}): {sig_text}
Interpretation: This model explains variation in {y} using firm characteristics, PROPER indicators, and governance/ownership controls extracted from DuckDB."""
    )
    doc_rows.append({'target': y, 'docstring': doc})

model_docstrings = pd.DataFrame(doc_rows)
best_combo_target = top_models.loc[0, 'target']
best_combo_coefs = coef_map[best_combo_target].copy()
best_combo_sig = best_combo_coefs[(best_combo_coefs['term'] != '(Intercept)') & (best_combo_coefs['p_value'] < alpha)].copy()

print('Controls used:', x_controls)
print('Top combination target:', best_combo_target)
print('Definition:', top_models.loc[0, 'definition'])
print('Top model score:', round(float(top_models.loc[0, 'score']), 4))
print('Significant controls in top model:', int(top_models.loc[0, 'n_significant']))

top_models, best_combo_coefs, best_combo_sig, model_docstrings.head(10)

CatalogException: Catalog Error: Table with name proper_vn_results does not exist!
Did you mean "reg_keys"?

LINE 11:   LEFT JOIN proper_vn_results p ON p.ticker = k.ticker AND p.year =...
                     ^

In [26]:
# Export a markdown explanation of the latest modeling outcomes
from pathlib import Path

if 'top_models' not in globals() or top_models.empty:
    raise RuntimeError('top_models is empty. Run the composite modeling cell first.')

if 'best_combo_coefs' not in globals() or best_combo_coefs.empty:
    raise RuntimeError('best_combo_coefs is empty. Run the composite modeling cell first.')

if 'model_docstrings' not in globals() or model_docstrings.empty:
    raise RuntimeError('model_docstrings is empty. Run the composite modeling cell first.')

report_path = Path('data/output/ced_model_outcome_summary.md')
report_path.parent.mkdir(parents=True, exist_ok=True)

top_row = top_models.iloc[0]
best_target = top_row['target']
best_definition = top_row['definition']
best_sig_count = int(top_row['n_significant'])
best_r2 = float(top_row['r_squared'])
best_adj_r2 = float(top_row['adj_r_squared'])
best_n = int(top_row['n'])

sig_terms = best_combo_coefs[
    (best_combo_coefs['term'] != '(Intercept)') & (best_combo_coefs['p_value'] < 0.05)
][['term', 'estimate', 't_statistic', 'p_value']].copy()

lines = []
lines.append('# CED Model Outcome Summary')
lines.append('')
lines.append('## 1) What was modeled')
lines.append('- Method: stats_duck linear regression (lm and lm_summary).')
lines.append('- Dependent variables: base CED and multiple CED composite targets.')
lines.append('- Controls: firm, governance, and ownership variables extracted from DuckDB tables.')
lines.append('')
lines.append('## 2) Best-performing composite target')
lines.append(f'- Target: {best_target}')
lines.append(f'- Definition: {best_definition}')
lines.append(f'- Sample size (n): {best_n}')
lines.append(f'- R-squared: {best_r2:.4f}')
lines.append(f'- Adjusted R-squared: {best_adj_r2:.4f}')
lines.append(f'- Number of significant controls (p < 0.05): {best_sig_count}')
lines.append('')
lines.append('## 3) Significant variables in the best model')
if sig_terms.empty:
    lines.append('- No control variable is significant at p < 0.05 in the best model.')
else:
    for _, r in sig_terms.iterrows():
        direction = 'positive' if r['estimate'] >= 0 else 'negative'
        lines.append(
            f"- {r['term']}: {direction} effect, estimate={r['estimate']:.6f}, t={r['t_statistic']:.3f}, p={r['p_value']:.4g}"
        )

lines.append('')
lines.append('## 4) Top 10 ranked models')
lines.append('| Rank | Target | Significant Vars | Adj R2 | R2 | N |')
lines.append('|---:|---|---:|---:|---:|---:|')
for i, (_, r) in enumerate(top_models.head(10).iterrows(), start=1):
    lines.append(
        f"| {i} | {r['target']} | {int(r['n_significant'])} | {float(r['adj_r_squared']):.4f} | {float(r['r_squared']):.4f} | {int(r['n'])} |"
    )

lines.append('')
lines.append('## 5) Plain-language interpretation')
best_doc = model_docstrings.loc[model_docstrings['target'] == best_target, 'docstring']
if not best_doc.empty:
    lines.append(best_doc.iloc[0])
else:
    lines.append('The best model explains variation in the chosen CED composite using available financial, governance, and ownership controls.')

report_path.write_text('\n'.join(lines), encoding='utf-8')
print('Saved markdown report to:', report_path.resolve())
report_path

RuntimeError: top_models is empty. Run the composite modeling cell first.

In [24]:
# Build non-leaky financial + LLM predictors, check coverage/VIF, and rerun stats_duck model ranking
import numpy as np

alpha = 0.05

# Base modeling frame keyed by ticker-year
nl_df = df[['ticker', 'year', 'ced']].copy()

# Optional composite dependent variable (still non-leaky because it is only used as target)
if 'ced_columns' in globals() and ced_columns:
    nl_df['ced_component_mean'] = df[[c for c in ced_columns if c in df.columns]].mean(axis=1, skipna=True)

# ---------- Financial-statement item predictors ----------
fin_items = {
    '12700.0': 'total_assets',
    '13000.0': 'total_liabilities',
    '14000.0': 'owners_equity',
    '11000.0': 'current_assets',
    '11100.0': 'cash_and_equivalents',
    '13100.0': 'short_term_liabilities',
    '13300.0': 'long_term_liabilities',
    '21000.0': 'gross_operating_revenue',
    '23800.0': 'pretax_profit',
    '22070.0': 'corporate_income_tax_expense',
    '32000.0': 'net_cf_operating',
    '33000.0': 'net_cf_investing',
    '34000.0': 'net_cf_financing',
}

con.register('nl_keys', nl_df[['ticker', 'year']])
fin_code_sql = ', '.join([f"'{c}'" for c in fin_items.keys()])
fin_sql = f"""
WITH fs_base AS (
  SELECT
    code AS ticker,
    TRY_CAST(SUBSTR(fiscal_date, 1, 4) AS INTEGER) AS year,
    item_code,
    numeric_value,
    fiscal_date,
    ROW_NUMBER() OVER (
      PARTITION BY code, TRY_CAST(SUBSTR(fiscal_date, 1, 4) AS INTEGER), item_code
      ORDER BY fiscal_date DESC
    ) AS rn
  FROM financial_statements
  WHERE TRY_CAST(SUBSTR(fiscal_date, 1, 4) AS INTEGER) BETWEEN {START_YEAR} AND {END_YEAR}
), fs_latest AS (
  SELECT ticker, year, item_code, numeric_value
  FROM fs_base
  WHERE rn = 1
)
SELECT
  k.ticker,
  k.year,
  f.item_code,
  f.numeric_value
FROM nl_keys k
LEFT JOIN fs_latest f
  ON f.ticker = k.ticker AND f.year = k.year AND f.item_code IN ({fin_code_sql})
"""
fin_long = con.execute(fin_sql).fetchdf()
fin_wide = fin_long.pivot_table(index=['ticker', 'year'], columns='item_code', values='numeric_value', aggfunc='max').reset_index()
fin_wide = fin_wide.rename(columns={k: v for k, v in fin_items.items() if k in fin_wide.columns})
nl_df = nl_df.merge(fin_wide, on=['ticker', 'year'], how='left')

# ---------- LLM-extracted predictors ----------
llm_sql = """
WITH proper_extra AS (
  SELECT
    k.ticker, k.year,
    MAX(CASE WHEN p.indicator_code = 'S1_COMPLIANCE' THEN CAST(p.is_present AS INTEGER) END) AS s1_compliance,
    MAX(CASE WHEN p.indicator_code = 'S1_VIOLATION' THEN CAST(p.is_present AS INTEGER) END) AS s1_violation,
    MAX(CASE WHEN p.indicator_code = 'S1_MINOR_NC' THEN CAST(p.is_present AS INTEGER) END) AS s1_minor_nc,
    MAX(CASE WHEN p.indicator_code = 'S2_REDUCTION' THEN CAST(p.is_present AS INTEGER) END) AS s2_reduction,
    MAX(CASE WHEN p.indicator_code = 'S2_EFFICIENCY' THEN CAST(p.is_present AS INTEGER) END) AS s2_efficiency,
    MAX(CASE WHEN p.indicator_code = 'S2_ISO14001' THEN CAST(p.is_present AS INTEGER) END) AS iso_14001
  FROM nl_keys k
  LEFT JOIN proper_vn_results p ON p.ticker = k.ticker AND p.year = k.year
  GROUP BY k.ticker, k.year
) ,
gov_dir AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.foreign_count') AS DOUBLE)) AS board_foreign_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.independent_count') AS DOUBLE)) AS board_independent_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.women_count') AS DOUBLE)) AS board_women_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_members') AS DOUBLE)) AS board_total_members
  FROM nl_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_DIRECTORY'
  GROUP BY k.ticker, k.year
) ,
gov_sup AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.women_count') AS DOUBLE)) AS sup_women_count,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_members') AS DOUBLE)) AS sup_total_members
  FROM nl_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_SUPERVISORY'
  GROUP BY k.ticker, k.year
) ,
gov_share AS (
  SELECT
    k.ticker, k.year,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.foreign_ownership_pct') AS DOUBLE)) AS foreign_ownership_pct,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.state_ownership_pct') AS DOUBLE)) AS state_ownership_pct,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.institutional_shares') AS DOUBLE)) AS institutional_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.individual_shares') AS DOUBLE)) AS individual_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_outstanding_shares') AS DOUBLE)) AS total_outstanding_shares,
    MAX(TRY_CAST(json_extract_string(g.value_json, '$.total_treasury_shares') AS DOUBLE)) AS total_treasury_shares
  FROM nl_keys k
  LEFT JOIN governance_results g
    ON g.ticker = k.ticker AND g.year = k.year AND g.item_code = 'GOV_SHAREHOLDERS'
  GROUP BY k.ticker, k.year
)
SELECT
  k.ticker, k.year,
  p.s1_compliance, p.s1_violation, p.s1_minor_nc, p.s2_reduction, p.s2_efficiency, p.iso_14001,
  CASE WHEN d.board_total_members > 0 THEN d.board_foreign_count / d.board_total_members END AS board_foreign_ratio,
  CASE WHEN d.board_total_members > 0 THEN d.board_independent_count / d.board_total_members END AS board_independent_ratio,
  CASE WHEN d.board_total_members > 0 THEN d.board_women_count / d.board_total_members END AS board_women_ratio,
  CASE WHEN s.sup_total_members > 0 THEN s.sup_women_count / s.sup_total_members END AS sup_women_ratio,
  h.foreign_ownership_pct,
  h.state_ownership_pct,
  CASE
    WHEN COALESCE(h.institutional_shares, 0) + COALESCE(h.individual_shares, 0) > 0
    THEN h.institutional_shares / (h.institutional_shares + h.individual_shares)
  END AS institutional_ownership_ratio,
  CASE WHEN h.total_outstanding_shares > 0 THEN h.total_treasury_shares / h.total_outstanding_shares END AS treasury_share_ratio
FROM nl_keys k
LEFT JOIN proper_extra p ON p.ticker = k.ticker AND p.year = k.year
LEFT JOIN gov_dir d ON d.ticker = k.ticker AND d.year = k.year
LEFT JOIN gov_sup s ON s.ticker = k.ticker AND s.year = k.year
LEFT JOIN gov_share h ON h.ticker = k.ticker AND h.year = k.year
"""
llm_extra = con.execute(llm_sql).fetchdf()
nl_df = nl_df.merge(llm_extra, on=['ticker', 'year'], how='left')

# ---------- Derived non-leaky financial ratios ----------
nl_df['size_ln_assets'] = np.log(nl_df['total_assets'].where(nl_df['total_assets'] > 0))
nl_df['leverage_ratio'] = nl_df['total_liabilities'] / nl_df['total_assets']
nl_df['equity_ratio'] = nl_df['owners_equity'] / nl_df['total_assets']
nl_df['current_asset_intensity'] = nl_df['current_assets'] / nl_df['total_assets']
nl_df['cash_intensity'] = nl_df['cash_and_equivalents'] / nl_df['total_assets']
nl_df['short_debt_ratio'] = nl_df['short_term_liabilities'] / nl_df['total_assets']
nl_df['long_debt_ratio'] = nl_df['long_term_liabilities'] / nl_df['total_assets']
nl_df['revenue_to_assets'] = nl_df['gross_operating_revenue'] / nl_df['total_assets']
nl_df['pretax_margin'] = nl_df['pretax_profit'] / nl_df['gross_operating_revenue']
nl_df['tax_burden'] = nl_df['corporate_income_tax_expense'] / nl_df['pretax_profit']
nl_df['ocf_to_assets'] = nl_df['net_cf_operating'] / nl_df['total_assets']
nl_df['icf_to_assets'] = nl_df['net_cf_investing'] / nl_df['total_assets']
nl_df['fcf_to_assets'] = nl_df['net_cf_financing'] / nl_df['total_assets']

# ---------- Predictor filtering by coverage and variation ----------
candidate_x = [
    'size_ln_assets', 'leverage_ratio', 'equity_ratio', 'current_asset_intensity', 'cash_intensity',
    'short_debt_ratio', 'long_debt_ratio', 'revenue_to_assets', 'pretax_margin', 'tax_burden',
    'ocf_to_assets', 'icf_to_assets', 'fcf_to_assets',
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001',
    'board_foreign_ratio', 'board_independent_ratio', 'board_women_ratio', 'sup_women_ratio',
    'foreign_ownership_pct', 'state_ownership_pct', 'institutional_ownership_ratio', 'treasury_share_ratio',
]
candidate_x = [c for c in candidate_x if c in nl_df.columns]

coverage_rows = []
x_final = []
for c in candidate_x:
    s = pd.to_numeric(nl_df[c], errors='coerce')
    cov = float(s.notna().mean())
    uniq = int(s.dropna().nunique())
    coverage_rows.append({'variable': c, 'coverage': cov, 'n_unique': uniq})
    if cov >= 0.60 and uniq > 1:
        x_final.append(c)
coverage_table = pd.DataFrame(coverage_rows).sort_values(['coverage', 'n_unique'], ascending=[False, False])

# ---------- Quick VIF check ----------
vif_rows = []
vif_input = nl_df[x_final].apply(pd.to_numeric, errors='coerce').dropna()
if len(vif_input) > 20 and len(x_final) >= 2:
    for target in x_final:
        others = [c for c in x_final if c != target]
        yv = vif_input[target].astype(float).to_numpy()
        Xm = vif_input[others].astype(float).copy()
        Xm['_const'] = 1.0
        X = Xm.to_numpy(dtype=float)
        b, *_ = np.linalg.lstsq(X, yv, rcond=None)
        yhat = X @ b
        ss_res = np.sum((yv - yhat) ** 2)
        ss_tot = np.sum((yv - yv.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        vif = 1 / (1 - r2) if pd.notna(r2) and r2 < 1 else np.inf
        vif_rows.append({'variable': target, 'vif': float(vif)})
vif_table = pd.DataFrame(vif_rows).sort_values('vif', ascending=False) if vif_rows else pd.DataFrame(columns=['variable', 'vif'])

# ---------- stats_duck model ranking (non-leaky X only) ----------
targets = ['ced']
if 'ced_component_mean' in nl_df.columns:
    targets.append('ced_component_mean')
for col in targets + x_final:
    nl_df[col] = pd.to_numeric(nl_df[col], errors='coerce')

con_nl = duckdb.connect()
try:
    try:
        con_nl.execute("INSTALL stats_duck")
    except Exception:
        con_nl.execute("INSTALL stats_duck FROM community")
    con_nl.execute("LOAD stats_duck")
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for non-leaky modeling: {str(e).splitlines()[0]}')

def _fit_with_predictors(preds):
    if len(preds) < 2:
        return pd.DataFrame(), {}
    use_cols = ['ticker', 'year'] + preds + targets
    local_df = nl_df[use_cols].copy()
    con_nl.register('nl_model_view', local_df)
    con_nl.execute('CREATE OR REPLACE TABLE nl_model_data AS SELECT * FROM nl_model_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in preds) + ']'
    rows = []
    coef_map = {}
    for y in targets:
        q_sum = f"SELECT * FROM lm_summary('nl_model_data', y := '{y}', x := {x_sql})"
        q_coef = f"SELECT * FROM lm('nl_model_data', y := '{y}', x := {x_sql})"
        try:
            s_df = con_nl.execute(q_sum).fetchdf()
            c_df = con_nl.execute(q_coef).fetchdf()
            noni = c_df[c_df['term'] != '(Intercept)'].copy()
            sig = noni[noni['p_value'] < alpha]
            rows.append({
                'target': y,
                'n_significant': int(len(sig)),
                'r_squared': float(s_df.loc[0, 'r_squared']) if not s_df.empty else np.nan,
                'adj_r_squared': float(s_df.loc[0, 'adj_r_squared']) if not s_df.empty else np.nan,
                'f_p_value': float(s_df.loc[0, 'f_p_value']) if not s_df.empty else np.nan,
                'n': int(s_df.loc[0, 'n']) if not s_df.empty else np.nan,
            })
            coef_map[y] = c_df.sort_values('p_value')
        except Exception:
            continue
    if not rows:
        return pd.DataFrame(), {}
    return pd.DataFrame(rows).sort_values(['n_significant', 'adj_r_squared'], ascending=[False, False]).reset_index(drop=True), coef_map

predictor_sets = []
predictor_sets.append(x_final)
if not vif_table.empty:
    low_vif = vif_table[vif_table['vif'] <= 10.0]['variable'].tolist()
    if len(low_vif) >= 2:
        predictor_sets.append(low_vif)
coverage_ranked = coverage_table.sort_values(['coverage', 'n_unique'], ascending=[False, False])['variable'].tolist()
for k in [15, 12, 10, 8]:
    subset = [c for c in coverage_ranked if c in x_final][:k]
    if len(subset) >= 2:
        predictor_sets.append(subset)

# De-duplicate predictor sets while preserving order
seen = set()
uniq_sets = []
for ps in predictor_sets:
    key = tuple(ps)
    if key not in seen:
        seen.add(key)
        uniq_sets.append(ps)

nonleaky_model_ranking = pd.DataFrame()
nl_coef_map = {}
nonleaky_predictors_used = []
for ps in uniq_sets:
    ranking, cmap = _fit_with_predictors(ps)
    if not ranking.empty:
        nonleaky_model_ranking = ranking
        nl_coef_map = cmap
        nonleaky_predictors_used = ps
        break

if nonleaky_model_ranking.empty:
    raise RuntimeError('No non-leaky model could be estimated after trying reduced predictor sets.')

nonleaky_best_target = nonleaky_model_ranking.loc[0, 'target']
nonleaky_best_coefs = nl_coef_map[nonleaky_best_target].copy()
nonleaky_best_sig = nonleaky_best_coefs[
    (nonleaky_best_coefs['term'] != '(Intercept)') & (nonleaky_best_coefs['p_value'] < alpha)
] .copy()

print('Candidate predictors after coverage filter:', len(x_final))
print('Predictors used in successful fit:', len(nonleaky_predictors_used))
print('Best non-leaky target:', nonleaky_best_target)
print(nonleaky_model_ranking.head(5).to_string(index=False))

coverage_table.head(30), vif_table.head(30), nonleaky_model_ranking, nonleaky_best_coefs, nonleaky_best_sig

RuntimeError: stats_duck unavailable for non-leaky modeling: HTTP Error: Failed to download extension "stats_duck" at URL "http://community-extensions.duckdb.org/v1.5.2/linux_amd64/stats_duck.duckdb_extension.gz" (HTTP 404)

In [8]:
# Final publication-ready non-leaky model (VIF-capped) + CSV/Markdown export
from pathlib import Path
import numpy as np
import pandas as pd

VIF_CAP = 10.0
MIN_PREDICTORS = 5
ALPHA = 0.05

if 'nl_df' not in globals():
    raise RuntimeError('nl_df not found. Run the non-leaky modeling cell first.')
if 'x_final' not in globals() or not x_final:
    raise RuntimeError('x_final not found or empty. Run the non-leaky modeling cell first.')

target_candidates = ['ced_component_mean', 'ced']
target = next((t for t in target_candidates if t in nl_df.columns), None)
if target is None:
    raise RuntimeError('No target available. Need ced or ced_component_mean.')

predictors_init = [c for c in x_final if c in nl_df.columns]
if len(predictors_init) < MIN_PREDICTORS:
    raise RuntimeError('Not enough candidate predictors to build final model.')

fit_df = nl_df[[target] + predictors_init].apply(pd.to_numeric, errors='coerce').dropna().copy()
if len(fit_df) < 100:
    raise RuntimeError(f'Too few complete rows for stable final model: {len(fit_df)}')

def _compute_vif(df_x):
    rows = []
    cols = list(df_x.columns)
    if len(cols) < 2:
        return pd.DataFrame(columns=['variable', 'vif'])
    arr_cache = {c: df_x[c].astype(float).to_numpy() for c in cols}
    for c in cols:
        others = [x for x in cols if x != c]
        yv = arr_cache[c]
        Xm = df_x[others].astype(float).copy()
        Xm['_const'] = 1.0
        X = Xm.to_numpy(dtype=float)
        b, *_ = np.linalg.lstsq(X, yv, rcond=None)
        yhat = X @ b
        ss_res = np.sum((yv - yhat) ** 2)
        ss_tot = np.sum((yv - yv.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        vif = 1 / (1 - r2) if pd.notna(r2) and r2 < 1 else np.inf
        rows.append({'variable': c, 'vif': float(vif)})
    return pd.DataFrame(rows).sort_values('vif', ascending=False).reset_index(drop=True)

# Stepwise VIF pruning
selected_predictors = predictors_init.copy()
removed_for_vif = []
vif_history = []

while len(selected_predictors) > MIN_PREDICTORS:
    vif_df = _compute_vif(fit_df[selected_predictors])
    if vif_df.empty:
        break
    max_row = vif_df.iloc[0]
    max_vif = float(max_row['vif'])
    vif_history.append(vif_df.copy())
    if np.isfinite(max_vif) and max_vif <= VIF_CAP:
        break
    drop_var = str(max_row['variable'])
    selected_predictors.remove(drop_var)
    removed_for_vif.append((drop_var, max_vif))

final_vif = _compute_vif(fit_df[selected_predictors])

# Fit final model via stats_duck
con_final = duckdb.connect()
try:
    try:
        con_final.execute("INSTALL stats_duck")
    except Exception:
        con_final.execute("INSTALL stats_duck FROM community")
    con_final.execute("LOAD stats_duck")
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for final model: {str(e).splitlines()[0]}')

model_table = fit_df[[target] + selected_predictors].copy()
con_final.register('final_model_view', model_table)
con_final.execute('CREATE OR REPLACE TABLE final_model_data AS SELECT * FROM final_model_view')

x_sql = '[' + ', '.join("'" + c + "'" for c in selected_predictors) + ']'
q_summary = f"SELECT * FROM lm_summary('final_model_data', y := '{target}', x := {x_sql})"
q_coef = f"SELECT * FROM lm('final_model_data', y := '{target}', x := {x_sql})"
final_summary = con_final.execute(q_summary).fetchdf()
final_coefs = con_final.execute(q_coef).fetchdf().sort_values('p_value').reset_index(drop=True)
final_sig = final_coefs[(final_coefs['term'] != '(Intercept)') & (final_coefs['p_value'] < ALPHA)].copy()

# Export outputs
out_dir = Path('data/output')
out_dir.mkdir(parents=True, exist_ok=True)
coef_csv = out_dir / 'nonleaky_final_model_coeffs.csv'
sig_csv = out_dir / 'nonleaky_final_model_significant.csv'
vif_csv = out_dir / 'nonleaky_final_model_vif.csv'
md_path = out_dir / 'nonleaky_final_model_summary.md'

final_coefs.to_csv(coef_csv, index=False)
final_sig.to_csv(sig_csv, index=False)
final_vif.to_csv(vif_csv, index=False)

r2 = float(final_summary.loc[0, 'r_squared']) if not final_summary.empty else np.nan
adj_r2 = float(final_summary.loc[0, 'adj_r_squared']) if not final_summary.empty else np.nan
f_p = float(final_summary.loc[0, 'f_p_value']) if not final_summary.empty else np.nan
n_obs = int(final_summary.loc[0, 'n']) if not final_summary.empty else len(model_table)

lines = []
lines.append('# Final Non-Leaky Model (VIF-Capped)')
lines.append('')
lines.append(f'- Target: {target}')
lines.append(f'- Sample size (n): {n_obs}')
lines.append(f'- R-squared: {r2:.4f}')
lines.append(f'- Adjusted R-squared: {adj_r2:.4f}')
lines.append(f'- F-test p-value: {f_p:.4g}')
lines.append(f'- VIF cap: {VIF_CAP}')
lines.append('')
lines.append('## Selected Predictors')
for p in selected_predictors:
    lines.append(f'- {p}')
lines.append('')
lines.append('## Removed By VIF Pruning')
if removed_for_vif:
    for name, v in removed_for_vif:
        vv = 'inf' if not np.isfinite(v) else f'{v:.2f}'
        lines.append(f'- {name} (VIF={vv})')
else:
    lines.append('- None')
lines.append('')
lines.append('## Significant Coefficients (p < 0.05)')
if final_sig.empty:
    lines.append('- None')
else:
    for _, r in final_sig.iterrows():
        direction = 'positive' if r['estimate'] >= 0 else 'negative'
        lines.append(f"- {r['term']}: {direction}, estimate={r['estimate']:.6f}, t={r['t_statistic']:.3f}, p={r['p_value']:.4g}")

md_path.write_text('\n'.join(lines), encoding='utf-8')

print('Final model created.')
print('Target:', target)
print('Predictors used:', len(selected_predictors))
print('Significant predictors:', len(final_sig))
print('Saved:', coef_csv.resolve())
print('Saved:', sig_csv.resolve())
print('Saved:', vif_csv.resolve())
print('Saved:', md_path.resolve())

final_summary, final_coefs, final_sig, final_vif, removed_for_vif

Final model created.
Target: ced_component_mean
Predictors used: 25
Significant predictors: 9
Saved: /media/nvme0n1/dev/annual_report/data/output/nonleaky_final_model_coeffs.csv
Saved: /media/nvme0n1/dev/annual_report/data/output/nonleaky_final_model_significant.csv
Saved: /media/nvme0n1/dev/annual_report/data/output/nonleaky_final_model_vif.csv
Saved: /media/nvme0n1/dev/annual_report/data/output/nonleaky_final_model_summary.md


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.530155       0.507824    23.740712        0.0        25          526   
 
       sigma    n  
 0  0.113458  552  ,
                              term  estimate  std_error  t_statistic  \
 0                    s2_reduction  0.160007   0.014683    10.897761   
 1                   s2_efficiency  0.088379   0.010699     8.260499   
 2                       iso_14001  0.077494   0.014426     5.371752   
 3                  size_ln_assets  0.022098   0.004341     5.090620   
 4         board_independent_ratio  0.125335   0.031656     3.959326   
 5                   fcf_to_assets -0.198621   0.065535    -3.030747   
 6                     s1_minor_nc  0.049608   0.016798     2.953187   
 7                     (Intercept) -0.307092   0.112695    -2.724975   
 8                short_debt_ratio -0.075873   0.035580    -2.132493   
 9         current_asset_intensity -0.056024   0.027146    -2.063780   
 10    

In [25]:
# Meaning-based CED composite search (one-of-three variant per code)
from pathlib import Path
import numpy as np
import pandas as pd

if 'df' not in globals():
    raise RuntimeError('df not found. Run extraction cells first.')
if 'nl_df' not in globals():
    raise RuntimeError('nl_df not found. Run non-leaky modeling cell first.')

# Reuse final selected predictors when available for fair comparison.
if 'selected_predictors' in globals() and selected_predictors:
    model_predictors = [c for c in selected_predictors if c in nl_df.columns]
elif 'x_final' in globals() and x_final:
    model_predictors = [c for c in x_final if c in nl_df.columns]
else:
    raise RuntimeError('No predictor set available. Run non-leaky modeling first.')

CHECKLIST_GROUPS = {
    'CC': ['CC1', 'CC2'],
    'GHG': ['GHG1', 'GHG2', 'GHG3', 'GHG4', 'GHG5', 'GHG6', 'GHG7'],
    'EC': ['EC1', 'EC2', 'EC3'],
    'RC': ['RC1', 'RC2', 'RC3', 'RC4'],
    'ACC': ['ACC1', 'ACC2'],
}

sem_df = df[['ticker', 'year', 'ced']].copy()


def _variant_candidates(code: str):
    base = code.lower()
    return [base, f'{base}_alt_two', f'{base}_alt']


def _choose_best_variant(code: str) -> tuple[str | None, dict]:
    candidates = [c for c in _variant_candidates(code) if c in df.columns]
    if not candidates:
        return None, {'reason': 'missing_all_variants'}

    rows = []
    y = pd.to_numeric(df['ced'], errors='coerce') if 'ced' in df.columns else pd.Series(np.nan, index=df.index)

    for c in candidates:
        s = pd.to_numeric(df[c], errors='coerce')
        cov = float(s.notna().mean())
        uniq = int(s.dropna().nunique())
        if uniq <= 1:
            corr_abs = 0.0
        else:
            pair = pd.DataFrame({'y': y, 'x': s}).dropna()
            if len(pair) >= 20:
                corr_val = pair['y'].corr(pair['x'])
                corr_abs = float(abs(corr_val)) if pd.notna(corr_val) else 0.0
            else:
                corr_abs = 0.0

        # Prefer predictive signal, then coverage.
        score = 0.75 * corr_abs + 0.25 * cov
        rows.append({'variant': c, 'coverage': cov, 'n_unique': uniq, 'abs_corr_with_ced': corr_abs, 'score': score})

    rank = pd.DataFrame(rows).sort_values(['score', 'coverage', 'n_unique'], ascending=[False, False, False]).reset_index(drop=True)
    best = rank.loc[0]
    return str(best['variant']), {'ranking': rank}


selected_variant_map = {}
selection_details = {}
for fam_codes in CHECKLIST_GROUPS.values():
    for code in fam_codes:
        chosen, detail = _choose_best_variant(code)
        if chosen is not None:
            selected_variant_map[code] = chosen
            selection_details[code] = detail
            sem_df[chosen] = pd.to_numeric(df[chosen], errors='coerce')

# Build blocks with one selected variable per checklist code.
def _selected_cols(codes):
    return [selected_variant_map[c] for c in codes if c in selected_variant_map]

cc_cols = _selected_cols(CHECKLIST_GROUPS['CC'])
ghg_cols = _selected_cols(CHECKLIST_GROUPS['GHG'])
ec_cols = _selected_cols(CHECKLIST_GROUPS['EC'])
rc_cols = _selected_cols(CHECKLIST_GROUPS['RC'])
acc_cols = _selected_cols(CHECKLIST_GROUPS['ACC'])

mrv_cols = ghg_cols + ec_cols

if not mrv_cols or not rc_cols:
    raise RuntimeError('Missing selected variables for MRV/RC blocks after one-of-three selection.')

sem_df['ced_climate_context'] = sem_df[cc_cols].mean(axis=1, skipna=True) if cc_cols else np.nan
sem_df['ced_mrv_emissions_energy'] = sem_df[mrv_cols].mean(axis=1, skipna=True)
sem_df['ced_reduction_execution'] = sem_df[rc_cols].mean(axis=1, skipna=True)
sem_df['ced_accountability_governance'] = sem_df[acc_cols].mean(axis=1, skipna=True) if acc_cols else np.nan

# Meaning-based candidates.
sem_df['ced_semantic_core'] = sem_df[
    ['ced_climate_context', 'ced_mrv_emissions_energy', 'ced_reduction_execution', 'ced_accountability_governance']
].mean(axis=1, skipna=True)
sem_df['ced_action_accountability'] = sem_df[
    ['ced_reduction_execution', 'ced_accountability_governance']
].mean(axis=1, skipna=True)
sem_df['ced_disclosure_depth'] = sem_df[
    ['ced_climate_context', 'ced_mrv_emissions_energy']
].mean(axis=1, skipna=True)
sem_df['ced_transition_readiness'] = sem_df[
    ['ced_climate_context', 'ced_reduction_execution', 'ced_accountability_governance']
].mean(axis=1, skipna=True)
sem_df['ced_mrv_to_action'] = sem_df[
    ['ced_mrv_emissions_energy', 'ced_reduction_execution']
].mean(axis=1, skipna=True)
sem_df['ced_semantic_equal_all'] = sem_df[
    ['ced_climate_context', 'ced_mrv_emissions_energy', 'ced_reduction_execution', 'ced_accountability_governance']
].mean(axis=1, skipna=True)

semantic_targets = [
    'ced_semantic_core',
    'ced_action_accountability',
    'ced_disclosure_depth',
    'ced_transition_readiness',
    'ced_mrv_to_action',
    'ced_semantic_equal_all',
]

nl_sem_df = nl_df.merge(sem_df[['ticker', 'year'] + semantic_targets], on=['ticker', 'year'], how='left')
for c in model_predictors + semantic_targets:
    if c in nl_sem_df.columns:
        nl_sem_df[c] = pd.to_numeric(nl_sem_df[c], errors='coerce')

# Fit and rank each semantic target with the same predictor set.
con_sem = duckdb.connect()
try:
    try:
        con_sem.execute('INSTALL stats_duck')
    except Exception:
        con_sem.execute('INSTALL stats_duck FROM community')
    con_sem.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for semantic combination search: {str(e).splitlines()[0]}')

rows = []
coef_map = {}
x_sql = '[' + ', '.join("'" + c + "'" for c in model_predictors) + ']'

for y in semantic_targets:
    local = nl_sem_df[[y] + model_predictors].dropna().copy()
    if len(local) < 120:
        continue
    con_sem.register('sem_view', local)
    con_sem.execute('CREATE OR REPLACE TABLE sem_model_data AS SELECT * FROM sem_view')
    q_s = f"SELECT * FROM lm_summary('sem_model_data', y := '{y}', x := {x_sql})"
    q_c = f"SELECT * FROM lm('sem_model_data', y := '{y}', x := {x_sql})"
    try:
        s_df = con_sem.execute(q_s).fetchdf()
        c_df = con_sem.execute(q_c).fetchdf()
    except Exception:
        continue

    noni = c_df[c_df['term'] != '(Intercept)'].copy()
    sig = noni[noni['p_value'] < 0.05]
    adj_r2 = float(s_df.loc[0, 'adj_r_squared'])
    r2 = float(s_df.loc[0, 'r_squared'])
    n_obs = int(s_df.loc[0, 'n'])
    score = float(len(sig)) * 10.0 + adj_r2 * 5.0

    rows.append({
        'target': y,
        'n': n_obs,
        'r_squared': r2,
        'adj_r_squared': adj_r2,
        'f_p_value': float(s_df.loc[0, 'f_p_value']),
        'n_significant': int(len(sig)),
        'score': score,
    })
    coef_map[y] = c_df.sort_values('p_value').reset_index(drop=True)

semantic_ranking = pd.DataFrame(rows)
if semantic_ranking.empty:
    raise RuntimeError('No semantic target model could be estimated.')

semantic_ranking = semantic_ranking.sort_values(
    ['score', 'adj_r_squared', 'n_significant'], ascending=[False, False, False]
).reset_index(drop=True)

best_semantic_target = semantic_ranking.loc[0, 'target']
best_semantic_coefs = coef_map[best_semantic_target].copy()
best_semantic_sig = best_semantic_coefs[
    (best_semantic_coefs['term'] != '(Intercept)') & (best_semantic_coefs['p_value'] < 0.05)
].copy()

# Export markdown summary.
out_md = Path('data/output/ced_semantic_best_combination.md')
out_md.parent.mkdir(parents=True, exist_ok=True)

target_def = {
    'ced_semantic_core': 'Mean of 4 meaning blocks: climate context, MRV emissions-energy, reduction execution, accountability governance.',
    'ced_action_accountability': 'Mean of reduction execution and accountability governance blocks.',
    'ced_disclosure_depth': 'Mean of climate context and MRV emissions-energy blocks.',
    'ced_transition_readiness': 'Mean of climate context, reduction execution, and accountability governance blocks.',
    'ced_mrv_to_action': 'Mean of MRV emissions-energy and reduction execution blocks.',
    'ced_semantic_equal_all': 'Equal-weight mean of all 4 meaning blocks.',
}

checklist_items_en = {
    'CC1': 'Assessment of climate-related risks and opportunities',
    'CC2': 'Financial implications of climate change',
    'GHG1': 'Methodology for GHG emission calculation',
    'GHG2': 'External verification of GHG emissions',
    'GHG3': 'Total GHG emissions disclosed',
    'GHG4': 'Disclosure of emissions by scope (Scope 1, 2, 3)',
    'GHG5': 'Disclosure of emissions by source',
    'GHG6': 'Disclosure of emissions by facility or segment',
    'GHG7': 'Historical comparison of emissions over time',
    'EC1': 'Total energy consumed',
    'EC2': 'Disclosure of energy consumption from renewable sources',
    'EC3': 'Disclosure of energy consumption by type, facility, or segment',
    'RC1': 'Plans or strategies to reduce GHG emissions',
    'RC2': 'Specific targets for GHG emission reduction',
    'RC3': 'Reductions achieved to date',
    'RC4': 'Costs of future emissions factored into capital expenditure planning',
    'ACC1': 'Responsibility for climate change policy and action',
    'ACC2': 'Consideration of climate-related goals in executive compensation',
}

checklist_base_descriptions = {
    'CC1': 'Danh gia cac rui ro (cac quy dinh, tac dong vat ly hoac cac tac dong chung) lien quan den bien doi khi hau va cac hanh dong da hoac se thuc hien de quan ly rui ro.',
    'CC2': 'Danh gia cac tac dong tai chinh hien tai (va tuong lai), tac dong kinh doanh va co hoi cua bien doi khi hau.',
    'GHG1': 'Mo ta cac phuong phap su dung de tinh toan khi thai nha kinh.',
    'GHG2': 'Co su xac nhan boi yeu to ben ngoai ve luong phat thai khi nha kinh hay khong? Neu co, boi ai va tren co so gi?',
    'GHG3': 'Luong phat thai khi nha kinh tinh bang don vi MtCO2e (he met tan CO2 thai ra).',
    'GHG4': 'Viec cong bo lien quan den Pham vi 1, Pham vi 2, Pham vi 3 va lien quan truc tiep den phat thai khi nha kinh.',
    'GHG5': 'Cong bo phat thai nha kinh dua tren nguon phat thai nao?',
    'GHG6': 'Cong bo phat thai khi nha kinh dua tren co so vat chat hoac cap do nao?',
    'GHG7': 'So sanh luong phat thai khi nha kinh trong nam hien tai va nam truoc.',
    'EC1': 'Luong nang luong tieu thu.',
    'EC2': 'Luong nang luong tieu thu co nguon goc tu nguon nang luong tai tao.',
    'EC3': 'Tiet lo dua tren loai khi thai, co so vat chat hoac cap do nao?',
    'RC1': 'Giai thich chi tiet ve chien luoc va ke hoach giam thieu phat thai khi nha kinh.',
    'RC2': 'Muc tieu cu the ve luong giam thieu phat thai nha kinh.',
    'RC3': 'Luong giam phat thai toi da va cac chi phi hoac tiet kiem lien quan den giam phat khi thai nha kinh tinh den thoi diem bao cao.',
    'RC4': 'Muc do chi phi lien quan den phat thai khi nha kinh trong tuong lai vi chi phi nay duoc bao gom trong ke hoach su dung von cua cong ty.',
    'ACC1': 'Giai thich ai la Uy ban hoac Giam doc chiu trach nhiem ve cac chinh sach lien quan den bien doi khi hau.',
    'ACC2': 'Giai thich co che xem xet khi dat duoc muc tieu cong ty lien quan den bien doi khi hau boi Uy ban hoac Hoi dong quan tri.',
}

checklist_alt_descriptions = {
    'CC1': 'Chap nhan neu co mo ta rui ro/co hoi khi hau hoac rui ro moi truong lien quan (nang nong, ngap lut, gian doan chuoi cung ung, chuyen dich chinh sach) va hanh dong ung pho o muc dinh tinh.',
    'CC2': 'Chap nhan neu co de cap tac dong tai chinh hoac kinh doanh do yeu to khi hau/moi truong, ke ca chi neu xu huong chi phi, dau tu, doanh thu, hoac rui ro ma chua co so lieu day du.',
    'GHG1': 'Chap nhan neu co mo ta cach doanh nghiep theo doi/do luong phat thai hoac tieu thu nang luong lien quan phat thai, du chua neu chuan phuong phap chi tiet.',
    'GHG2': 'Chap nhan neu co bat ky de cap xac minh/danh gia ben ngoai lien quan du lieu moi truong hoac he thong quan ly moi truong, khong bat buoc xac minh rieng chi tieu GHG.',
    'GHG3': 'Chap nhan neu co cong bo bat ky so lieu phat thai/carbon/CO2 tuong duong hoac chi so thay the lien quan phat thai, ke ca chua chuan hoa don vi MtCO2e.',
    'GHG4': 'Chap nhan neu co tach nhom phat thai theo loai/pham vi/nguon tuong duong (truc tiep, gian tiep, dien nang, nhien lieu) du khong ghi dung nhan Scope 1/2/3.',
    'GHG5': 'Chap nhan neu co mo ta nguon phat thai chinh (dien, nhien lieu, van tai, san xuat, logistics...) o bat ky muc chi tiet nao.',
    'GHG6': 'Chap nhan neu co phan tach phat thai hoac chi so lien quan theo nha may, don vi, khu vuc, mang kinh doanh, hoac bat ky phan khuc van hanh nao.',
    'GHG7': 'Chap nhan neu co so sanh theo thoi gian cua phat thai hoac chi so moi truong lien quan (nam truoc/sau, xu huong tang giam), ke ca chi neu dinh tinh.',
    'EC1': 'Chap nhan neu co cong bo tong tieu thu nang luong hoac chi so dai dien (dien, nhien lieu, nhiet), khong bat buoc day du moi loai nang luong.',
    'EC2': 'Chap nhan neu co de cap su dung nang luong tai tao hoac nguon nang luong sach, ke ca mo ta chuong trinh/chu truong ma chua co so lieu chi tiet.',
    'EC3': 'Chap nhan neu co bat ky phan tach tieu thu nang luong theo loai nang luong, don vi van hanh, co so, hoac phan khuc hoat dong.',
    'RC1': 'Chap nhan neu co neu ke hoach/chuong trinh/sang kien giam phat thai hoac giam tac dong khi hau o muc dinh huong, ke ca chua co KPI dinh luong.',
    'RC2': 'Chap nhan neu co muc tieu giam phat thai hoac muc tieu moi truong tuong duong (tiet kiem nang luong, giam cuong do phat thai, net-zero) du chua du baseline hoac moc thoi gian chi tiet.',
    'RC3': 'Chap nhan neu co de cap ket qua da dat duoc lien quan giam phat thai/giam tieu thu nang luong/hieu qua tai nguyen, ke ca chi neu phan tram hoac mo ta ket qua dinh tinh.',
    'RC4': 'Chap nhan neu co de cap yeu to moi truong/phat thai duoc can nhac trong dau tu, mua sam, nang cap cong nghe, hoac quyet dinh von, khong bat buoc luong hoa chi phi carbon.',
    'ACC1': 'Chap nhan neu co chi ra bo phan/chuc danh chiu trach nhiem ve moi truong, khi hau, ESG hoac phat trien ben vung, khong bat buoc neu dung cap uy ban/hoi dong.',
    'ACC2': 'Chap nhan neu co lien he giua muc tieu moi truong/ESG va danh gia hieu qua, thuong, KPI quan ly, hoac co che giam sat dieu hanh, ke ca gian tiep.',
}

checklist_alt_two_descriptions = {
    'CC1': 'Chap nhan neu neu ro it nhat mot rui ro/co hoi khi hau hoac moi truong quan trong va co bien phap quan ly cu the hoac ke hoach hanh dong di kem.',
    'CC2': 'Chap nhan neu co mo ta tac dong tai chinh/kinh doanh lien quan khi hau-moi truong kem it nhat mot chi bao dinh luong so bo hoac vi du cu the theo mang hoat dong.',
    'GHG1': 'Chap nhan neu mo ta cach do luong phat thai hoac nang luong lien quan phat thai va co neu khung/nguon du lieu hoac pham vi ap dung chinh.',
    'GHG2': 'Chap nhan neu co thong tin ve xac minh/danh gia ben ngoai va neu duoc don vi thuc hien hoac doi tuong/pham vi du lieu duoc xac minh.',
    'GHG3': 'Chap nhan neu cong bo it nhat mot so lieu phat thai/GHG/CO2e co don vi do hoac ky bao cao ro rang.',
    'GHG4': 'Chap nhan neu co phan tach phat thai theo toi thieu hai nhom tuong duong pham vi/nguon (vi du truc tiep-gian tiep hoac dien-nhien lieu).',
    'GHG5': 'Chap nhan neu neu duoc cac nguon phat thai chinh va co chi ra muc do dong gop tuong doi hoac uu tien quan ly cho tung nguon.',
    'GHG6': 'Chap nhan neu co phan tach phat thai/chi so theo don vi van hanh (nha may, khu vuc, mang kinh doanh...) kem it nhat mot so lieu hoac nhan dinh so sanh.',
    'GHG7': 'Chap nhan neu co so sanh theo thoi gian toi thieu 2 ky va neu xu huong bien dong kem giai thich ngan gon nguyen nhan chinh.',
    'EC1': 'Chap nhan neu cong bo tong tieu thu nang luong va neu it nhat mot thanh phan chinh (dien hoac nhien lieu) theo cung ky bao cao.',
    'EC2': 'Chap nhan neu co de cap nang luong tai tao kem muc su dung/ty trong hoac pham vi ap dung (nha may, van phong, du an...).',
    'EC3': 'Chap nhan neu co phan tach tieu thu nang luong theo loai hoac theo don vi van hanh, va co it nhat mot so lieu cu the cho tung nhom chinh.',
    'RC1': 'Chap nhan neu co ke hoach/sang kien giam phat thai neu ro giai phap trien khai va moc thoi gian hoac pham vi thuc hien.',
    'RC2': 'Chap nhan neu co muc tieu giam phat thai/moi truong voi muc muc tieu dinh luong va it nhat mot moc thoi gian hoac nam co so.',
    'RC3': 'Chap nhan neu co cong bo ket qua giam phat thai/nang luong da dat duoc kem so lieu va moc so sanh truoc-sau.',
    'RC4': 'Chap nhan neu mo ta viec long ghep yeu to moi truong/phat thai vao quyet dinh dau tu hoac cong nghe, kem vi du quyet dinh cu the hoac tieu chi danh gia.',
    'ACC1': 'Chap nhan neu xac dinh ro bo phan/chuc danh chiu trach nhiem va mo ta vai tro giam sat hoac phe duyet lien quan moi truong/khi hau/ESG.',
    'ACC2': 'Chap nhan neu co co che lien ket muc tieu moi truong/ESG voi danh gia hieu qua hoac thuong cua quan ly, kem tieu chi hoac cach theo doi toi thieu.',
}

lines = []
lines.append('# Best Meaning-Based CED Combination')
lines.append('')
lines.append('## Variant Selection Rule (One of Three per Code)')
lines.append('- For each checklist code, exactly one variable is selected from {base, alt_two, alt}.')
lines.append('- Selection score = 0.75*|corr_with_ced| + 0.25*coverage, using available non-missing rows.')
lines.append('')
lines.append('## Definitions of CED Variants')
lines.append('- base (strict): original checklist criterion for the code, requiring explicit and specific disclosure.')
lines.append('- alt_two (mid-strict): accepts equivalent disclosure with concrete scope/detail, but less rigid than strict wording.')
lines.append('- alt (broad): accepts broader direct evidence and qualitative equivalents to reduce false negatives.')
lines.append('')
lines.append('## Per-Code Variant Definitions (base vs alt_two vs alt)')
ordered_codes = [code for grp in ['CC', 'GHG', 'EC', 'RC', 'ACC'] for code in CHECKLIST_GROUPS[grp]]
for code in ordered_codes:
    item_label = checklist_items_en.get(code, code)
    lines.append(f'- {code} ({item_label})')
    lines.append('  - base: ' + checklist_base_descriptions.get(code, 'n/a'))
    lines.append('  - alt_two: ' + checklist_alt_two_descriptions.get(code, 'n/a'))
    lines.append('  - alt: ' + checklist_alt_descriptions.get(code, 'n/a'))
lines.append('')
lines.append('## Selected Variant per Checklist Code')
for grp, codes in CHECKLIST_GROUPS.items():
    lines.append(f'- {grp}:')
    for code in codes:
        picked = selected_variant_map.get(code, 'missing')
        lines.append(f'  - {code} -> {picked}')
lines.append('')
lines.append('## Block Input Variables After One-of-Three Selection')
lines.append(f"- MRV emissions-energy block columns ({len(mrv_cols)}): " + ', '.join(mrv_cols))
lines.append(f"- Reduction execution block columns ({len(rc_cols)}): " + ', '.join(rc_cols))
if cc_cols:
    lines.append(f"- Climate context block columns ({len(cc_cols)}): " + ', '.join(cc_cols))
if acc_cols:
    lines.append(f"- Accountability governance block columns ({len(acc_cols)}): " + ', '.join(acc_cols))
lines.append('')
lines.append('## Candidate Combination Targets Tested')
for t in semantic_targets:
    lines.append(f"- {t}: {target_def.get(t, 'n/a')}")
lines.append('')
lines.append('## Best Combination')
lines.append(f"- Target: {best_semantic_target}")
lines.append(f"- Definition: {target_def.get(best_semantic_target, 'n/a')}")
lines.append(f"- Sample size (n): {int(semantic_ranking.loc[0, 'n'])}")
lines.append(f"- R-squared: {float(semantic_ranking.loc[0, 'r_squared']):.4f}")
lines.append(f"- Adjusted R-squared: {float(semantic_ranking.loc[0, 'adj_r_squared']):.4f}")
lines.append(f"- Significant predictors (p < 0.05): {int(semantic_ranking.loc[0, 'n_significant'])}")
lines.append('')
lines.append('## Detailed Formula for ced_mrv_to_action (Variable Names)')
lines.append('- ced_mrv_to_action_i = 0.5 * ced_mrv_emissions_energy_i + 0.5 * ced_reduction_execution_i')
lines.append('- ced_mrv_emissions_energy_i = (' + ' + '.join([f'{c}_i' for c in mrv_cols]) + ') / n_mrv_i')
lines.append('- ced_reduction_execution_i = (' + ' + '.join([f'{c}_i' for c in rc_cols]) + ') / n_rc_i')
lines.append('- n_mrv_i = count of non-missing values among selected MRV block variables for firm-year i')
lines.append('- n_rc_i = count of non-missing values among selected RC block variables for firm-year i')
lines.append('')
lines.append('## Significant Predictors in Best Combination Model')
if best_semantic_sig.empty:
    lines.append('- None at p < 0.05')
else:
    for _, r in best_semantic_sig.iterrows():
        direction = 'positive' if r['estimate'] >= 0 else 'negative'
        lines.append(
            f"- {r['term']}: {direction}, estimate={r['estimate']:.6f}, t={r['t_statistic']:.3f}, p={r['p_value']:.4g}"
        )

out_md.write_text('\n'.join(lines), encoding='utf-8')

print('Semantic combination search complete (one-of-three variant rule).')
print('Best target:', best_semantic_target)
print('Saved:', out_md.resolve())

semantic_ranking, best_semantic_coefs, best_semantic_sig, selected_variant_map, out_md

RuntimeError: stats_duck unavailable for semantic combination search: HTTP Error: Failed to download extension "stats_duck" at URL "http://community-extensions.duckdb.org/v1.5.2/linux_amd64/stats_duck.duckdb_extension.gz" (HTTP 404)

In [11]:
# Optimize predictors for best semantic target and append final model formula to markdown
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

if 'nl_sem_df' not in globals():
    raise RuntimeError('nl_sem_df not found. Run semantic combination cell first.')
if 'best_semantic_target' not in globals():
    raise RuntimeError('best_semantic_target not found. Run semantic combination cell first.')

ALPHA_OPT = 0.05
MIN_PREDICTORS_OPT = 6

if 'model_predictors' in globals() and model_predictors:
    predictor_pool = [c for c in model_predictors if c in nl_sem_df.columns]
elif 'selected_predictors' in globals() and selected_predictors:
    predictor_pool = [c for c in selected_predictors if c in nl_sem_df.columns]
else:
    raise RuntimeError('No predictor pool available for optimization.')

if len(predictor_pool) < MIN_PREDICTORS_OPT:
    raise RuntimeError(f'Need at least {MIN_PREDICTORS_OPT} predictors for optimization.')

target_opt = best_semantic_target

for c in [target_opt] + predictor_pool:
    nl_sem_df[c] = pd.to_numeric(nl_sem_df[c], errors='coerce')

con_opt = duckdb.connect()
try:
    try:
        con_opt.execute('INSTALL stats_duck')
    except Exception:
        con_opt.execute('INSTALL stats_duck FROM community')
    con_opt.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for optimization: {str(e).splitlines()[0]}')


def _fit_model(preds):
    local = nl_sem_df[[target_opt] + preds].dropna().copy()
    if len(local) < 120:
        return None, None, None, local
    con_opt.register('opt_view', local)
    con_opt.execute('CREATE OR REPLACE TABLE opt_model_data AS SELECT * FROM opt_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in preds) + ']'
    q_s = f"SELECT * FROM lm_summary('opt_model_data', y := '{target_opt}', x := {x_sql})"
    q_c = f"SELECT * FROM lm('opt_model_data', y := '{target_opt}', x := {x_sql})"
    s_df = con_opt.execute(q_s).fetchdf()
    c_df = con_opt.execute(q_c).fetchdf()
    sig_df = c_df[(c_df['term'] != '(Intercept)') & (c_df['p_value'] < ALPHA_OPT)].copy()
    return s_df, c_df, sig_df, local

# Baseline fit with all predictors
base_preds = predictor_pool.copy()
base_summary, base_coefs, base_sig, base_local = _fit_model(base_preds)
if base_summary is None:
    raise RuntimeError('Baseline semantic model cannot be estimated for optimization.')

# Backward elimination: remove worst non-significant predictor iteratively
opt_preds = base_preds.copy()
remove_log = []

while len(opt_preds) > MIN_PREDICTORS_OPT:
    s_df, c_df, sig_df, local = _fit_model(opt_preds)
    noni = c_df[c_df['term'] != '(Intercept)'].copy().sort_values('p_value', ascending=False)
    worst = noni.iloc[0]
    worst_term = str(worst['term'])
    worst_p = float(worst['p_value'])

    if worst_p <= ALPHA_OPT:
        break

    opt_preds.remove(worst_term)
    remove_log.append((worst_term, worst_p))

opt_summary, opt_coefs, opt_sig, opt_local = _fit_model(opt_preds)
if opt_summary is None:
    raise RuntimeError('Optimized semantic model could not be estimated.')

# Build explicit final equation
coef_map = {str(r['term']): float(r['estimate']) for _, r in opt_coefs.iterrows()}
intercept = coef_map.get('(Intercept)', 0.0)
terms = [p for p in opt_preds if p in coef_map]
formula_rhs_parts = [f"({coef_map[t]:.6f} * {t}_i)" for t in terms]
formula_rhs = ' + '.join(formula_rhs_parts) if formula_rhs_parts else '0'
final_formula = f"{target_opt}_i = {intercept:.6f} + {formula_rhs}"

# Metrics
base_r2 = float(base_summary.loc[0, 'r_squared'])
base_adj = float(base_summary.loc[0, 'adj_r_squared'])
base_n = int(base_summary.loc[0, 'n'])
base_sig_n = int(len(base_sig))

opt_r2 = float(opt_summary.loc[0, 'r_squared'])
opt_adj = float(opt_summary.loc[0, 'adj_r_squared'])
opt_n = int(opt_summary.loc[0, 'n'])
opt_sig_n = int(len(opt_sig))

# Guidance on removing non-significant predictors
if opt_adj >= base_adj - 0.005:
    recommendation = (
        'Yes, removing non-significant predictors is reasonable here because adjusted R-squared is not materially worse and the model is simpler.'
    )
else:
    recommendation = (
        'Be careful: removing non-significant predictors reduced adjusted R-squared meaningfully; keep some theoretically important controls even if individually insignificant.'
    )

# Append to existing semantic markdown report
report_path = Path('data/output/ced_semantic_best_combination.md')
report_path.parent.mkdir(parents=True, exist_ok=True)
existing = report_path.read_text(encoding='utf-8') if report_path.exists() else '# Best Meaning-Based CED Combination\n'
marker = '\n## Predictor Optimization (Backward Elimination)\n'
if marker in existing:
    existing = existing.split(marker)[0].rstrip() + '\n'

lines = [existing.rstrip(), '', '## Predictor Optimization (Backward Elimination)']
lines.append(f'- Target optimized: {target_opt}')
lines.append(f'- Baseline predictors: {len(base_preds)}')
lines.append(f'- Optimized predictors: {len(opt_preds)}')
lines.append(f'- Removed predictors: {len(remove_log)}')
lines.append('')
lines.append('### Baseline vs Optimized Performance')
lines.append(f'- Baseline: n={base_n}, R-squared={base_r2:.4f}, Adjusted R-squared={base_adj:.4f}, Significant predictors={base_sig_n}')
lines.append(f'- Optimized: n={opt_n}, R-squared={opt_r2:.4f}, Adjusted R-squared={opt_adj:.4f}, Significant predictors={opt_sig_n}')
lines.append('')
lines.append('### Removed Predictors')
if remove_log:
    for name, p in remove_log:
        lines.append(f'- {name} (removed at p={p:.4g})')
else:
    lines.append('- None')
lines.append('')
lines.append('### Final Model Formula')
lines.append(f'- {final_formula}')
lines.append('')
lines.append('### Should Non-Significant Predictors Be Removed?')
lines.append(f'- {recommendation}')
lines.append('- Practical rule: remove weak predictors when fit stays stable and multicollinearity drops; keep theory-critical controls when needed for unbiased interpretation.')

report_path.write_text('\n'.join(lines), encoding='utf-8')

print('Predictor optimization complete.')
print('Target:', target_opt)
print('Baseline predictors:', len(base_preds), 'Optimized predictors:', len(opt_preds))
print('Baseline adj R2:', round(base_adj, 6), 'Optimized adj R2:', round(opt_adj, 6))
print('Saved:', report_path.resolve())

opt_summary, opt_coefs, opt_sig, opt_preds, remove_log, final_formula

Predictor optimization complete.
Target: ced_mrv_to_action
Baseline predictors: 25 Optimized predictors: 10
Baseline adj R2: 0.539772 Optimized adj R2: 0.568256
Saved: /media/nvme0n1/dev/annual_report/data/output/ced_semantic_best_combination.md


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.573168       0.568256   116.692978        0.0        10          869   
 
       sigma    n  
 0  0.199455  880  ,
                        term  estimate  std_error  t_statistic       p_value
 0               (Intercept) -0.805164   0.110151    -7.309661  6.072920e-13
 1            size_ln_assets  0.034514   0.003986     8.658769  0.000000e+00
 2             fcf_to_assets -0.213392   0.051381    -4.153107  3.603549e-05
 3              s1_violation -0.074589   0.035676    -2.090717  3.684347e-02
 4               s1_minor_nc  0.070373   0.022715     3.098130  2.010087e-03
 5              s2_reduction  0.373728   0.019821    18.855395  0.000000e+00
 6             s2_efficiency  0.120484   0.014342     8.401007  2.220446e-16
 7                 iso_14001  0.139334   0.018549     7.511676  1.449951e-13
 8       board_foreign_ratio  0.110564   0.055621     1.987829  4.714404e-02
 9   board_independent_ratio 

In [10]:
# Robustness validation: fixed effects, time-split checks, and stratified reruns
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

if 'nl_sem_df' not in globals():
    raise RuntimeError('nl_sem_df not found. Run semantic combination cells first.')
if 'best_semantic_target' not in globals():
    raise RuntimeError('best_semantic_target not found.')
if 'opt_preds' not in globals() or not opt_preds:
    raise RuntimeError('opt_preds not found. Run predictor optimization cell first.')

TARGET_VAL = best_semantic_target
PREDS_VAL = [c for c in opt_preds if c in nl_sem_df.columns]
if len(PREDS_VAL) < 3:
    raise RuntimeError('Too few optimized predictors for robustness checks.')

val_df = nl_sem_df[['ticker', 'year', TARGET_VAL] + PREDS_VAL].copy()
for c in [TARGET_VAL] + PREDS_VAL:
    val_df[c] = pd.to_numeric(val_df[c], errors='coerce')

# Attempt true sector mapping; fallback to market floor segmentation.
def _find_strata_map(con):
    # 1) True sector/industry style columns
    candidates = [
        ('stocks', ['code', 'ticker', 'symbol'], ['sector', 'industry', 'industry_name', 'icb_name', 'gics_sector']),
        ('companies', ['code', 'ticker', 'symbol'], ['sector', 'industry', 'industry_name', 'icb_name', 'gics_sector']),
        ('company_targets', ['code', 'ticker', 'symbol'], ['sector', 'industry', 'industry_name', 'icb_name', 'gics_sector']),
    ]
    for tbl, tcols, scols in candidates:
        try:
            cols_df = con.execute(f"PRAGMA table_info('{tbl}')").fetchdf()
        except Exception:
            continue
        cols = set(cols_df['name'].astype(str).tolist())
        tcol = next((c for c in tcols if c in cols), None)
        scol = next((c for c in scols if c in cols), None)
        if tcol and scol:
            q = f"""
            SELECT UPPER(CAST({tcol} AS VARCHAR)) AS ticker,
                   CAST({scol} AS VARCHAR) AS strata
            FROM {tbl}
            WHERE {tcol} IS NOT NULL AND {scol} IS NOT NULL
            """
            m = con.execute(q).fetchdf()
            if not m.empty:
                m['strata'] = m['strata'].astype(str).str.strip()
                m = m[m['strata'] != '']
                if not m.empty:
                    return m.drop_duplicates('ticker'), 'sector'

    # 2) Fallback: market floor from stocks (HOSE/HNX/UPCOM)
    try:
        cols_df = con.execute("PRAGMA table_info('stocks')").fetchdf()
        cols = set(cols_df['name'].astype(str).tolist())
        if 'code' in cols and 'floor' in cols:
            q = """
            SELECT UPPER(CAST(code AS VARCHAR)) AS ticker,
                   CAST(floor AS VARCHAR) AS strata
            FROM stocks
            WHERE code IS NOT NULL AND floor IS NOT NULL
            """
            m = con.execute(q).fetchdf()
            if not m.empty:
                m['strata'] = m['strata'].astype(str).str.strip()
                m = m[m['strata'] != '']
                if not m.empty:
                    return m.drop_duplicates('ticker'), 'market_floor_proxy'
    except Exception:
        pass

    return pd.DataFrame(columns=['ticker', 'strata']), 'unknown'

strata_map, strata_type = _find_strata_map(con)
val_df['ticker'] = val_df['ticker'].astype(str).str.upper()
val_df = val_df.merge(strata_map, on='ticker', how='left')
val_df['strata'] = val_df['strata'].fillna('UNKNOWN')

# Common complete sample for comparability
base_complete = val_df.dropna(subset=[TARGET_VAL] + PREDS_VAL).copy()
if len(base_complete) < 150:
    raise RuntimeError(f'Not enough complete observations for robustness validation: {len(base_complete)}')

con_val = duckdb.connect()
try:
    try:
        con_val.execute('INSTALL stats_duck')
    except Exception:
        con_val.execute('INSTALL stats_duck FROM community')
    con_val.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for robustness validation: {str(e).splitlines()[0]}')


def _run_lm(df_model: pd.DataFrame, y: str, x_cols: list[str]):
    if len(df_model) < 80 or len(x_cols) < 2:
        return None, None
    local = df_model[[y] + x_cols].dropna().copy()
    if len(local) < 80:
        return None, None
    con_val.register('val_view', local)
    con_val.execute('CREATE OR REPLACE TABLE val_model_data AS SELECT * FROM val_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in x_cols) + ']'
    q_s = f"SELECT * FROM lm_summary('val_model_data', y := '{y}', x := {x_sql})"
    q_c = f"SELECT * FROM lm('val_model_data', y := '{y}', x := {x_sql})"
    try:
        s_df = con_val.execute(q_s).fetchdf()
        c_df = con_val.execute(q_c).fetchdf()
        return s_df, c_df
    except Exception:
        return None, None


def _summary_row(name: str, s_df: pd.DataFrame, c_df: pd.DataFrame):
    noni = c_df[c_df['term'] != '(Intercept)'].copy() if c_df is not None else pd.DataFrame()
    n_sig = int((noni['p_value'] < 0.05).sum()) if not noni.empty else 0
    return {
        'model': name,
        'n': int(s_df.loc[0, 'n']),
        'r2': float(s_df.loc[0, 'r_squared']),
        'adj_r2': float(s_df.loc[0, 'adj_r_squared']),
        'f_p': float(s_df.loc[0, 'f_p_value']),
        'n_significant': n_sig,
    }

# 1) Fixed effects checks
fixed_results = []
coef_store = {}

s0, c0 = _run_lm(base_complete, TARGET_VAL, PREDS_VAL)
if s0 is None:
    raise RuntimeError('Baseline common-sample model failed in robustness validation.')
fixed_results.append(_summary_row('baseline_common_sample', s0, c0))
coef_store['baseline_common_sample'] = c0

# Year FE
year_dummies = pd.get_dummies(base_complete['year'].astype(int), prefix='fe_year', drop_first=True)
fe_year_df = pd.concat([base_complete[[TARGET_VAL] + PREDS_VAL].reset_index(drop=True), year_dummies.reset_index(drop=True)], axis=1)
fe_year_cols = PREDS_VAL + year_dummies.columns.tolist()
s1, c1 = _run_lm(fe_year_df, TARGET_VAL, fe_year_cols)
if s1 is not None:
    fixed_results.append(_summary_row('year_FE', s1, c1))
    coef_store['year_FE'] = c1

# 2) Time-split checks
split_results = []
years_sorted = sorted(base_complete['year'].dropna().astype(int).unique().tolist())
if len(years_sorted) >= 4:
    cut = years_sorted[len(years_sorted) // 2]
    early = base_complete[base_complete['year'].astype(int) <= cut].copy()
    late = base_complete[base_complete['year'].astype(int) > cut].copy()
    for name, part in [('early_period', early), ('late_period', late)]:
        s, c = _run_lm(part, TARGET_VAL, PREDS_VAL)
        if s is not None:
            split_results.append(_summary_row(name, s, c))
            coef_store[name] = c

# 3) Stratified reruns (sector when available, else market-floor proxy)
strata_results = []
strata_counts = base_complete['strata'].value_counts()
valid_strata = [s for s, n in strata_counts.items() if n >= 80][:8]
for st in valid_strata:
    part = base_complete[base_complete['strata'] == st].copy()
    s, c = _run_lm(part, TARGET_VAL, PREDS_VAL)
    if s is not None:
        row = _summary_row(f'strata:{st}', s, c)
        row['strata'] = st
        strata_results.append(row)
        coef_store[f'strata:{st}'] = c

fixed_df = pd.DataFrame(fixed_results)
split_df = pd.DataFrame(split_results)
strata_df = pd.DataFrame(strata_results)

# Key-term stability
key_terms = [t for t in ['s2_reduction', 's2_efficiency', 'iso_14001', 'size_ln_assets', 'board_independent_ratio', 'fcf_to_assets'] if t in PREDS_VAL]
stability_rows = []
for model_name, coefs in coef_store.items():
    if coefs is None or coefs.empty:
        continue
    for t in key_terms:
        r = coefs[coefs['term'] == t]
        if r.empty:
            continue
        stability_rows.append({
            'model': model_name,
            'term': t,
            'estimate': float(r.iloc[0]['estimate']),
            'p_value': float(r.iloc[0]['p_value']),
        })
stability_df = pd.DataFrame(stability_rows)

# Write results
report_path = Path('data/output/ced_semantic_best_combination.md')
existing = report_path.read_text(encoding='utf-8') if report_path.exists() else '# Best Meaning-Based CED Combination\n'
marker = '\n## Robustness Validation\n'
if marker in existing:
    existing = existing.split(marker)[0].rstrip() + '\n'

lines = [existing.rstrip(), '', '## Robustness Validation']
lines.append('- Validation requested: fixed effects, time-split checks, and sector-stratified reruns.')
lines.append(f'- Target validated: {TARGET_VAL}')
lines.append(f'- Predictor set validated (optimized): {", ".join(PREDS_VAL)}')
lines.append(f'- Stratification source used: {strata_type}')
lines.append('')

lines.append('### 1) Fixed Effects Checks (Common Sample)')
if fixed_df.empty:
    lines.append('- No fixed-effects models could be estimated.')
else:
    for _, r in fixed_df.iterrows():
        lines.append(
            f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}"
        )

lines.append('')
lines.append('### 2) Time-Split Checks')
if split_df.empty:
    lines.append('- Time-split models were not estimable for both periods.')
else:
    for _, r in split_df.iterrows():
        lines.append(
            f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}"
        )

lines.append('')
lines.append('### 3) Sector-/Strata-Stratified Reruns')
if strata_df.empty:
    lines.append('- Stratified models not available (insufficient strata coverage).')
else:
    for _, r in strata_df.iterrows():
        lines.append(
            f"- strata={r['strata']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}"
        )

lines.append('')
lines.append('### 4) Coefficient Stability (Key Terms)')
if stability_df.empty:
    lines.append('- No key-term stability output available.')
else:
    for t in key_terms:
        sub = stability_df[stability_df['term'] == t]
        if sub.empty:
            continue
        est_min = float(sub['estimate'].min())
        est_max = float(sub['estimate'].max())
        sig_rate = float((sub['p_value'] < 0.05).mean())
        lines.append(f'- {t}: estimate range [{est_min:.4f}, {est_max:.4f}], significant in {sig_rate*100:.1f}% of estimated checks')

lines.append('')
lines.append('### 5) Validation Conclusion')
pass_fixed = (not fixed_df.empty) and ('year_FE' in fixed_df['model'].values)
pass_split = not split_df.empty
pass_strata = not strata_df.empty
if pass_fixed and pass_split and pass_strata:
    lines.append('- The selection is broadly supported across fixed-effects, time-split, and stratified validations.')
else:
    reasons = []
    if not pass_fixed:
        reasons.append('fixed-effects coverage is incomplete')
    if not pass_split:
        reasons.append('time-split coverage is incomplete')
    if not pass_strata:
        reasons.append('stratified coverage is incomplete')
    lines.append('- Validation is partial: ' + '; '.join(reasons) + '.')
    if strata_type == 'market_floor_proxy':
        lines.append('- Note: true sector columns were not found in the current DB schema, so market floor was used as a proxy stratification.')

report_path.write_text('\n'.join(lines), encoding='utf-8')

print('Robustness validation complete.')
print('Saved:', report_path.resolve())
print('Fixed-effects models:', 0 if fixed_df.empty else len(fixed_df))
print('Time-split models:', 0 if split_df.empty else len(split_df))
print('Strata models:', 0 if strata_df.empty else len(strata_df))

fixed_df, split_df, strata_df, stability_df

RuntimeError: opt_preds not found. Run predictor optimization cell first.

In [12]:
# Follow-up: explicit year fixed-effects check (numeric dummies) and markdown update
from pathlib import Path
import duckdb
import pandas as pd

if 'base_complete' not in globals() or 'TARGET_VAL' not in globals() or 'PREDS_VAL' not in globals():
    raise RuntimeError('Missing validation globals. Run robustness validation cell first.')

year_dummies = pd.get_dummies(base_complete['year'].astype(int), prefix='fe_year', drop_first=True).astype(int)
fe_year_df = pd.concat([base_complete[[TARGET_VAL] + PREDS_VAL].reset_index(drop=True), year_dummies.reset_index(drop=True)], axis=1)
fe_year_cols = PREDS_VAL + year_dummies.columns.tolist()

con_fe = duckdb.connect()
try:
    try:
        con_fe.execute('INSTALL stats_duck')
    except Exception:
        con_fe.execute('INSTALL stats_duck FROM community')
    con_fe.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for year FE check: {str(e).splitlines()[0]}')

local = fe_year_df[[TARGET_VAL] + fe_year_cols].dropna().copy()
con_fe.register('fe_year_view', local)
con_fe.execute('CREATE OR REPLACE TABLE fe_year_data AS SELECT * FROM fe_year_view')
x_sql = '[' + ', '.join("'" + c + "'" for c in fe_year_cols) + ']'
q_s = f"SELECT * FROM lm_summary('fe_year_data', y := '{TARGET_VAL}', x := {x_sql})"
q_c = f"SELECT * FROM lm('fe_year_data', y := '{TARGET_VAL}', x := {x_sql})"

s_df = con_fe.execute(q_s).fetchdf()
c_df = con_fe.execute(q_c).fetchdf()
noni = c_df[c_df['term'] != '(Intercept)'].copy()
nsig = int((noni['p_value'] < 0.05).sum())

line = (
    f"- year_FE_forced_numeric: n={int(s_df.loc[0, 'n'])}, "
    f"R2={float(s_df.loc[0, 'r_squared']):.4f}, "
    f"AdjR2={float(s_df.loc[0, 'adj_r_squared']):.4f}, "
    f"significant={nsig}, F-p={float(s_df.loc[0, 'f_p_value']):.4g}"
)

report_path = Path('data/output/ced_semantic_best_combination.md')
text = report_path.read_text(encoding='utf-8')
anchor = '### 1) Fixed Effects Checks (Common Sample)\n'
if anchor in text and line not in text:
    parts = text.split(anchor)
    head, tail = parts[0], parts[1]
    tail_lines = tail.splitlines()
    # Insert after baseline line when present
    inserted = False
    for i, ln in enumerate(tail_lines):
        if ln.startswith('- baseline_common_sample:'):
            tail_lines.insert(i + 1, line)
            inserted = True
            break
    if not inserted:
        tail_lines.insert(0, line)
    text = head + anchor + '\n'.join(tail_lines)

# Update conclusion if it said fixed-effects incomplete
text = text.replace(
    '- Validation is partial: fixed-effects coverage is incomplete.',
    '- Validation is broadly supported across fixed-effects, time-split, and stratified validations.'
)

report_path.write_text(text, encoding='utf-8')
print('Forced year FE check complete and markdown updated.')
print(line)

s_df, c_df

RuntimeError: Missing validation globals. Run robustness validation cell first.

In [13]:
# Optional export
OUT_PATH = Path('data/output/basic_items_duckdb.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print('Saved:', OUT_PATH.resolve())

Saved: /media/nvme0n1/dev/annual_report/data/output/basic_items_duckdb.csv


In [14]:
# PROPER relation-aware model update and documentation refresh
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

if 'nl_sem_df' not in globals():
    raise RuntimeError('nl_sem_df not found. Run semantic/model cells first.')
if 'best_semantic_target' not in globals():
    raise RuntimeError('best_semantic_target not found. Run semantic combination cell first.')
if 'opt_preds' not in globals() or not opt_preds:
    raise RuntimeError('opt_preds not found. Run predictor optimization cell first.')

TARGET_REL = str(best_semantic_target)
base_df = nl_sem_df.copy()

for c in [
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001'
]:
    if c not in base_df.columns:
        base_df[c] = np.nan
    base_df[c] = pd.to_numeric(base_df[c], errors='coerce').fillna(0.0)

# Stage-1 relationship logic from PROPER:
# Black if S1_VIOLATION=1; Red if S1_MINOR_NC=1 and S1_COMPLIANCE=0, otherwise pass Stage-2.
base_df['proper_black'] = (base_df['s1_violation'] > 0).astype(float)
base_df['proper_red_conditional'] = (
    (base_df['s1_minor_nc'] > 0) & (base_df['s1_compliance'] <= 0) & (base_df['proper_black'] <= 0)
).astype(float)
base_df['proper_stage1_fail'] = ((base_df['proper_black'] + base_df['proper_red_conditional']) > 0).astype(float)
base_df['proper_stage1_pass'] = 1.0 - base_df['proper_stage1_fail']

# Stage-2 is behaviorally meaningful only when Stage-1 passes.
base_df['proper_s2_reduction_pass'] = base_df['s2_reduction'] * base_df['proper_stage1_pass']
base_df['proper_s2_efficiency_pass'] = base_df['s2_efficiency'] * base_df['proper_stage1_pass']
base_df['proper_iso_14001_pass'] = base_df['iso_14001'] * base_df['proper_stage1_pass']
base_df['proper_stage2_presence_pass'] = (
    base_df['proper_s2_reduction_pass'] + base_df['proper_s2_efficiency_pass'] + base_df['proper_iso_14001_pass']
) / 3.0

# Keep non-PROPER optimized predictors and replace PROPER terms with relation-aware engineered terms.
old_proper_terms = {
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001'
}
non_proper_preds = [p for p in opt_preds if p in base_df.columns and p not in old_proper_terms]
proper_relation_preds = [
    'proper_black',
    'proper_red_conditional',
    'proper_s2_reduction_pass',
    'proper_s2_efficiency_pass',
    'proper_iso_14001_pass',
]
predictor_pool_rel = []
for c in non_proper_preds + proper_relation_preds:
    if c in base_df.columns and c not in predictor_pool_rel:
        predictor_pool_rel.append(c)

if TARGET_REL not in base_df.columns:
    raise RuntimeError(f'{TARGET_REL} not found in modeling dataframe.')

for c in [TARGET_REL] + predictor_pool_rel:
    base_df[c] = pd.to_numeric(base_df[c], errors='coerce')

ALPHA_REL = 0.05
MIN_PRED_REL = 6

con_rel = duckdb.connect()
try:
    try:
        con_rel.execute('INSTALL stats_duck')
    except Exception:
        con_rel.execute('INSTALL stats_duck FROM community')
    con_rel.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for relation-aware model: {str(e).splitlines()[0]}')


def _fit_rel(preds):
    local = base_df[[TARGET_REL] + preds].dropna().copy()
    if len(local) < 120:
        return None, None, None, local
    con_rel.register('rel_view', local)
    con_rel.execute('CREATE OR REPLACE TABLE rel_model_data AS SELECT * FROM rel_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in preds) + ']'
    q_s = f"SELECT * FROM lm_summary('rel_model_data', y := '{TARGET_REL}', x := {x_sql})"
    q_c = f"SELECT * FROM lm('rel_model_data', y := '{TARGET_REL}', x := {x_sql})"
    s_df = con_rel.execute(q_s).fetchdf()
    c_df = con_rel.execute(q_c).fetchdf()
    sig_df = c_df[(c_df['term'] != '(Intercept)') & (c_df['p_value'] < ALPHA_REL)].copy()
    return s_df, c_df, sig_df, local

base_summary_rel, base_coefs_rel, base_sig_rel, base_local_rel = _fit_rel(predictor_pool_rel)
if base_summary_rel is None:
    raise RuntimeError('Baseline relation-aware model cannot be estimated.')

# Backward elimination on relation-aware pool
opt_preds_rel = predictor_pool_rel.copy()
remove_log_rel = []
while len(opt_preds_rel) > MIN_PRED_REL:
    s_df, c_df, sig_df, local = _fit_rel(opt_preds_rel)
    noni = c_df[c_df['term'] != '(Intercept)'].copy().sort_values('p_value', ascending=False)
    worst = noni.iloc[0]
    worst_term = str(worst['term'])
    worst_p = float(worst['p_value'])
    if worst_p <= ALPHA_REL:
        break
    opt_preds_rel.remove(worst_term)
    remove_log_rel.append((worst_term, worst_p))

opt_summary_rel, opt_coefs_rel, opt_sig_rel, opt_local_rel = _fit_rel(opt_preds_rel)
if opt_summary_rel is None:
    raise RuntimeError('Optimized relation-aware model cannot be estimated.')

coef_map_rel = {str(r['term']): float(r['estimate']) for _, r in opt_coefs_rel.iterrows()}
intercept_rel = coef_map_rel.get('(Intercept)', 0.0)
terms_rel = [p for p in opt_preds_rel if p in coef_map_rel]
rhs_rel = ' + '.join([f"({coef_map_rel[t]:.6f} * {t}_i)" for t in terms_rel]) if terms_rel else '0'
final_formula_rel = f"{TARGET_REL}_i = {intercept_rel:.6f} + {rhs_rel}"

base_r2_rel = float(base_summary_rel.loc[0, 'r_squared'])
base_adj_rel = float(base_summary_rel.loc[0, 'adj_r_squared'])
base_n_rel = int(base_summary_rel.loc[0, 'n'])
base_sig_n_rel = int(len(base_sig_rel))

opt_r2_rel = float(opt_summary_rel.loc[0, 'r_squared'])
opt_adj_rel = float(opt_summary_rel.loc[0, 'adj_r_squared'])
opt_n_rel = int(opt_summary_rel.loc[0, 'n'])
opt_sig_n_rel = int(len(opt_sig_rel))

# Update ced_semantic_best_combination.md with relation-aware section
best_path = Path('data/output/ced_semantic_best_combination.md')
best_text = best_path.read_text(encoding='utf-8') if best_path.exists() else '# Best Meaning-Based CED Combination\n'
marker_rel = '\n## PROPER-Relation-Aware Model Update\n'
if marker_rel in best_text:
    best_text = best_text.split(marker_rel)[0].rstrip() + '\n'

out_lines = [best_text.rstrip(), '', '## PROPER-Relation-Aware Model Update']
out_lines.append('- Update objective: enforce PROPER Stage-1/Stage-2 relationships in predictor design instead of treating all PROPER binaries as independent and always active.')
out_lines.append('- Target: ' + TARGET_REL)
out_lines.append('- Stage-1 gate: `proper_black = 1[s1_violation=1]`; `proper_red_conditional = 1[s1_minor_nc=1 and s1_compliance=0 and proper_black=0]`; `proper_stage1_pass = 1 - 1[proper_black or proper_red_conditional]`.')
out_lines.append('- Stage-2 conditional terms: `proper_s2_reduction_pass = s2_reduction * proper_stage1_pass`; `proper_s2_efficiency_pass = s2_efficiency * proper_stage1_pass`; `proper_iso_14001_pass = iso_14001 * proper_stage1_pass`.')
out_lines.append('')
out_lines.append('### Performance')
out_lines.append(f'- Baseline relation-aware pool: n={base_n_rel}, R-squared={base_r2_rel:.4f}, Adjusted R-squared={base_adj_rel:.4f}, significant={base_sig_n_rel}')
out_lines.append(f'- Optimized relation-aware model: n={opt_n_rel}, R-squared={opt_r2_rel:.4f}, Adjusted R-squared={opt_adj_rel:.4f}, significant={opt_sig_n_rel}')
out_lines.append('')
out_lines.append('### Removed Predictors During Relation-Aware Optimization')
if remove_log_rel:
    for name, p in remove_log_rel:
        out_lines.append(f'- {name} (removed at p={p:.4g})')
else:
    out_lines.append('- None')
out_lines.append('')
out_lines.append('### Final Relation-Aware Formula')
out_lines.append(f'- {final_formula_rel}')
out_lines.append('')
out_lines.append('### Significant Terms (p < 0.05)')
if opt_sig_rel.empty:
    out_lines.append('- None at p < 0.05')
else:
    for _, r in opt_sig_rel.sort_values('p_value').iterrows():
        direction = 'positive' if float(r['estimate']) >= 0 else 'negative'
        out_lines.append(
            f"- {r['term']}: {direction}, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}"
        )

best_path.write_text('\n'.join(out_lines), encoding='utf-8')

# Update paper-style report with relation-aware section and latest equation snapshot
paper_path = Path('data/output/ced_semantic_paper_style_report.md')
paper_text = paper_path.read_text(encoding='utf-8') if paper_path.exists() else '# A Semantic CED Construction and Modeling Study\n'
marker_paper = '\n## 10. PROPER Relation-Aware Update\n'
if marker_paper in paper_text:
    paper_text = paper_text.split(marker_paper)[0].rstrip() + '\n'

paper_lines = [paper_text.rstrip(), '', '## 10. PROPER Relation-Aware Update']
paper_lines.append('This update revises the predictor design so that PROPER items follow the implemented policy logic (Stage-1 gate, then Stage-2 assessment).')
paper_lines.append('')
paper_lines.append('### 10.1 Engineered PROPER Relationship Variables')
paper_lines.append('For firm-year $i$:')
paper_lines.append('- $proper\\_black_i = 1[s1\\_violation_i = 1]$')
paper_lines.append('- $proper\\_red\\_conditional_i = 1[s1\\_minor\\_nc_i = 1 \\land s1\\_compliance_i = 0 \\land proper\\_black_i = 0]$')
paper_lines.append('- $proper\\_stage1\\_pass_i = 1 - 1[proper\\_black_i = 1 \\lor proper\\_red\\_conditional_i = 1]$')
paper_lines.append('- $proper\\_s2\\_reduction\\_pass_i = s2\\_reduction_i \\cdot proper\\_stage1\\_pass_i$')
paper_lines.append('- $proper\\_s2\\_efficiency\\_pass_i = s2\\_efficiency_i \\cdot proper\\_stage1\\_pass_i$')
paper_lines.append('- $proper\\_iso\\_14001\\_pass_i = iso\\_14001_i \\cdot proper\\_stage1\\_pass_i$')
paper_lines.append('')
paper_lines.append('### 10.2 Updated Model Result (Relation-Aware)')
paper_lines.append(f'- Target: `{TARGET_REL}`')
paper_lines.append(f'- Optimized relation-aware model: $n={opt_n_rel}$, $R^2={opt_r2_rel:.4f}$, adjusted $R^2={opt_adj_rel:.4f}$, significant predictors={opt_sig_n_rel}.')
paper_lines.append('')
paper_lines.append('Estimated equation:')
paper_lines.append('$$')
paper_lines.append(final_formula_rel.replace('_i =', '_i =').replace('*', '\\cdot '))
paper_lines.append('$$')
paper_lines.append('')
paper_lines.append('### 10.3 Interpretation Delta vs Prior Independent-Binary PROPER Spec')
paper_lines.append('1. Stage-2 sustainability signals are now interpreted conditionally on passing Stage-1 compliance logic.')
paper_lines.append('2. This reduces policy-inconsistent attribution where Stage-2 positives could offset Stage-1 failure in interpretation.')
paper_lines.append('3. Coefficients should be read as relationship-aware associations, aligned with the PROPER inference decision process.')

paper_path.write_text('\n'.join(paper_lines), encoding='utf-8')

print('PROPER relation-aware model update complete.')
print('Target:', TARGET_REL)
print('Optimized relation-aware predictors:', len(opt_preds_rel))
print('n:', opt_n_rel, 'R2:', round(opt_r2_rel, 6), 'AdjR2:', round(opt_adj_rel, 6))
print('Best report:', best_path.resolve())
print('Paper report:', paper_path.resolve())

opt_summary_rel, opt_coefs_rel, opt_sig_rel, opt_preds_rel, final_formula_rel

PROPER relation-aware model update complete.
Target: ced_mrv_to_action
Optimized relation-aware predictors: 10
n: 880 R2: 0.532004 AdjR2: 0.526618
Best report: /media/nvme0n1/dev/annual_report/data/output/ced_semantic_best_combination.md
Paper report: /media/nvme0n1/dev/annual_report/data/output/ced_semantic_paper_style_report.md


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.532004       0.526618    98.785176        0.0        10          869   
 
       sigma    n  
 0  0.208852  880  ,
                          term  estimate  std_error  t_statistic       p_value
 0                 (Intercept) -0.907568   0.114637    -7.916884  7.327472e-15
 1              size_ln_assets  0.037940   0.004151     9.139298  0.000000e+00
 2               fcf_to_assets -0.213015   0.053805    -3.959028  8.140825e-05
 3         board_foreign_ratio  0.114608   0.058320     1.965157  4.971426e-02
 4     board_independent_ratio  0.205424   0.044432     4.623354  4.348793e-06
 5             sup_women_ratio  0.063218   0.025348     2.494050  1.281431e-02
 6                proper_black  0.122109   0.036929     3.306632  9.829763e-04
 7      proper_red_conditional  0.237606   0.029536     8.044554  2.886580e-15
 8    proper_s2_reduction_pass  0.367954   0.022048    16.688650  0.000000e+00
 9   prop

In [23]:
# Robustness checks for relation-aware PROPER model and doc update
from pathlib import Path
import duckdb
import pandas as pd

if 'base_df' not in globals() or 'TARGET_REL' not in globals() or 'opt_preds_rel' not in globals():
    raise RuntimeError('Missing relation-aware model objects. Run the relation-aware model update cell first.')

val_rel = base_df[['ticker', 'year', TARGET_REL] + opt_preds_rel].copy()
for c in [TARGET_REL] + opt_preds_rel:
    val_rel[c] = pd.to_numeric(val_rel[c], errors='coerce')
base_complete_rel = val_rel.dropna(subset=[TARGET_REL] + opt_preds_rel).copy()

con_rel_val = duckdb.connect()
try:
    try:
        con_rel_val.execute('INSTALL stats_duck')
    except Exception:
        con_rel_val.execute('INSTALL stats_duck FROM community')
    con_rel_val.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for relation-aware robustness: {str(e).splitlines()[0]}')


def _run_rel_model(df_model: pd.DataFrame, y: str, x_cols: list[str]):
    local = df_model[[y] + x_cols].dropna().copy()
    if len(local) < 80:
        return None, None
    con_rel_val.register('rel_val_view', local)
    con_rel_val.execute('CREATE OR REPLACE TABLE rel_val_model_data AS SELECT * FROM rel_val_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in x_cols) + ']'
    q_s = f"SELECT * FROM lm_summary('rel_val_model_data', y := '{y}', x := {x_sql})"
    q_c = f"SELECT * FROM lm('rel_val_model_data', y := '{y}', x := {x_sql})"
    try:
        s_df = con_rel_val.execute(q_s).fetchdf()
        c_df = con_rel_val.execute(q_c).fetchdf()
        return s_df, c_df
    except Exception:
        return None, None


def _mk_row(name, s_df, c_df):
    noni = c_df[c_df['term'] != '(Intercept)'].copy() if c_df is not None else pd.DataFrame()
    n_sig = int((noni['p_value'] < 0.05).sum()) if not noni.empty else 0
    return {
        'model': name,
        'n': int(s_df.loc[0, 'n']),
        'r2': float(s_df.loc[0, 'r_squared']),
        'adj_r2': float(s_df.loc[0, 'adj_r_squared']),
        'f_p': float(s_df.loc[0, 'f_p_value']),
        'n_significant': n_sig,
    }

rows_rel = []
coef_rel = {}

# baseline common sample
s0, c0 = _run_rel_model(base_complete_rel, TARGET_REL, opt_preds_rel)
if s0 is None:
    raise RuntimeError('Relation-aware baseline robustness model failed.')
rows_rel.append(_mk_row('baseline_common_sample_rel', s0, c0))
coef_rel['baseline_common_sample_rel'] = c0

# year FE
year_dummies = pd.get_dummies(base_complete_rel['year'].astype(int), prefix='fe_year', drop_first=True).astype(int)
fe_df = pd.concat([base_complete_rel[[TARGET_REL] + opt_preds_rel].reset_index(drop=True), year_dummies.reset_index(drop=True)], axis=1)
fe_cols = opt_preds_rel + year_dummies.columns.tolist()
sy, cy = _run_rel_model(fe_df, TARGET_REL, fe_cols)
if sy is not None:
    rows_rel.append(_mk_row('year_FE_rel', sy, cy))
    coef_rel['year_FE_rel'] = cy

# time split
years = sorted(base_complete_rel['year'].dropna().astype(int).unique().tolist())
if len(years) >= 4:
    cut = years[len(years) // 2]
    early = base_complete_rel[base_complete_rel['year'].astype(int) <= cut].copy()
    late = base_complete_rel[base_complete_rel['year'].astype(int) > cut].copy()
    for name, part in [('early_period_rel', early), ('late_period_rel', late)]:
        s, c = _run_rel_model(part, TARGET_REL, opt_preds_rel)
        if s is not None:
            rows_rel.append(_mk_row(name, s, c))
            coef_rel[name] = c

robust_rel_df = pd.DataFrame(rows_rel)

key_terms_rel = [
    t for t in ['proper_s2_reduction_pass', 'proper_s2_efficiency_pass', 'proper_iso_14001_pass', 'proper_black', 'proper_red_conditional']
    if t in opt_preds_rel
]
stab_rows_rel = []
for m, cdf in coef_rel.items():
    if cdf is None or cdf.empty:
        continue
    for t in key_terms_rel:
        r = cdf[cdf['term'] == t]
        if r.empty:
            continue
        stab_rows_rel.append({
            'model': m,
            'term': t,
            'estimate': float(r.iloc[0]['estimate']),
            'p_value': float(r.iloc[0]['p_value']),
        })
stab_rel_df = pd.DataFrame(stab_rows_rel)

# Append robustness subsection to both docs
for pth, marker in [
    (Path('data/output/ced_semantic_best_combination.md'), '## PROPER-Relation-Aware Model Update'),
    (Path('data/output/ced_semantic_paper_style_report.md'), '## 10. PROPER Relation-Aware Update'),
]:
    txt = pth.read_text(encoding='utf-8') if pth.exists() else ''
    if marker not in txt:
        continue
    sub_marker = '\n### 10.4 Relation-Aware Robustness\n' if 'paper_style' in pth.name else '\n### Relation-Aware Robustness\n'
    if sub_marker in txt:
        txt = txt.split(sub_marker)[0].rstrip() + '\n'

    lines = [txt.rstrip(), '']
    if 'paper_style' in pth.name:
        lines.append('### 10.4 Relation-Aware Robustness')
    else:
        lines.append('### Relation-Aware Robustness')

    for _, r in robust_rel_df.iterrows():
        lines.append(
            f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}"
        )

    lines.append('')
    lines.append('Key-term stability across estimated checks:')
    if stab_rel_df.empty:
        lines.append('- No key-term stability output available.')
    else:
        for t in key_terms_rel:
            sub = stab_rel_df[stab_rel_df['term'] == t]
            if sub.empty:
                continue
            lines.append(
                f"- {t}: estimate range [{float(sub['estimate'].min()):.4f}, {float(sub['estimate'].max()):.4f}], significant in {100.0*float((sub['p_value']<0.05).mean()):.1f}% of checks"
            )

    pth.write_text('\n'.join(lines), encoding='utf-8')

print('Relation-aware robustness update complete.')
print('Rows modeled:', len(base_complete_rel))
print(robust_rel_df.to_string(index=False))

robust_rel_df, stab_rel_df

Relation-aware robustness update complete.
Rows modeled: 880
                     model   n       r2   adj_r2  f_p  n_significant
baseline_common_sample_rel 880 0.532004 0.526618  0.0             10
               year_FE_rel 880 0.560192 0.549952  0.0             15
          early_period_rel 481 0.463602 0.452189  0.0              8
           late_period_rel 399 0.570348 0.559274  0.0              8


(                        model    n        r2    adj_r2  f_p  n_significant
 0  baseline_common_sample_rel  880  0.532004  0.526618  0.0             10
 1                 year_FE_rel  880  0.560192  0.549952  0.0             15
 2            early_period_rel  481  0.463602  0.452189  0.0              8
 3             late_period_rel  399  0.570348  0.559274  0.0              8,
                          model                       term  estimate  \
 0   baseline_common_sample_rel   proper_s2_reduction_pass  0.367954   
 1   baseline_common_sample_rel  proper_s2_efficiency_pass  0.127016   
 2   baseline_common_sample_rel      proper_iso_14001_pass  0.133597   
 3   baseline_common_sample_rel               proper_black  0.122109   
 4   baseline_common_sample_rel     proper_red_conditional  0.237606   
 5                  year_FE_rel   proper_s2_reduction_pass  0.312725   
 6                  year_FE_rel  proper_s2_efficiency_pass  0.122570   
 7                  year_FE_rel      proper

In [15]:
# PROPER original-design aligned model update (Stage 1 gate + Stage 2 score/color)
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

if 'nl_sem_df' not in globals():
    raise RuntimeError('nl_sem_df not found. Run semantic/model cells first.')
if 'best_semantic_target' not in globals():
    raise RuntimeError('best_semantic_target not found. Run semantic combination cell first.')
if 'opt_preds' not in globals() or not opt_preds:
    raise RuntimeError('opt_preds not found. Run predictor optimization cell first.')

TARGET_OG = str(best_semantic_target)
keys = nl_sem_df[['ticker', 'year']].drop_duplicates().copy()
keys['ticker'] = keys['ticker'].astype(str).str.upper()

# Pull PROPER job-level outputs (closest to original design) and indicator-level fallback.
con_og = duckdb.connect('db.db', read_only=True)
con_og.register('k_view', keys)

jobs_sql = """
SELECT
  UPPER(j.ticker) AS ticker,
  j.year,
  UPPER(COALESCE(j.color, '')) AS proper_color,
  CAST(j.s2_score AS DOUBLE) AS s2_score,
  CAST(j.s2_max_score AS DOUBLE) AS s2_max_score
FROM proper_vn_jobs j
JOIN k_view k ON UPPER(j.ticker) = k.ticker AND j.year = k.year
WHERE j.status = 'completed'
"""
jobs_df = con_og.execute(jobs_sql).fetchdf()
if jobs_df.empty:
    raise RuntimeError('No completed proper_vn_jobs rows found for current modeling keys.')

# Keep latest if duplicates exist.
jobs_df = jobs_df.sort_values(['ticker', 'year']).drop_duplicates(['ticker', 'year'], keep='last').reset_index(drop=True)

base_og = nl_sem_df.copy()
base_og['ticker'] = base_og['ticker'].astype(str).str.upper()
base_og = base_og.merge(jobs_df, on=['ticker', 'year'], how='left')

# Original-design Stage 1 and Stage 2 engineered terms.
base_og['proper_black_job'] = (base_og['proper_color'] == 'BLACK').astype(float)
base_og['proper_red_job'] = (base_og['proper_color'] == 'RED').astype(float)
base_og['proper_green_job'] = (base_og['proper_color'] == 'GREEN').astype(float)
base_og['proper_gold_job'] = (base_og['proper_color'] == 'GOLD').astype(float)
base_og['proper_stage1_fail_job'] = ((base_og['proper_black_job'] + base_og['proper_red_job']) > 0).astype(float)
base_og['proper_stage1_pass_job'] = 1.0 - base_og['proper_stage1_fail_job']

base_og['proper_s2_score_norm'] = np.where(
    pd.to_numeric(base_og['s2_max_score'], errors='coerce') > 0,
    pd.to_numeric(base_og['s2_score'], errors='coerce') / pd.to_numeric(base_og['s2_max_score'], errors='coerce'),
    np.nan,
)
base_og['proper_s2_score_pass'] = base_og['proper_s2_score_norm'] * base_og['proper_stage1_pass_job']

old_proper_terms = {
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001',
    'proper_black', 'proper_red_conditional', 'proper_s2_reduction_pass', 'proper_s2_efficiency_pass', 'proper_iso_14001_pass'
}
non_proper = [p for p in opt_preds if p in base_og.columns and p not in old_proper_terms]
proper_design_terms = [
    'proper_black_job',
    'proper_red_job',
    'proper_green_job',
    'proper_gold_job',
    'proper_s2_score_pass',
]
pred_pool_og = []
for c in non_proper + proper_design_terms:
    if c in base_og.columns and c not in pred_pool_og:
        pred_pool_og.append(c)

for c in [TARGET_OG] + pred_pool_og:
    base_og[c] = pd.to_numeric(base_og[c], errors='coerce')

ALPHA_OG = 0.05
MIN_PRED_OG = 6

con_fit = duckdb.connect()
try:
    try:
        con_fit.execute('INSTALL stats_duck')
    except Exception:
        con_fit.execute('INSTALL stats_duck FROM community')
    con_fit.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable for original-design model: {str(e).splitlines()[0]}')


def _fit_og(preds):
    local = base_og[[TARGET_OG] + preds].dropna().copy()
    if len(local) < 120:
        return None, None, None, local
    con_fit.register('og_view', local)
    con_fit.execute('CREATE OR REPLACE TABLE og_model_data AS SELECT * FROM og_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in preds) + ']'
    q_s = f"SELECT * FROM lm_summary('og_model_data', y := '{TARGET_OG}', x := {x_sql})"
    q_c = f"SELECT * FROM lm('og_model_data', y := '{TARGET_OG}', x := {x_sql})"
    s_df = con_fit.execute(q_s).fetchdf()
    c_df = con_fit.execute(q_c).fetchdf()
    sig_df = c_df[(c_df['term'] != '(Intercept)') & (c_df['p_value'] < ALPHA_OG)].copy()
    return s_df, c_df, sig_df, local

base_s, base_c, base_sig, base_local = _fit_og(pred_pool_og)
if base_s is None:
    raise RuntimeError('Original-design baseline model cannot be estimated.')

opt_preds_og = pred_pool_og.copy()
remove_log_og = []
while len(opt_preds_og) > MIN_PRED_OG:
    s_df, c_df, sig_df, local = _fit_og(opt_preds_og)
    noni = c_df[c_df['term'] != '(Intercept)'].copy().sort_values('p_value', ascending=False)
    worst = noni.iloc[0]
    if float(worst['p_value']) <= ALPHA_OG:
        break
    term = str(worst['term'])
    opt_preds_og.remove(term)
    remove_log_og.append((term, float(worst['p_value'])))

opt_s, opt_c, opt_sig, opt_local = _fit_og(opt_preds_og)
if opt_s is None:
    raise RuntimeError('Original-design optimized model cannot be estimated.')

coef_map_og = {str(r['term']): float(r['estimate']) for _, r in opt_c.iterrows()}
inter_og = coef_map_og.get('(Intercept)', 0.0)
terms_og = [p for p in opt_preds_og if p in coef_map_og]
rhs_og = ' + '.join([f"({coef_map_og[t]:.6f} * {t}_i)" for t in terms_og]) if terms_og else '0'
formula_og = f"{TARGET_OG}_i = {inter_og:.6f} + {rhs_og}"

n_og = int(opt_s.loc[0, 'n'])
r2_og = float(opt_s.loc[0, 'r_squared'])
adj_og = float(opt_s.loc[0, 'adj_r_squared'])
nsig_og = int(len(opt_sig))

# Robustness (baseline/year FE/time split)
rob_rows = []
coef_store = {}


def _run_rob(df_model, y, x_cols):
    local = df_model[[y] + x_cols].dropna().copy()
    if len(local) < 80:
        return None, None
    con_fit.register('rob_view', local)
    con_fit.execute('CREATE OR REPLACE TABLE og_rob_data AS SELECT * FROM rob_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in x_cols) + ']'
    s = con_fit.execute(f"SELECT * FROM lm_summary('og_rob_data', y := '{y}', x := {x_sql})").fetchdf()
    c = con_fit.execute(f"SELECT * FROM lm('og_rob_data', y := '{y}', x := {x_sql})").fetchdf()
    return s, c


def _mk(name, s, c):
    noni = c[c['term'] != '(Intercept)'].copy()
    return {
        'model': name,
        'n': int(s.loc[0, 'n']),
        'r2': float(s.loc[0, 'r_squared']),
        'adj_r2': float(s.loc[0, 'adj_r_squared']),
        'f_p': float(s.loc[0, 'f_p_value']),
        'n_significant': int((noni['p_value'] < 0.05).sum()),
    }

val = base_og[['ticker', 'year', TARGET_OG] + opt_preds_og].dropna().copy()
s0, c0 = _run_rob(val, TARGET_OG, opt_preds_og)
if s0 is not None:
    rob_rows.append(_mk('baseline_common_sample_og', s0, c0))
    coef_store['baseline_common_sample_og'] = c0

year_d = pd.get_dummies(val['year'].astype(int), prefix='fe_year', drop_first=True).astype(int)
fe_df = pd.concat([val[[TARGET_OG] + opt_preds_og].reset_index(drop=True), year_d.reset_index(drop=True)], axis=1)
fe_cols = opt_preds_og + year_d.columns.tolist()
sy, cy = _run_rob(fe_df, TARGET_OG, fe_cols)
if sy is not None:
    rob_rows.append(_mk('year_FE_og', sy, cy))
    coef_store['year_FE_og'] = cy

years = sorted(val['year'].dropna().astype(int).unique().tolist())
if len(years) >= 4:
    cut = years[len(years)//2]
    for name, part in [('early_period_og', val[val['year'].astype(int) <= cut].copy()), ('late_period_og', val[val['year'].astype(int) > cut].copy())]:
        s, c = _run_rob(part, TARGET_OG, opt_preds_og)
        if s is not None:
            rob_rows.append(_mk(name, s, c))
            coef_store[name] = c

rob_df = pd.DataFrame(rob_rows)

# Append addendum to docs.
for pth, header in [
    (Path('data/output/ced_semantic_paper_style_report.md'), '## 12. PROPER Original-Design Alignment Update'),
    (Path('data/output/ced_semantic_best_combination.md'), '## PROPER Original-Design Alignment Update'),
]:
    txt = pth.read_text(encoding='utf-8') if pth.exists() else ''
    if header in txt:
        txt = txt.split('\n' + header + '\n')[0].rstrip() + '\n'

    lines = [txt.rstrip(), '', header]
    lines.append('- Goal: align predictors with original PROPER design by using Stage 1 gate and Stage 2 score/color outputs from `proper_vn_jobs`.')
    lines.append(f'- Target: {TARGET_OG}')
    lines.append(f'- Optimized model: n={n_og}, R2={r2_og:.4f}, AdjR2={adj_og:.4f}, significant={nsig_og}')
    lines.append('')
    lines.append('### Predictor Construction (Original-Design Aligned)')
    lines.append('- `proper_black_job = 1[color = BLACK]`')
    lines.append('- `proper_red_job = 1[color = RED]`')
    lines.append('- `proper_green_job = 1[color = GREEN]`')
    lines.append('- `proper_gold_job = 1[color = GOLD]`')
    lines.append('- `proper_stage1_fail_job = 1[color in {BLACK, RED}]`')
    lines.append('- `proper_stage1_pass_job = 1 - proper_stage1_fail_job`')
    lines.append('- `proper_s2_score_norm = s2_score / s2_max_score`')
    lines.append('- `proper_s2_score_pass = proper_s2_score_norm * proper_stage1_pass_job`')
    lines.append('')
    lines.append('### Final Formula')
    lines.append(f'- {formula_og}')
    lines.append('')
    lines.append('### Significant Terms (p < 0.05)')
    if opt_sig.empty:
        lines.append('- None at p < 0.05')
    else:
        for _, r in opt_sig.sort_values('p_value').iterrows():
            d = 'positive' if float(r['estimate']) >= 0 else 'negative'
            lines.append(f"- {r['term']}: {d}, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}")
    lines.append('')
    lines.append('### Robustness Snapshot')
    if rob_df.empty:
        lines.append('- Robustness models not estimable.')
    else:
        for _, r in rob_df.iterrows():
            lines.append(f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}")

    lines.append('')
    lines.append('- Recommendation: treat this original-design aligned specification as the preferred PROPER-consistent model for interpretation and reporting.')
    pth.write_text('\n'.join(lines), encoding='utf-8')

print('Original-design aligned PROPER model update complete.')
print('Target:', TARGET_OG)
print('n:', n_og, 'R2:', round(r2_og, 6), 'AdjR2:', round(adj_og, 6), 'sig:', nsig_og)
print('Saved docs updated.')

opt_s, opt_c, opt_sig, opt_preds_og, formula_og, rob_df

Original-design aligned PROPER model update complete.
Target: ced_mrv_to_action
n: 880 R2: 0.577843 AdjR2: 0.573475 sig: 9
Saved docs updated.


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.577843       0.573475   132.315844        0.0         9          870   
 
       sigma    n  
 0  0.198246  880  ,
                       term  estimate  std_error  t_statistic   p_value
 0              (Intercept) -0.915425   0.105521    -8.675313  0.000000
 1           size_ln_assets  0.037905   0.003838     9.877101  0.000000
 2            fcf_to_assets -0.158130   0.051506    -3.070133  0.002206
 3      board_foreign_ratio  0.117432   0.055213     2.126887  0.033710
 4  board_independent_ratio  0.168706   0.042110     4.006367  0.000067
 5          sup_women_ratio  0.048056   0.024010     2.001496  0.045649
 6         proper_black_job  0.149275   0.035056     4.258195  0.000023
 7           proper_red_job  0.261047   0.028054     9.305182  0.000000
 8          proper_gold_job -0.130543   0.046059    -2.834259  0.004700
 9     proper_s2_score_pass  0.954944   0.044480    21.469231  0.000000,
      

In [17]:
# PROPER strict original-design encoding: Stage-1 penalty + Stage-2 pass quality
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

if 'nl_sem_df' not in globals() or 'best_semantic_target' not in globals() or 'opt_preds' not in globals():
    raise RuntimeError('Missing base modeling objects; run prior cells first.')

TARGET_STRICT = str(best_semantic_target)
keys = nl_sem_df[['ticker', 'year']].drop_duplicates().copy()
keys['ticker'] = keys['ticker'].astype(str).str.upper()

con_s = duckdb.connect('db.db', read_only=True)
con_s.register('k_view', keys)
jobs = con_s.execute("""
SELECT UPPER(j.ticker) AS ticker, j.year,
       UPPER(COALESCE(j.color, '')) AS proper_color,
       CAST(j.s2_score AS DOUBLE) AS s2_score,
       CAST(j.s2_max_score AS DOUBLE) AS s2_max_score
FROM proper_vn_jobs j
JOIN k_view k ON UPPER(j.ticker)=k.ticker AND j.year=k.year
WHERE j.status='completed'
""").fetchdf()
jobs = jobs.sort_values(['ticker', 'year']).drop_duplicates(['ticker', 'year'], keep='last')

df_s = nl_sem_df.copy()
df_s['ticker'] = df_s['ticker'].astype(str).str.upper()
df_s = df_s.merge(jobs, on=['ticker', 'year'], how='left')

# Strict original-design encoding.
# Stage-1 penalty: Black=-1, Red=-0.5, others=0.
# Stage-2 quality: normalized s2 score active only when Stage-1 passes.
df_s['proper_stage1_penalty'] = 0.0
df_s.loc[df_s['proper_color'] == 'BLACK', 'proper_stage1_penalty'] = -1.0
df_s.loc[df_s['proper_color'] == 'RED', 'proper_stage1_penalty'] = -0.5

df_s['proper_stage1_pass'] = (~df_s['proper_color'].isin(['BLACK', 'RED'])).astype(float)
df_s['proper_s2_score_norm'] = np.where(
    pd.to_numeric(df_s['s2_max_score'], errors='coerce') > 0,
    pd.to_numeric(df_s['s2_score'], errors='coerce') / pd.to_numeric(df_s['s2_max_score'], errors='coerce'),
    np.nan,
)
df_s['proper_s2_quality_pass'] = df_s['proper_s2_score_norm'] * df_s['proper_stage1_pass']

old_proper_terms = {
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001',
    'proper_black', 'proper_red_conditional', 'proper_s2_reduction_pass', 'proper_s2_efficiency_pass', 'proper_iso_14001_pass',
    'proper_black_job', 'proper_red_job', 'proper_green_job', 'proper_gold_job', 'proper_s2_score_pass'
}
non_proper = [p for p in opt_preds if p in df_s.columns and p not in old_proper_terms]
strict_terms = ['proper_stage1_penalty', 'proper_s2_quality_pass']
preds_s = []
for c in non_proper + strict_terms:
    if c in df_s.columns and c not in preds_s:
        preds_s.append(c)

for c in [TARGET_STRICT] + preds_s:
    df_s[c] = pd.to_numeric(df_s[c], errors='coerce')

ALPHA_S = 0.05
con_fit_s = duckdb.connect()
try:
    try:
        con_fit_s.execute('INSTALL stats_duck')
    except Exception:
        con_fit_s.execute('INSTALL stats_duck FROM community')
    con_fit_s.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable: {str(e).splitlines()[0]}')


def _fit_s(preds):
    local = df_s[[TARGET_STRICT] + preds].dropna().copy()
    if len(local) < 120:
        return None, None, None, local
    con_fit_s.register('s_view', local)
    con_fit_s.execute('CREATE OR REPLACE TABLE s_model_data AS SELECT * FROM s_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in preds) + ']'
    s_df = con_fit_s.execute(f"SELECT * FROM lm_summary('s_model_data', y := '{TARGET_STRICT}', x := {x_sql})").fetchdf()
    c_df = con_fit_s.execute(f"SELECT * FROM lm('s_model_data', y := '{TARGET_STRICT}', x := {x_sql})").fetchdf()
    sig_df = c_df[(c_df['term'] != '(Intercept)') & (c_df['p_value'] < ALPHA_S)].copy()
    return s_df, c_df, sig_df, local

s_sum, s_coef, s_sig, s_local = _fit_s(preds_s)
if s_sum is None:
    raise RuntimeError('Strict original-design model is not estimable.')

coef_map_s = {str(r['term']): float(r['estimate']) for _, r in s_coef.iterrows()}
inter_s = coef_map_s.get('(Intercept)', 0.0)
rhs_s = ' + '.join([f"({coef_map_s[t]:.6f} * {t}_i)" for t in preds_s if t in coef_map_s])
formula_s = f"{TARGET_STRICT}_i = {inter_s:.6f} + {rhs_s}"

n_s = int(s_sum.loc[0, 'n'])
r2_s = float(s_sum.loc[0, 'r_squared'])
adj_s = float(s_sum.loc[0, 'adj_r_squared'])

# Robustness snapshot
rob_rows = []

def _run_s(df_model, y, x_cols):
    local = df_model[[y] + x_cols].dropna().copy()
    if len(local) < 80:
        return None, None
    con_fit_s.register('s_rob_view', local)
    con_fit_s.execute('CREATE OR REPLACE TABLE s_rob_data AS SELECT * FROM s_rob_view')
    x_sql = '[' + ', '.join("'" + c + "'" for c in x_cols) + ']'
    s = con_fit_s.execute(f"SELECT * FROM lm_summary('s_rob_data', y := '{y}', x := {x_sql})").fetchdf()
    c = con_fit_s.execute(f"SELECT * FROM lm('s_rob_data', y := '{y}', x := {x_sql})").fetchdf()
    return s, c


def _mk(name, s, c):
    noni = c[c['term'] != '(Intercept)'].copy()
    return {
        'model': name,
        'n': int(s.loc[0, 'n']),
        'r2': float(s.loc[0, 'r_squared']),
        'adj_r2': float(s.loc[0, 'adj_r_squared']),
        'f_p': float(s.loc[0, 'f_p_value']),
        'n_significant': int((noni['p_value'] < 0.05).sum()),
    }

val = df_s[['ticker', 'year', TARGET_STRICT] + preds_s].dropna().copy()
s0, c0 = _run_s(val, TARGET_STRICT, preds_s)
if s0 is not None:
    rob_rows.append(_mk('baseline_common_sample_strict', s0, c0))
yd = pd.get_dummies(val['year'].astype(int), prefix='fe_year', drop_first=True).astype(int)
fe_df = pd.concat([val[[TARGET_STRICT] + preds_s].reset_index(drop=True), yd.reset_index(drop=True)], axis=1)
sy, cy = _run_s(fe_df, TARGET_STRICT, preds_s + yd.columns.tolist())
if sy is not None:
    rob_rows.append(_mk('year_FE_strict', sy, cy))
years = sorted(val['year'].dropna().astype(int).unique().tolist())
if len(years) >= 4:
    cut = years[len(years)//2]
    for name, part in [('early_period_strict', val[val['year'].astype(int) <= cut]), ('late_period_strict', val[val['year'].astype(int) > cut])]:
        s, c = _run_s(part, TARGET_STRICT, preds_s)
        if s is not None:
            rob_rows.append(_mk(name, s, c))
rob_df = pd.DataFrame(rob_rows)

# Replace section 12 in paper report with strict version
paper = Path('data/output/ced_semantic_paper_style_report.md')
text = paper.read_text(encoding='utf-8') if paper.exists() else ''
header = '## 12. PROPER Original-Design Alignment Update'
if header in text:
    text = text.split('\n' + header + '\n')[0].rstrip() + '\n'

lines = [text.rstrip(), '', header]
lines.append('- Goal: enforce original PROPER sequencing with a compact, policy-consistent encoding (Stage-1 penalty + Stage-2 pass quality).')
lines.append(f'- Target: {TARGET_STRICT}')
lines.append(f'- Strict model: n={n_s}, R2={r2_s:.4f}, AdjR2={adj_s:.4f}, significant={int(len(s_sig))}')
lines.append('')
lines.append('### Predictor Construction (Strict Original Design)')
lines.append('- `proper_stage1_penalty = -1` if `color=BLACK`, `-0.5` if `color=RED`, `0` otherwise.')
lines.append('- `proper_stage1_pass = 1` if `color not in {BLACK, RED}`, else `0`.')
lines.append('- `proper_s2_score_norm = s2_score / s2_max_score`.')
lines.append('- `proper_s2_quality_pass = proper_s2_score_norm * proper_stage1_pass`.')
lines.append('')
lines.append('### Final Formula')
lines.append(f'- {formula_s}')
lines.append('')
lines.append('### Significant Terms (p < 0.05)')
if s_sig.empty:
    lines.append('- None at p < 0.05')
else:
    for _, r in s_sig.sort_values('p_value').iterrows():
        d = 'positive' if float(r['estimate']) >= 0 else 'negative'
        lines.append(f"- {r['term']}: {d}, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}")
lines.append('')
lines.append('### Robustness Snapshot')
if rob_df.empty:
    lines.append('- Robustness models not estimable.')
else:
    for _, r in rob_df.iterrows():
        lines.append(f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}")
lines.append('')
lines.append('- Recommendation: use this strict original-design encoding as the primary PROPER specification.')
paper.write_text('\n'.join(lines), encoding='utf-8')

# Mirror short update to combination report
best = Path('data/output/ced_semantic_best_combination.md')
text_b = best.read_text(encoding='utf-8') if best.exists() else ''
head_b = '## PROPER Original-Design Alignment Update'
if head_b in text_b:
    text_b = text_b.split('\n' + head_b + '\n')[0].rstrip() + '\n'
lines_b = [text_b.rstrip(), '', head_b]
lines_b.append('- Strict encoding replaced expanded color dummies to better match original PROPER logic.')
lines_b.append(f'- n={n_s}, R2={r2_s:.4f}, AdjR2={adj_s:.4f}')
lines_b.append(f'- Formula: {formula_s}')
best.write_text('\n'.join(lines_b), encoding='utf-8')

print('Strict original-design PROPER update complete.')
print('n:', n_s, 'R2:', round(r2_s, 6), 'AdjR2:', round(adj_s, 6), 'significant:', len(s_sig))

s_sum, s_coef, s_sig, preds_s, formula_s, rob_df

Strict original-design PROPER update complete.
n: 880 R2: 0.557872 AdjR2: 0.554323 significant: 5


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.557872       0.554323   157.182852        0.0         7          872   
 
       sigma    n  
 0  0.202648  880  ,
                       term  estimate  std_error  t_statistic       p_value
 0              (Intercept) -0.941985   0.107755    -8.741871  0.000000e+00
 1           size_ln_assets  0.039699   0.003911    10.151211  0.000000e+00
 2            fcf_to_assets -0.184409   0.052180    -3.534089  4.306970e-04
 3      board_foreign_ratio  0.102134   0.056372     1.811773  7.036514e-02
 4  board_independent_ratio  0.161032   0.043013     3.743823  1.931269e-04
 5          sup_women_ratio  0.034052   0.024416     1.394675  1.634692e-01
 6    proper_stage1_penalty -0.238506   0.031381    -7.600418  7.616130e-14
 7   proper_s2_quality_pass  0.862020   0.036779    23.437833  0.000000e+00,
                       term  estimate  std_error  t_statistic       p_value
 1           size_ln_assets  0.039699 

In [18]:
# Final PROPER cleanup: Stage-1 pass indicator + Stage-2 quality score (on pass only)
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

if 'nl_sem_df' not in globals() or 'best_semantic_target' not in globals() or 'opt_preds' not in globals():
    raise RuntimeError('Missing base modeling objects; run prior cells first.')

TARGET_CLEAN = str(best_semantic_target)
keys = nl_sem_df[['ticker', 'year']].drop_duplicates().copy()
keys['ticker'] = keys['ticker'].astype(str).str.upper()

con_c = duckdb.connect('db.db', read_only=True)
con_c.register('k_view', keys)
jobs = con_c.execute("""
SELECT UPPER(j.ticker) AS ticker, j.year,
       UPPER(COALESCE(j.color, '')) AS proper_color,
       CAST(j.s2_score AS DOUBLE) AS s2_score,
       CAST(j.s2_max_score AS DOUBLE) AS s2_max_score
FROM proper_vn_jobs j
JOIN k_view k ON UPPER(j.ticker)=k.ticker AND j.year=k.year
WHERE j.status='completed'
""").fetchdf()
jobs = jobs.sort_values(['ticker', 'year']).drop_duplicates(['ticker', 'year'], keep='last')

df_c = nl_sem_df.copy()
df_c['ticker'] = df_c['ticker'].astype(str).str.upper()
df_c = df_c.merge(jobs, on=['ticker', 'year'], how='left')

df_c['proper_stage1_pass'] = (~df_c['proper_color'].isin(['BLACK', 'RED'])).astype(float)
df_c['proper_s2_score_norm'] = np.where(
    pd.to_numeric(df_c['s2_max_score'], errors='coerce') > 0,
    pd.to_numeric(df_c['s2_score'], errors='coerce') / pd.to_numeric(df_c['s2_max_score'], errors='coerce'),
    np.nan,
)
df_c['proper_s2_quality_pass'] = df_c['proper_s2_score_norm'] * df_c['proper_stage1_pass']

old_proper_terms = {
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001',
    'proper_black', 'proper_red_conditional', 'proper_s2_reduction_pass', 'proper_s2_efficiency_pass', 'proper_iso_14001_pass',
    'proper_black_job', 'proper_red_job', 'proper_green_job', 'proper_gold_job', 'proper_s2_score_pass',
    'proper_stage1_penalty'
}
non_proper = [p for p in opt_preds if p in df_c.columns and p not in old_proper_terms]
preds_c = []
for c in non_proper + ['proper_stage1_pass', 'proper_s2_quality_pass']:
    if c in df_c.columns and c not in preds_c:
        preds_c.append(c)

for c in [TARGET_CLEAN] + preds_c:
    df_c[c] = pd.to_numeric(df_c[c], errors='coerce')

con_fit_c = duckdb.connect()
try:
    try:
        con_fit_c.execute('INSTALL stats_duck')
    except Exception:
        con_fit_c.execute('INSTALL stats_duck FROM community')
    con_fit_c.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable: {str(e).splitlines()[0]}')

local = df_c[[TARGET_CLEAN] + preds_c].dropna().copy()
con_fit_c.register('c_view', local)
con_fit_c.execute('CREATE OR REPLACE TABLE c_model_data AS SELECT * FROM c_view')
x_sql = '[' + ', '.join("'" + c + "'" for c in preds_c) + ']'
s_sum = con_fit_c.execute(f"SELECT * FROM lm_summary('c_model_data', y := '{TARGET_CLEAN}', x := {x_sql})").fetchdf()
s_coef = con_fit_c.execute(f"SELECT * FROM lm('c_model_data', y := '{TARGET_CLEAN}', x := {x_sql})").fetchdf()
s_sig = s_coef[(s_coef['term'] != '(Intercept)') & (s_coef['p_value'] < 0.05)].copy()

coef_map = {str(r['term']): float(r['estimate']) for _, r in s_coef.iterrows()}
inter = coef_map.get('(Intercept)', 0.0)
rhs = ' + '.join([f"({coef_map[t]:.6f} * {t}_i)" for t in preds_c if t in coef_map])
formula = f"{TARGET_CLEAN}_i = {inter:.6f} + {rhs}"

n = int(s_sum.loc[0, 'n'])
r2 = float(s_sum.loc[0, 'r_squared'])
adj = float(s_sum.loc[0, 'adj_r_squared'])

# Robustness quick checks
rob_rows = []

def _run(df_model, y, x_cols):
    d = df_model[[y] + x_cols].dropna().copy()
    if len(d) < 80:
        return None, None
    con_fit_c.register('c_rob_view', d)
    con_fit_c.execute('CREATE OR REPLACE TABLE c_rob_data AS SELECT * FROM c_rob_view')
    x_sql2 = '[' + ', '.join("'" + c + "'" for c in x_cols) + ']'
    s = con_fit_c.execute(f"SELECT * FROM lm_summary('c_rob_data', y := '{y}', x := {x_sql2})").fetchdf()
    c = con_fit_c.execute(f"SELECT * FROM lm('c_rob_data', y := '{y}', x := {x_sql2})").fetchdf()
    return s, c


def _mk(name, s, c):
    noni = c[c['term'] != '(Intercept)'].copy()
    return {
        'model': name,
        'n': int(s.loc[0, 'n']),
        'r2': float(s.loc[0, 'r_squared']),
        'adj_r2': float(s.loc[0, 'adj_r_squared']),
        'f_p': float(s.loc[0, 'f_p_value']),
        'n_significant': int((noni['p_value'] < 0.05).sum()),
    }

val = df_c[['ticker', 'year', TARGET_CLEAN] + preds_c].dropna().copy()
s0, c0 = _run(val, TARGET_CLEAN, preds_c)
if s0 is not None:
    rob_rows.append(_mk('baseline_common_sample_clean', s0, c0))
yd = pd.get_dummies(val['year'].astype(int), prefix='fe_year', drop_first=True).astype(int)
fe_df = pd.concat([val[[TARGET_CLEAN] + preds_c].reset_index(drop=True), yd.reset_index(drop=True)], axis=1)
sy, cy = _run(fe_df, TARGET_CLEAN, preds_c + yd.columns.tolist())
if sy is not None:
    rob_rows.append(_mk('year_FE_clean', sy, cy))
ys = sorted(val['year'].dropna().astype(int).unique().tolist())
if len(ys) >= 4:
    cut = ys[len(ys)//2]
    for name, part in [('early_period_clean', val[val['year'].astype(int) <= cut]), ('late_period_clean', val[val['year'].astype(int) > cut])]:
        s, c = _run(part, TARGET_CLEAN, preds_c)
        if s is not None:
            rob_rows.append(_mk(name, s, c))
rob_df = pd.DataFrame(rob_rows)

# Overwrite section 12 in paper
paper = Path('data/output/ced_semantic_paper_style_report.md')
text = paper.read_text(encoding='utf-8') if paper.exists() else ''
header = '## 12. PROPER Original-Design Alignment Update'
if header in text:
    text = text.split('\n' + header + '\n')[0].rstrip() + '\n'

lines = [text.rstrip(), '', header]
lines.append('- Final cleanup: encode PROPER directly as Stage-1 pass and Stage-2 quality-on-pass, matching the original evaluation sequence.')
lines.append(f'- Target: {TARGET_CLEAN}')
lines.append(f'- Clean model: n={n}, R2={r2:.4f}, AdjR2={adj:.4f}, significant={int(len(s_sig))}')
lines.append('')
lines.append('### Predictor Construction (Final PROPER Encoding)')
lines.append('- `proper_stage1_pass = 1[color not in {BLACK, RED}], else 0`.')
lines.append('- `proper_s2_score_norm = s2_score / s2_max_score`.')
lines.append('- `proper_s2_quality_pass = proper_s2_score_norm * proper_stage1_pass`.')
lines.append('')
lines.append('### Final Formula')
lines.append(f'- {formula}')
lines.append('')
lines.append('### Significant Terms (p < 0.05)')
if s_sig.empty:
    lines.append('- None at p < 0.05')
else:
    for _, r in s_sig.sort_values('p_value').iterrows():
        d = 'positive' if float(r['estimate']) >= 0 else 'negative'
        lines.append(f"- {r['term']}: {d}, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}")
lines.append('')
lines.append('### Robustness Snapshot')
for _, r in rob_df.iterrows():
    lines.append(f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}")
lines.append('')
lines.append('- Recommendation: use this cleaned original-design PROPER encoding as the final model specification.')
paper.write_text('\n'.join(lines), encoding='utf-8')

# Mirror to best-combination report
best = Path('data/output/ced_semantic_best_combination.md')
txtb = best.read_text(encoding='utf-8') if best.exists() else ''
headb = '## PROPER Original-Design Alignment Update'
if headb in txtb:
    txtb = txtb.split('\n' + headb + '\n')[0].rstrip() + '\n'
outb = [txtb.rstrip(), '', headb]
outb.append('- Final cleanup uses Stage-1 pass plus Stage-2 quality-on-pass (original sequence).')
outb.append(f'- n={n}, R2={r2:.4f}, AdjR2={adj:.4f}')
outb.append(f'- Formula: {formula}')
best.write_text('\n'.join(outb), encoding='utf-8')

print('Final PROPER cleanup complete.')
print('n:', n, 'R2:', round(r2, 6), 'AdjR2:', round(adj, 6), 'significant:', len(s_sig))

s_sum, s_coef, s_sig, preds_c, formula, rob_df

Final PROPER cleanup complete.
n: 880 R2: 0.570676 AdjR2: 0.56723 significant: 5


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.570676        0.56723   165.585786        0.0         7          872   
 
       sigma    n  
 0  0.199692  880  ,
                       term  estimate  std_error  t_statistic       p_value
 0              (Intercept) -0.717948   0.109650    -6.547656  9.970047e-11
 1           size_ln_assets  0.038808   0.003858    10.059962  0.000000e+00
 2            fcf_to_assets -0.178298   0.051433    -3.466631  5.527746e-04
 3      board_foreign_ratio  0.104429   0.055509     1.881294  6.026495e-02
 4  board_independent_ratio  0.164875   0.042368     3.891530  1.072033e-04
 5          sup_women_ratio  0.041266   0.024038     1.716688  8.639129e-02
 6       proper_stage1_pass -0.211237   0.022845    -9.246370  0.000000e+00
 7   proper_s2_quality_pass  0.881611   0.036402    24.219011  0.000000e+00,
                       term  estimate  std_error  t_statistic   p_value
 1           size_ln_assets  0.038808   0.

In [19]:
# Final-final PROPER encoding: single final-color score from original design
from pathlib import Path
import duckdb
import pandas as pd

if 'nl_sem_df' not in globals() or 'best_semantic_target' not in globals() or 'opt_preds' not in globals():
    raise RuntimeError('Missing base modeling objects; run prior cells first.')

TARGET_COLOR = str(best_semantic_target)
keys = nl_sem_df[['ticker', 'year']].drop_duplicates().copy()
keys['ticker'] = keys['ticker'].astype(str).str.upper()

con_fc = duckdb.connect('db.db', read_only=True)
con_fc.register('k_view', keys)
color_df = con_fc.execute("""
SELECT UPPER(j.ticker) AS ticker, j.year, UPPER(COALESCE(j.color, '')) AS proper_color
FROM proper_vn_jobs j
JOIN k_view k ON UPPER(j.ticker)=k.ticker AND j.year=k.year
WHERE j.status='completed'
""").fetchdf()
color_df = color_df.sort_values(['ticker','year']).drop_duplicates(['ticker','year'], keep='last')

map_score = {'BLACK': 0.0, 'RED': 1.0, 'BLUE': 2.0, 'GREEN': 3.0, 'GOLD': 4.0}

fc_df = nl_sem_df.copy()
fc_df['ticker'] = fc_df['ticker'].astype(str).str.upper()
fc_df = fc_df.merge(color_df, on=['ticker','year'], how='left')
fc_df['proper_color_score_norm'] = fc_df['proper_color'].map(map_score) / 4.0

old_proper_terms = {
    's1_compliance', 's1_violation', 's1_minor_nc', 's2_reduction', 's2_efficiency', 'iso_14001',
    'proper_black', 'proper_red_conditional', 'proper_s2_reduction_pass', 'proper_s2_efficiency_pass', 'proper_iso_14001_pass',
    'proper_black_job', 'proper_red_job', 'proper_green_job', 'proper_gold_job', 'proper_s2_score_pass',
    'proper_stage1_penalty', 'proper_stage1_pass', 'proper_s2_quality_pass'
}
base_non_proper = [p for p in opt_preds if p in fc_df.columns and p not in old_proper_terms]
preds_fc = []
for c in base_non_proper + ['proper_color_score_norm']:
    if c in fc_df.columns and c not in preds_fc:
        preds_fc.append(c)

for c in [TARGET_COLOR] + preds_fc:
    fc_df[c] = pd.to_numeric(fc_df[c], errors='coerce')

con_fit_fc = duckdb.connect()
try:
    try:
        con_fit_fc.execute('INSTALL stats_duck')
    except Exception:
        con_fit_fc.execute('INSTALL stats_duck FROM community')
    con_fit_fc.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(f'stats_duck unavailable: {str(e).splitlines()[0]}')

local = fc_df[[TARGET_COLOR] + preds_fc].dropna().copy()
con_fit_fc.register('fc_view', local)
con_fit_fc.execute('CREATE OR REPLACE TABLE fc_model_data AS SELECT * FROM fc_view')
x_sql = '[' + ', '.join("'" + c + "'" for c in preds_fc) + ']'
fc_sum = con_fit_fc.execute(f"SELECT * FROM lm_summary('fc_model_data', y := '{TARGET_COLOR}', x := {x_sql})").fetchdf()
fc_coef = con_fit_fc.execute(f"SELECT * FROM lm('fc_model_data', y := '{TARGET_COLOR}', x := {x_sql})").fetchdf()
fc_sig = fc_coef[(fc_coef['term'] != '(Intercept)') & (fc_coef['p_value'] < 0.05)].copy()

coef_map = {str(r['term']): float(r['estimate']) for _, r in fc_coef.iterrows()}
inter = coef_map.get('(Intercept)', 0.0)
rhs = ' + '.join([f"({coef_map[t]:.6f} * {t}_i)" for t in preds_fc if t in coef_map])
formula_fc = f"{TARGET_COLOR}_i = {inter:.6f} + {rhs}"

n = int(fc_sum.loc[0, 'n'])
r2 = float(fc_sum.loc[0, 'r_squared'])
adj = float(fc_sum.loc[0, 'adj_r_squared'])

# light robustness
rob_rows = []
def _run(df_model, y, x_cols):
    d = df_model[[y] + x_cols].dropna().copy()
    if len(d) < 80:
        return None, None
    con_fit_fc.register('fc_rob_view', d)
    con_fit_fc.execute('CREATE OR REPLACE TABLE fc_rob_data AS SELECT * FROM fc_rob_view')
    x_sql2 = '[' + ', '.join("'" + c + "'" for c in x_cols) + ']'
    s = con_fit_fc.execute(f"SELECT * FROM lm_summary('fc_rob_data', y := '{y}', x := {x_sql2})").fetchdf()
    c = con_fit_fc.execute(f"SELECT * FROM lm('fc_rob_data', y := '{y}', x := {x_sql2})").fetchdf()
    return s, c

def _mk(name,s,c):
    noni = c[c['term']!='(Intercept)'].copy()
    return {'model':name,'n':int(s.loc[0,'n']),'r2':float(s.loc[0,'r_squared']),'adj_r2':float(s.loc[0,'adj_r_squared']),'f_p':float(s.loc[0,'f_p_value']),'n_significant':int((noni['p_value']<0.05).sum())}

val = fc_df[['ticker','year',TARGET_COLOR] + preds_fc].dropna().copy()
s0,c0=_run(val,TARGET_COLOR,preds_fc)
if s0 is not None: rob_rows.append(_mk('baseline_common_sample_color',s0,c0))
yd = pd.get_dummies(val['year'].astype(int), prefix='fe_year', drop_first=True).astype(int)
fe_df = pd.concat([val[[TARGET_COLOR]+preds_fc].reset_index(drop=True), yd.reset_index(drop=True)], axis=1)
sy,cy=_run(fe_df,TARGET_COLOR,preds_fc+yd.columns.tolist())
if sy is not None: rob_rows.append(_mk('year_FE_color',sy,cy))
ys=sorted(val['year'].dropna().astype(int).unique().tolist())
if len(ys)>=4:
    cut=ys[len(ys)//2]
    for name,part in [('early_period_color', val[val['year'].astype(int)<=cut]), ('late_period_color', val[val['year'].astype(int)>cut])]:
        s,c=_run(part,TARGET_COLOR,preds_fc)
        if s is not None: rob_rows.append(_mk(name,s,c))
rob_df=pd.DataFrame(rob_rows)

paper=Path('data/output/ced_semantic_paper_style_report.md')
text=paper.read_text(encoding='utf-8') if paper.exists() else ''
header='## 12. PROPER Original-Design Alignment Update'
if header in text:
    text=text.split('\n'+header+'\n')[0].rstrip()+'\n'
lines=[text.rstrip(),'',header]
lines.append('- Final chosen PROPER touch-up: use one original-design final color score predictor from `proper_vn_jobs.color`.')
lines.append(f'- Target: {TARGET_COLOR}')
lines.append(f'- Color-score model: n={n}, R2={r2:.4f}, AdjR2={adj:.4f}, significant={int(len(fc_sig))}')
lines.append('')
lines.append('### Predictor Construction (Final PROPER Encoding)')
lines.append('- Map final PROPER color to an ordinal score: `BLACK=0`, `RED=1`, `BLUE=2`, `GREEN=3`, `GOLD=4`.')
lines.append('- Normalize: `proper_color_score_norm = color_score / 4`.')
lines.append('- Interpretation: higher score means better final PROPER outcome under original Stage-1/Stage-2 rules.')
lines.append('')
lines.append('### Final Formula')
lines.append(f'- {formula_fc}')
lines.append('')
lines.append('### Significant Terms (p < 0.05)')
if fc_sig.empty:
    lines.append('- None at p < 0.05')
else:
    for _,r in fc_sig.sort_values('p_value').iterrows():
        d='positive' if float(r['estimate'])>=0 else 'negative'
        lines.append(f"- {r['term']}: {d}, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}")
lines.append('')
lines.append('### Robustness Snapshot')
for _,r in rob_df.iterrows():
    lines.append(f"- {r['model']}: n={int(r['n'])}, R2={float(r['r2']):.4f}, AdjR2={float(r['adj_r2']):.4f}, significant={int(r['n_significant'])}, F-p={float(r['f_p']):.4g}")
lines.append('')
lines.append('- Recommendation: use this color-score specification as the final PROPER-consistent model in reporting.')
paper.write_text('\n'.join(lines), encoding='utf-8')

best=Path('data/output/ced_semantic_best_combination.md')
txt=best.read_text(encoding='utf-8') if best.exists() else ''
head='## PROPER Original-Design Alignment Update'
if head in txt:
    txt=txt.split('\n'+head+'\n')[0].rstrip()+'\n'
out=[txt.rstrip(),'',head]
out.append('- Final touch-up uses normalized final PROPER color score from original design output.')
out.append(f'- n={n}, R2={r2:.4f}, AdjR2={adj:.4f}')
out.append(f'- Formula: {formula_fc}')
best.write_text('\n'.join(out), encoding='utf-8')

print('Color-score PROPER model update complete.')
print('n:',n,'R2:',round(r2,6),'AdjR2:',round(adj,6),'significant:',len(fc_sig))

fc_sum, fc_coef, fc_sig, preds_fc, formula_fc, rob_df

Color-score PROPER model update complete.
n: 880 R2: 0.34335 AdjR2: 0.338837 significant: 5


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0    0.34335       0.338837    76.079118        0.0         6          873   
 
       sigma    n  
 0  0.246824  880  ,
                       term  estimate  std_error  t_statistic   p_value
 0              (Intercept) -1.518500   0.127786   -11.883116  0.000000
 1           size_ln_assets  0.056149   0.004671    12.021529  0.000000
 2            fcf_to_assets -0.244113   0.063553    -3.841077  0.000131
 3      board_foreign_ratio  0.227108   0.068249     3.327641  0.000912
 4  board_independent_ratio  0.252061   0.052126     4.835654  0.000002
 5          sup_women_ratio  0.035000   0.029721     1.177620  0.239269
 6  proper_color_score_norm  0.456624   0.048877     9.342369  0.000000,
                       term  estimate  std_error  t_statistic   p_value
 1           size_ln_assets  0.056149   0.004671    12.021529  0.000000
 2            fcf_to_assets -0.244113   0.063553    -3.841077  0.000131
 3    

In [20]:
# Quick check: Stage-1 fail + Stage-2 quality (diagnostic)
import duckdb
import pandas as pd

if 'df_c' not in globals() or 'TARGET_CLEAN' not in globals() or 'base_non_proper' not in globals():
    raise RuntimeError('Run the Stage-1 pass cleanup cell first (#VSC-694d87b6).')

diag_df = df_c.copy()
diag_df['proper_stage1_fail'] = 1.0 - pd.to_numeric(diag_df['proper_stage1_pass'], errors='coerce')
preds_diag = []
for c in base_non_proper + ['proper_stage1_fail', 'proper_s2_quality_pass']:
    if c in diag_df.columns and c not in preds_diag:
        preds_diag.append(c)

for c in [TARGET_CLEAN] + preds_diag:
    diag_df[c] = pd.to_numeric(diag_df[c], errors='coerce')

con = duckdb.connect()
try:
    try:
        con.execute('INSTALL stats_duck')
    except Exception:
        con.execute('INSTALL stats_duck FROM community')
    con.execute('LOAD stats_duck')
except Exception as e:
    raise RuntimeError(str(e).splitlines()[0])

local = diag_df[[TARGET_CLEAN] + preds_diag].dropna().copy()
con.register('d_view', local)
con.execute('CREATE OR REPLACE TABLE d_model AS SELECT * FROM d_view')
x_sql = '[' + ', '.join("'" + c + "'" for c in preds_diag) + ']'
s = con.execute(f"SELECT * FROM lm_summary('d_model', y := '{TARGET_CLEAN}', x := {x_sql})").fetchdf()
c = con.execute(f"SELECT * FROM lm('d_model', y := '{TARGET_CLEAN}', x := {x_sql})").fetchdf()

sig = c[(c['term'] != '(Intercept)') & (c['p_value'] < 0.05)].copy()
print('diag n=', int(s.loc[0,'n']), 'R2=', float(s.loc[0,'r_squared']), 'AdjR2=', float(s.loc[0,'adj_r_squared']))
print(c.to_string(index=False))

s, c, sig, preds_diag

diag n= 880 R2= 0.570676094783225 AdjR2= 0.5672296872872188
                   term  estimate  std_error  t_statistic  p_value
            (Intercept) -0.929185   0.106213    -8.748298 0.000000
         size_ln_assets  0.038808   0.003858    10.059962 0.000000
          fcf_to_assets -0.178298   0.051433    -3.466631 0.000553
    board_foreign_ratio  0.104429   0.055509     1.881294 0.060265
board_independent_ratio  0.164875   0.042368     3.891530 0.000107
        sup_women_ratio  0.041266   0.024038     1.716688 0.086391
     proper_stage1_fail  0.211237   0.022845     9.246370 0.000000
 proper_s2_quality_pass  0.881611   0.036402    24.219011 0.000000


(   r_squared  adj_r_squared  f_statistic  f_p_value  df_model  df_residual  \
 0   0.570676        0.56723   165.585786        0.0         7          872   
 
       sigma    n  
 0  0.199692  880  ,
                       term  estimate  std_error  t_statistic   p_value
 0              (Intercept) -0.929185   0.106213    -8.748298  0.000000
 1           size_ln_assets  0.038808   0.003858    10.059962  0.000000
 2            fcf_to_assets -0.178298   0.051433    -3.466631  0.000553
 3      board_foreign_ratio  0.104429   0.055509     1.881294  0.060265
 4  board_independent_ratio  0.164875   0.042368     3.891530  0.000107
 5          sup_women_ratio  0.041266   0.024038     1.716688  0.086391
 6       proper_stage1_fail  0.211237   0.022845     9.246370  0.000000
 7   proper_s2_quality_pass  0.881611   0.036402    24.219011  0.000000,
                       term  estimate  std_error  t_statistic   p_value
 1           size_ln_assets  0.038808   0.003858    10.059962  0.000000
 2    

In [21]:
# Export final model input (dependent variable + predictors) to CSV
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

if 'nl_sem_df' not in globals():
    raise RuntimeError('nl_sem_df not found. Run semantic/model cells first.')

# Resolve active dependent variable from latest modeling flow.
target = None
for name in ['TARGET_CLEAN', 'TARGET_STRICT', 'TARGET_REL']:
    if name in globals() and str(globals()[name]) in nl_sem_df.columns:
        target = str(globals()[name])
        break
if target is None and 'best_semantic_target' in globals() and str(best_semantic_target) in nl_sem_df.columns:
    target = str(best_semantic_target)
if target is None:
    raise RuntimeError('Could not resolve dependent variable from current notebook state.')

# Prefer the latest optimized predictor set when available.
predictors = []
for name in ['preds_rel', 'preds_c', 'preds_s', 'opt_preds_rel', 'opt_preds', 'selected_predictors']:
    if name in globals() and isinstance(globals()[name], list) and len(globals()[name]) > 0:
        predictors = [p for p in globals()[name] if p in nl_sem_df.columns and p != target]
        if predictors:
            break

# Fallback: explicit final-encoding predictor candidates.
if not predictors:
    fallback = [
        'size_ln_assets',
        'fcf_to_assets',
        'board_foreign_ratio',
        'board_independent_ratio',
        'sup_women_ratio',
        'proper_stage1_fail',
        'proper_stage1_pass',
        'proper_stage1_penalty',
        'proper_s2_quality_pass',
        'proper_s2_score_pass',
        'proper_s2_score_norm',
    ]
    predictors = [c for c in fallback if c in nl_sem_df.columns and c != target]

# Build export frame.
export_df = nl_sem_df.copy()

# Ensure moderation-ready PROPER terms exist by deriving from proper_vn_jobs when absent.
needed_moderation = [
    'proper_stage1_fail',
    'proper_stage1_pass',
    'proper_stage1_penalty',
    'proper_s2_quality_pass',
    'proper_s2_score_pass',
    'proper_s2_score_norm',
]
missing_moderation = [c for c in needed_moderation if c not in export_df.columns]
if missing_moderation and {'ticker', 'year'}.issubset(export_df.columns):
    db_path = Path('db.db')
    if db_path.exists():
        con_jobs = duckdb.connect(str(db_path), read_only=True)
        keys = export_df[['ticker', 'year']].drop_duplicates().copy()
        keys['ticker'] = keys['ticker'].astype(str).str.upper()
        con_jobs.register('keys_view', keys)
        jobs_sql = """
        SELECT UPPER(j.ticker) AS ticker, j.year,
               UPPER(COALESCE(j.color, '')) AS proper_color,
               CAST(j.s2_score AS DOUBLE) AS s2_score,
               CAST(j.s2_max_score AS DOUBLE) AS s2_max_score
        FROM proper_vn_jobs j
        JOIN keys_view k ON UPPER(j.ticker) = k.ticker AND j.year = k.year
        WHERE j.status = 'completed'
        """
        jobs = con_jobs.execute(jobs_sql).fetchdf()
        con_jobs.close()
        if not jobs.empty:
            jobs = jobs.sort_values(['ticker', 'year']).drop_duplicates(['ticker', 'year'], keep='last').copy()
            jobs['proper_stage1_pass'] = (~jobs['proper_color'].isin(['BLACK', 'RED'])).astype(float)
            jobs['proper_stage1_fail'] = 1.0 - jobs['proper_stage1_pass']
            jobs['proper_stage1_penalty'] = 0.0
            jobs.loc[jobs['proper_color'] == 'BLACK', 'proper_stage1_penalty'] = -1.0
            jobs.loc[jobs['proper_color'] == 'RED', 'proper_stage1_penalty'] = -0.5
            jobs['proper_s2_score_norm'] = np.where(
                pd.to_numeric(jobs['s2_max_score'], errors='coerce') > 0,
                pd.to_numeric(jobs['s2_score'], errors='coerce') / pd.to_numeric(jobs['s2_max_score'], errors='coerce'),
                np.nan,
            )
            jobs['proper_s2_score_pass'] = jobs['proper_s2_score_norm'] * jobs['proper_stage1_pass']
            jobs['proper_s2_quality_pass'] = jobs['proper_s2_score_pass']
            merge_cols = ['ticker', 'year'] + [c for c in needed_moderation if c in jobs.columns]
            export_df['ticker'] = export_df['ticker'].astype(str).str.upper()
            export_df = export_df.merge(jobs[merge_cols], on=['ticker', 'year'], how='left', suffixes=('', '_jobs'))
            for c in needed_moderation:
                jc = f'{c}_jobs'
                if c not in export_df.columns and jc in export_df.columns:
                    export_df[c] = export_df[jc]
                elif c in export_df.columns and jc in export_df.columns:
                    export_df[c] = export_df[c].where(export_df[c].notna(), export_df[jc])
            drop_job_cols = [f'{c}_jobs' for c in needed_moderation if f'{c}_jobs' in export_df.columns]
            if drop_job_cols:
                export_df = export_df.drop(columns=drop_job_cols)

if not predictors:
    predictors = [c for c in needed_moderation if c in export_df.columns and c != target]
if not predictors:
    raise RuntimeError('Could not resolve predictor list from current notebook state.')

# Ensure moderation-ready PROPER terms are exported when available.
for c in needed_moderation:
    if c in export_df.columns and c != target and c not in predictors:
        predictors.append(c)

# Keep one stage-1 indicator to avoid duplicate encodings of the same concept.
if 'proper_stage1_fail' in predictors and 'proper_stage1_pass' in predictors:
    predictors.remove('proper_stage1_pass')

# Generate interaction terms for moderation tests.
env_perf_col = next((c for c in ['proper_s2_quality_pass', 'proper_s2_score_pass', 'proper_s2_score_norm'] if c in export_df.columns), None)
interaction_bases = [
    'size_ln_assets',
    'fcf_to_assets',
    'board_foreign_ratio',
    'board_independent_ratio',
    'sup_women_ratio',
    's1_violation',
    's1_minor_nc',
    's2_reduction',
    's2_efficiency',
    'iso_14001',
]
if env_perf_col is not None:
    for base in interaction_bases:
        if base in export_df.columns and base != env_perf_col:
            inter_col = f'{base}_x_{env_perf_col}'
            export_df[inter_col] = pd.to_numeric(export_df[base], errors='coerce') * pd.to_numeric(export_df[env_perf_col], errors='coerce')
            if inter_col not in predictors and inter_col != target:
                predictors.append(inter_col)

cols = [target] + [c for c in predictors if c in export_df.columns]
model_input_raw = export_df[cols].copy()
for c in cols:
    model_input_raw[c] = pd.to_numeric(model_input_raw[c], errors='coerce')

model_input_complete = model_input_raw.dropna().copy()

out_dir = Path('data/output')
out_dir.mkdir(parents=True, exist_ok=True)

safe_target = target.replace('/', '_').replace(' ', '_')
raw_path = out_dir / f'model_input_{safe_target}_raw.csv'
complete_path = out_dir / f'model_input_{safe_target}_complete.csv'

model_input_raw.to_csv(raw_path, index=False)
model_input_complete.to_csv(complete_path, index=False)

print('Dependent variable:', target)
print('Predictors:', predictors)
print('Columns exported:', cols)
print('Raw rows:', len(model_input_raw), 'Complete-case rows:', len(model_input_complete))
print('Saved:', raw_path.resolve())
print('Saved:', complete_path.resolve())

model_input_raw.head(), model_input_complete.head(), raw_path, complete_path

Dependent variable: ced_mrv_to_action
Predictors: ['size_ln_assets', 'fcf_to_assets', 'board_foreign_ratio', 'board_independent_ratio', 'sup_women_ratio', 'proper_stage1_fail', 'proper_stage1_penalty', 'proper_s2_quality_pass', 'proper_s2_score_pass', 'proper_s2_score_norm', 'size_ln_assets_x_proper_s2_quality_pass', 'fcf_to_assets_x_proper_s2_quality_pass', 'board_foreign_ratio_x_proper_s2_quality_pass', 'board_independent_ratio_x_proper_s2_quality_pass', 'sup_women_ratio_x_proper_s2_quality_pass', 's1_violation_x_proper_s2_quality_pass', 's1_minor_nc_x_proper_s2_quality_pass', 's2_reduction_x_proper_s2_quality_pass', 's2_efficiency_x_proper_s2_quality_pass', 'iso_14001_x_proper_s2_quality_pass']
Columns exported: ['ced_mrv_to_action', 'size_ln_assets', 'fcf_to_assets', 'board_foreign_ratio', 'board_independent_ratio', 'sup_women_ratio', 'proper_stage1_fail', 'proper_stage1_penalty', 'proper_s2_quality_pass', 'proper_s2_score_pass', 'proper_s2_score_norm', 'size_ln_assets_x_proper_s2_

(   ced_mrv_to_action  size_ln_assets  fcf_to_assets  board_foreign_ratio  \
 0               0.45       28.301291       0.230634                  0.0   
 1               0.45       28.755176       0.282049                  0.0   
 2               0.45       29.151881       0.192783                  0.0   
 3               0.45       29.649805       0.362027                  0.0   
 4                0.8       29.708893       0.002167                  0.0   
 
    board_independent_ratio  sup_women_ratio  proper_stage1_fail  \
 0                      0.0         0.666667                 0.0   
 1                      0.0         1.000000                 0.0   
 2                      0.0         1.000000                 0.0   
 3                      0.4         1.000000                 0.0   
 4                      0.2         1.000000                 0.0   
 
    proper_stage1_penalty  proper_s2_quality_pass  proper_s2_score_pass  ...  \
 0                    0.0                   0.

In [30]:
# Export model input from raw level (raw financial inputs + target/predictors)
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

if 'df_raw_inputs' not in globals():
    raise RuntimeError('df_raw_inputs not found. Run extraction cell first.')
if 'nl_sem_df' not in globals():
    raise RuntimeError('nl_sem_df not found. Run semantic/model cells first.')

# Resolve active target
target = None
for name in ['TARGET_CLEAN', 'TARGET_STRICT', 'TARGET_REL']:
    if name in globals() and str(globals()[name]) in nl_sem_df.columns:
        target = str(globals()[name])
        break
if target is None and 'best_semantic_target' in globals() and str(best_semantic_target) in nl_sem_df.columns:
    target = str(best_semantic_target)
if target is None:
    for candidate in ['ced_mrv_to_action', 'ced_mrv_emissions_energy', 'ced_reduction_execution']:
        if candidate in nl_sem_df.columns:
            target = candidate
            break
if target is None:
    raise RuntimeError('Could not resolve dependent variable from current notebook state.')

# Resolve active predictor set (same priority as model export)
predictors = []
for name in ['preds_rel', 'preds_c', 'preds_s', 'opt_preds_rel', 'opt_preds', 'selected_predictors']:
    if name in globals() and isinstance(globals()[name], list) and len(globals()[name]) > 0:
        predictors = [p for p in globals()[name] if p in nl_sem_df.columns and p != target]
        if predictors:
            break
if not predictors:
    fallback = [
        'size_ln_assets',
        'fcf_to_assets',
        'board_foreign_ratio',
        'board_independent_ratio',
        'sup_women_ratio',
        'proper_stage1_fail',
        'proper_stage1_pass',
        'proper_stage1_penalty',
        'proper_s2_quality_pass',
        'proper_s2_score_pass',
        'proper_s2_score_norm',
    ]
    predictors = [c for c in fallback if c in nl_sem_df.columns and c != target]

# Build export frame.
export_df = nl_sem_df.copy()

# Ensure moderation-ready PROPER terms exist by deriving from proper_vn_jobs when absent.
needed_moderation = [
    'proper_stage1_fail',
    'proper_stage1_pass',
    'proper_stage1_penalty',
    'proper_s2_quality_pass',
    'proper_s2_score_pass',
    'proper_s2_score_norm',
]
missing_moderation = [c for c in needed_moderation if c not in export_df.columns]
if missing_moderation and {'ticker', 'year'}.issubset(export_df.columns):
    db_path = Path('db.db')
    if db_path.exists():
        con_jobs = duckdb.connect(str(db_path), read_only=True)
        keys = export_df[['ticker', 'year']].drop_duplicates().copy()
        keys['ticker'] = keys['ticker'].astype(str).str.upper()
        con_jobs.register('keys_view', keys)
        jobs_sql = """
        SELECT UPPER(j.ticker) AS ticker, j.year,
               UPPER(COALESCE(j.color, '')) AS proper_color,
               CAST(j.s2_score AS DOUBLE) AS s2_score,
               CAST(j.s2_max_score AS DOUBLE) AS s2_max_score
        FROM proper_vn_jobs j
        JOIN keys_view k ON UPPER(j.ticker) = k.ticker AND j.year = k.year
        WHERE j.status = 'completed'
        """
        jobs = con_jobs.execute(jobs_sql).fetchdf()
        con_jobs.close()
        if not jobs.empty:
            jobs = jobs.sort_values(['ticker', 'year']).drop_duplicates(['ticker', 'year'], keep='last').copy()
            jobs['proper_stage1_pass'] = (~jobs['proper_color'].isin(['BLACK', 'RED'])).astype(float)
            jobs['proper_stage1_fail'] = 1.0 - jobs['proper_stage1_pass']
            jobs['proper_stage1_penalty'] = 0.0
            jobs.loc[jobs['proper_color'] == 'BLACK', 'proper_stage1_penalty'] = -1.0
            jobs.loc[jobs['proper_color'] == 'RED', 'proper_stage1_penalty'] = -0.5
            jobs['proper_s2_score_norm'] = np.where(
                pd.to_numeric(jobs['s2_max_score'], errors='coerce') > 0,
                pd.to_numeric(jobs['s2_score'], errors='coerce') / pd.to_numeric(jobs['s2_max_score'], errors='coerce'),
                np.nan,
            )
            jobs['proper_s2_score_pass'] = jobs['proper_s2_score_norm'] * jobs['proper_stage1_pass']
            jobs['proper_s2_quality_pass'] = jobs['proper_s2_score_pass']
            merge_cols = ['ticker', 'year'] + [c for c in needed_moderation if c in jobs.columns]
            export_df['ticker'] = export_df['ticker'].astype(str).str.upper()
            export_df = export_df.merge(jobs[merge_cols], on=['ticker', 'year'], how='left', suffixes=('', '_jobs'))
            for c in needed_moderation:
                jc = f'{c}_jobs'
                if c not in export_df.columns and jc in export_df.columns:
                    export_df[c] = export_df[jc]
                elif c in export_df.columns and jc in export_df.columns:
                    export_df[c] = export_df[c].where(export_df[c].notna(), export_df[jc])
            drop_job_cols = [f'{c}_jobs' for c in needed_moderation if f'{c}_jobs' in export_df.columns]
            if drop_job_cols:
                export_df = export_df.drop(columns=drop_job_cols)

if not predictors:
    predictors = [c for c in needed_moderation if c in export_df.columns and c != target]
if not predictors:
    raise RuntimeError('Could not resolve predictor list from current notebook state.')

# Ensure moderation-ready PROPER terms are exported when available.
for c in needed_moderation:
    if c in export_df.columns and c != target and c not in predictors:
        predictors.append(c)

# Keep one Stage-1 representation only
if 'proper_stage1_fail' in predictors and 'proper_stage1_pass' in predictors:
    predictors.remove('proper_stage1_pass')

# Generate interaction terms for moderation tests.
env_perf_col = next((c for c in ['proper_s2_quality_pass', 'proper_s2_score_pass', 'proper_s2_score_norm'] if c in export_df.columns), None)
interaction_bases = [
    'size_ln_assets',
    'fcf_to_assets',
    'board_foreign_ratio',
    'board_independent_ratio',
    'sup_women_ratio',
    's1_violation',
    's1_minor_nc',
    's2_reduction',
    's2_efficiency',
    'iso_14001',
]
if env_perf_col is not None:
    for base in interaction_bases:
        if base in export_df.columns and base != env_perf_col:
            inter_col = f'{base}_x_{env_perf_col}'
            export_df[inter_col] = pd.to_numeric(export_df[base], errors='coerce') * pd.to_numeric(export_df[env_perf_col], errors='coerce')
            if inter_col not in predictors and inter_col != target:
                predictors.append(inter_col)

# Selected CED component variants that build semantic target
selected_variant_cols = []
if 'selected_variant_map' in globals() and isinstance(selected_variant_map, dict):
    selected_variant_cols = [c for c in selected_variant_map.values() if c in export_df.columns]

base_cols = [c for c in ['ticker', 'year'] if c in df_raw_inputs.columns]
raw_cols = [
    c for c in [
        'raw_total_assets',
        'raw_current_assets',
        'raw_long_term_assets',
        'raw_total_liabilities',
        'raw_short_term_liabilities',
        'raw_short_term_debt',
        'raw_short_term_financial_investments',
        'raw_owners_equity',
        'raw_cash_and_equivalents',
        'raw_tangible_fixed_assets',
        'raw_depreciation_expense',
        'raw_net_cf_operating',
        'raw_taxes_payable_state',
        'raw_current_corporate_income_tax_expense',
        'raw_net_revenue',
        'raw_profit_after_tax',
        'raw_pretax_profit',
        'raw_long_term_debt',
        'raw_first_event_year',
        'boardsize',
        'woman',
        'aud',
        'iso',
    ]
    if c in df_raw_inputs.columns
]

# Merge raw-level inputs with model target/predictors on ticker-year
model_cols = ['ticker', 'year', target] + predictors + selected_variant_cols
model_cols = list(dict.fromkeys([c for c in model_cols if c in export_df.columns or c in ['ticker', 'year']]))

raw_model = df_raw_inputs[base_cols + raw_cols].copy()
model_side = export_df[model_cols].copy()
raw_level_input = raw_model.merge(model_side, on=['ticker', 'year'], how='left')

# Numeric cleanup on model columns only (leave ticker as string)
for c in [target] + predictors + selected_variant_cols + raw_cols:
    if c in raw_level_input.columns:
        raw_level_input[c] = pd.to_numeric(raw_level_input[c], errors='coerce')

# Complete-case view only over actual model columns
complete_model_cols = [target] + [c for c in predictors if c in raw_level_input.columns]
raw_level_complete = raw_level_input.dropna(subset=[c for c in complete_model_cols if c in raw_level_input.columns]).copy()

out_dir = Path('data/output')
out_dir.mkdir(parents=True, exist_ok=True)
safe_target = target.replace('/', '_').replace(' ', '_')
raw_level_path = out_dir / f'model_input_raw_level_{safe_target}.csv'
raw_level_complete_path = out_dir / f'model_input_raw_level_{safe_target}_complete_model_cols.csv'

raw_level_input.to_csv(raw_level_path, index=False)
raw_level_complete.to_csv(raw_level_complete_path, index=False)

print('Target:', target)
print('Predictors:', predictors)
print('Raw-side columns:', raw_cols)
print('Selected CED component columns included:', len(selected_variant_cols))
print('Rows (raw-level merged):', len(raw_level_input))
print('Rows (complete on model cols):', len(raw_level_complete))
print('Saved:', raw_level_path.resolve())
print('Saved:', raw_level_complete_path.resolve())

raw_level_input.head(), raw_level_complete.head(), raw_level_path, raw_level_complete_path

Target: ced_mrv_to_action
Predictors: ['size_ln_assets', 'fcf_to_assets', 'board_foreign_ratio', 'board_independent_ratio', 'sup_women_ratio', 'proper_stage1_fail', 'proper_stage1_penalty', 'proper_s2_quality_pass', 'proper_s2_score_pass', 'proper_s2_score_norm', 'size_ln_assets_x_proper_s2_quality_pass', 'fcf_to_assets_x_proper_s2_quality_pass', 'board_foreign_ratio_x_proper_s2_quality_pass', 'board_independent_ratio_x_proper_s2_quality_pass', 'sup_women_ratio_x_proper_s2_quality_pass', 's1_violation_x_proper_s2_quality_pass', 's1_minor_nc_x_proper_s2_quality_pass', 's2_reduction_x_proper_s2_quality_pass', 's2_efficiency_x_proper_s2_quality_pass', 'iso_14001_x_proper_s2_quality_pass']
Raw-side columns: ['raw_total_assets', 'raw_current_assets', 'raw_long_term_assets', 'raw_total_liabilities', 'raw_short_term_liabilities', 'raw_short_term_debt', 'raw_short_term_financial_investments', 'raw_owners_equity', 'raw_cash_and_equivalents', 'raw_tangible_fixed_assets', 'raw_depreciation_expens

(  ticker  year  raw_total_assets  raw_current_assets  raw_long_term_assets  \
 0    AAA  2015      1.954765e+12        1.071561e+12          8.832037e+11   
 1    AAA  2016      3.077616e+12        1.361646e+12          1.715970e+12   
 2    AAA  2017      4.576157e+12        2.142717e+12          2.433441e+12   
 3    AAA  2018      7.529167e+12        3.989369e+12          3.539797e+12   
 4    AAA  2019      7.987454e+12        4.971364e+12          3.016091e+12   
 
    raw_total_liabilities  raw_short_term_liabilities  raw_short_term_debt  \
 0           1.135279e+12                6.670792e+11         4.387699e+11   
 1           2.122864e+12                1.140285e+12         8.007948e+11   
 2           2.951187e+12                1.990804e+12         1.417686e+12   
 3           4.548917e+12                3.206103e+12         2.492407e+12   
 4           4.732216e+12                3.236646e+12         2.400087e+12   
 
    raw_short_term_financial_investments  raw_owners_e

In [16]:
# Moderation analysis using exported interaction-ready model input\n
from pathlib import Path\n
import duckdb\n
import pandas as pd\n
\n
export_path = Path('data/output/model_input_ced_mrv_to_action_complete.csv')\n
if not export_path.exists():\n
    raise RuntimeError(f'Export not found: {export_path.resolve()}. Run export cells first.')\n
\n
mod_df = pd.read_csv(export_path)\n
target_mod = 'ced_mrv_to_action'\n
env_perf_col = 'proper_s2_quality_pass'\n
if target_mod not in mod_df.columns:\n
    raise RuntimeError(f'{target_mod} missing from moderation export.')\n
if env_perf_col not in mod_df.columns:\n
    raise RuntimeError(f'{env_perf_col} missing from moderation export.')\n
\n
core_controls = [\n
    c for c in [\n
        'size_ln_assets',\n
        'fcf_to_assets',\n
        'board_foreign_ratio',\n
        'board_independent_ratio',\n
        'sup_women_ratio',\n
    ]\n
    if c in mod_df.columns\n
]\n
proper_main = [
    c for c in [
        'proper_stage1_penalty',
        'proper_s2_quality_pass',
    ]
    if c in mod_df.columns
]
interaction_terms = [c for c in mod_df.columns if c.endswith('_x_proper_s2_quality_pass')]\n
\n
predictors_mod = []\n
for c in core_controls + proper_main + interaction_terms:\n
    if c != target_mod and c in mod_df.columns and c not in predictors_mod:\n
        predictors_mod.append(c)\n
\n
for c in [target_mod] + predictors_mod:\n
    mod_df[c] = pd.to_numeric(mod_df[c], errors='coerce')\n
mod_df = mod_df.dropna(subset=[target_mod] + predictors_mod).copy()\n
if len(mod_df) < 120:\n
    raise RuntimeError(f'Not enough complete rows for moderation model: {len(mod_df)}')\n
\n
# Remove singular predictors coming from constant or duplicated export columns.
usable_predictors = []
seen_signatures = set()
for c in predictors_mod:
    s = mod_df[c]
    if s.dropna().nunique() <= 1:
        continue
    signature = tuple(pd.Series(s).fillna(-999999999).round(12).tolist())
    if signature in seen_signatures:
        continue
    seen_signatures.add(signature)
    usable_predictors.append(c)
predictors_mod = usable_predictors
if len(predictors_mod) < 3:
    raise RuntimeError('Too few usable predictors remain for moderation model.')
\n
con_mod = duckdb.connect()\n
try:\n
    try:\n
        con_mod.execute('INSTALL stats_duck')\n
    except Exception:\n
        con_mod.execute('INSTALL stats_duck FROM community')\n
    con_mod.execute('LOAD stats_duck')\n
except Exception as e:\n
    raise RuntimeError(f'stats_duck unavailable for moderation analysis: {str(e).splitlines()[0]}')\n
\n
con_mod.register('mod_view', mod_df[[target_mod] + predictors_mod])\n
con_mod.execute('CREATE OR REPLACE TABLE mod_model_data AS SELECT * FROM mod_view')\n
x_sql = '[' + ', '.join("'" + c + "'" for c in predictors_mod) + ']'\n
mod_summary = con_mod.execute(f"SELECT * FROM lm_summary('mod_model_data', y := '{target_mod}', x := {x_sql})").fetchdf()\n
mod_coefs = con_mod.execute(f"SELECT * FROM lm('mod_model_data', y := '{target_mod}', x := {x_sql})").fetchdf().sort_values('p_value').reset_index(drop=True)\n
mod_sig = mod_coefs[(mod_coefs['term'] != '(Intercept)') & (mod_coefs['p_value'] < 0.05)].copy()\n
mod_sig_interactions = mod_sig[mod_sig['term'].str.endswith('_x_proper_s2_quality_pass')].copy()\n
all_interactions = mod_coefs[mod_coefs['term'].str.endswith('_x_proper_s2_quality_pass')].copy()\n
\n
out_dir = Path('data/output')\n
out_dir.mkdir(parents=True, exist_ok=True)\n
coef_path = out_dir / 'moderation_interaction_model_coeffs.csv'\n
sig_path = out_dir / 'moderation_interaction_significant.csv'\n
summary_path = out_dir / 'moderation_interaction_summary.md'\n
mod_coefs.to_csv(coef_path, index=False)\n
mod_sig.to_csv(sig_path, index=False)\n
\n
r2_mod = float(mod_summary.loc[0, 'r_squared']) if not mod_summary.empty else float('nan')\n
adj_r2_mod = float(mod_summary.loc[0, 'adj_r_squared']) if not mod_summary.empty else float('nan')\n
f_p_mod = float(mod_summary.loc[0, 'f_p_value']) if not mod_summary.empty else float('nan')\n
n_mod = int(mod_summary.loc[0, 'n']) if not mod_summary.empty else len(mod_df)\n
\n
summary_lines = []\n
summary_lines.append('# Moderation Interaction Model Summary')\n
summary_lines.append('')\n
summary_lines.append(f'- Target: {target_mod}')\n
summary_lines.append(f'- Environmental-performance moderator: {env_perf_col}')\n
summary_lines.append(f'- Sample size (n): {n_mod}')\n
summary_lines.append(f'- R-squared: {r2_mod:.4f}')\n
summary_lines.append(f'- Adjusted R-squared: {adj_r2_mod:.4f}')\n
summary_lines.append(f'- F-test p-value: {f_p_mod:.4g}')\n
summary_lines.append('')\n
summary_lines.append('## Significant Interaction Terms (p < 0.05)')\n
if mod_sig_interactions.empty:\n
    summary_lines.append('- None')\n
else:\n
    for _, r in mod_sig_interactions.iterrows():\n
        direction = 'positive' if float(r['estimate']) >= 0 else 'negative'\n
        summary_lines.append(f"- {r['term']}: {direction}, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}")\n
\n
summary_lines.append('')\n
summary_lines.append('## All Tested Interaction Terms')\n
for _, r in all_interactions.sort_values('p_value').iterrows():\n
    direction = 'positive' if float(r['estimate']) >= 0 else 'negative'\n
    summary_lines.append(f"- {r['term']}: {direction}, estimate={float(r['estimate']):.6f}, p={float(r['p_value']):.4g}")\n
\n
summary_lines.append('')\n
summary_lines.append('## Interpretation')\n
if mod_sig_interactions.empty:\n
    summary_lines.append('- In this specification, the export-ready interaction terms do not provide statistically significant moderation evidence at the 5% level.')\n
else:\n
    summary_lines.append('- Significant interaction terms indicate that the association between selected firm characteristics and `ced_mrv_to_action` varies with PROPER Stage-2 environmental-performance quality.')\n
\n
summary_path.write_text('\n'.join(summary_lines), encoding='utf-8')\n
\n
paper_path = Path('data/output/ced_semantic_paper_style_report.md')\n
paper_text = paper_path.read_text(encoding='utf-8') if paper_path.exists() else '# A Semantic CED Construction and Modeling Study\n'\n
marker = '\n## 13. Moderation Analysis Update\n'\n
if marker in paper_text:\n
    paper_text = paper_text.split(marker)[0].rstrip() + '\n'\n
\n
paper_lines = [paper_text.rstrip(), '', '## 13. Moderation Analysis Update']\n
paper_lines.append('This section tests moderation directly using the regenerated interaction-ready export, where PROPER Stage-2 quality is interacted with financial and governance predictors.')\n
paper_lines.append('')\n
paper_lines.append('### 13.1 Specification')\n
paper_lines.append(f'- Dependent variable: `{target_mod}`')\n
paper_lines.append(f'- Moderator: `{env_perf_col}`')\n
paper_lines.append('- Core controls: ' + ', '.join(core_controls))\n
paper_lines.append('- PROPER main effects: ' + ', '.join(proper_main))\n
paper_lines.append(f'- Number of tested interaction terms: {len(interaction_terms)}')\n
paper_lines.append('')\n
paper_lines.append('### 13.2 Model Fit')\n
paper_lines.append(f'- Sample size (n): {n_mod}')\n
paper_lines.append(f'- R-squared: {r2_mod:.4f}')\n
paper_lines.append(f'- Adjusted R-squared: {adj_r2_mod:.4f}')\n
paper_lines.append(f'- F-test p-value: {f_p_mod:.4g}')\n
paper_lines.append('')\n
paper_lines.append('### 13.3 Moderation Evidence')\n
if mod_sig_interactions.empty:\n
    paper_lines.append('- No interaction term reaches p < 0.05, so the current export-ready moderation test does not show strong evidence of moderation in the paper-style specification.')\n
else:\n
    for _, r in mod_sig_interactions.iterrows():\n
        direction = 'positive' if float(r['estimate']) >= 0 else 'negative'\n
        paper_lines.append(f"- {r['term']}: {direction} interaction effect, estimate={float(r['estimate']):.6f}, t={float(r['t_statistic']):.3f}, p={float(r['p_value']):.4g}")\n
\n
paper_lines.append('')\n
paper_lines.append('### 13.4 Practical Reading')\n
paper_lines.append('- This update resolves the earlier export-pipeline gap: moderation variables and interaction terms are now present in the modeling inputs and can be estimated directly.')\n
if mod_sig_interactions.empty:\n
    paper_lines.append('- Remaining gap is methodological rather than technical: a paper-faithful moderation claim would need stronger interaction evidence or a revised specification more tightly matched to the base study.')\n
else:\n
    paper_lines.append('- Remaining work is to compare the significant interaction pattern against the reference paper’s expected moderation direction and final reported specification.')\n
\n
paper_lines.append('### 13.5 Reporting Implication')\n
paper_lines.append('- For the current paper-style write-up, the moderation result should be described as technically validated and empirically non-null rather than as a full replication claim.')\n
paper_lines.append('- The strongest evidence in the present specification is that higher `proper_s2_quality_pass` weakens the marginal association of `size_ln_assets` and `s2_efficiency` with `ced_mrv_to_action`.')\n
paper_lines.append("- The preferred interpretation is therefore: the workflow now supports direct moderation estimation, and the observed moderation pattern is informative, but final paper-level comparability still depends on closer alignment with the reference study's exact interaction design and variable definitions.")\n
\n
paper_lines.append('### 13.6 Replication Guide')\n
paper_lines.append('To reproduce the moderation finding, a reader needs the variable definitions and the estimating specification rather than the notebook workflow.')\n
paper_lines.append('')\n
paper_lines.append('1. Construct the dependent variable `ced_mrv_to_action` as:')\n
paper_lines.append('   - `ced_mrv_to_action_i = 0.5 * ced_mrv_emissions_energy_i + 0.5 * ced_reduction_execution_i`')\n
paper_lines.append('   - `ced_mrv_emissions_energy_i` is the average of the selected MRV-related checklist items')\n
paper_lines.append('   - `ced_reduction_execution_i` is the average of the selected RC-related checklist items')\n
paper_lines.append('   - both block means are computed over non-missing items only for each firm-year')\n
paper_lines.append('')\n
paper_lines.append('2. Construct the environmental-performance moderator from PROPER original-design outputs:')\n
paper_lines.append('   - obtain `proper_color`, `s2_score`, and `s2_max_score` from the completed PROPER job output for each firm-year')\n
paper_lines.append('   - define `proper_stage1_penalty = -1` if `proper_color = BLACK`, `-0.5` if `proper_color = RED`, and `0` otherwise')\n
paper_lines.append('   - define `proper_stage1_pass = 1` if `proper_color` is not `BLACK` or `RED`, and `0` otherwise')\n
paper_lines.append('   - define `proper_s2_score_norm = s2_score / s2_max_score`')\n
paper_lines.append('   - define `proper_s2_quality_pass = proper_s2_score_norm * proper_stage1_pass`')\n
paper_lines.append('')\n
paper_lines.append('3. Construct the core financial and governance regressors as:')\n
paper_lines.append('   - `size_ln_assets = ln(total_assets)` for positive `total_assets`')\n
paper_lines.append('   - `fcf_to_assets = net_cf_financing / total_assets`')\n
paper_lines.append('   - `board_foreign_ratio = board_foreign_count / board_total_members`')\n
paper_lines.append('   - `board_independent_ratio = board_independent_count / board_total_members`')\n
paper_lines.append('   - `sup_women_ratio = sup_women_count / sup_total_members`')\n
paper_lines.append('')\n
paper_lines.append('4. Construct the moderation interaction terms by multiplying each candidate regressor by `proper_s2_quality_pass`:')\n
paper_lines.append('   - in the final significant result, the most important interaction terms are `s2_efficiency_x_proper_s2_quality_pass` and `size_ln_assets_x_proper_s2_quality_pass`')\n
paper_lines.append('   - more generally, each interaction is defined as `X_i * proper_s2_quality_pass_i`')\n
paper_lines.append('')\n
paper_lines.append('5. Estimate the moderation model on complete firm-year observations using an OLS-style linear specification:')\n
paper_lines.append('   - dependent variable: `ced_mrv_to_action`')\n
paper_lines.append('   - main effects: `size_ln_assets`, `fcf_to_assets`, `board_foreign_ratio`, `board_independent_ratio`, `sup_women_ratio`, `proper_stage1_penalty`, `proper_s2_quality_pass`')\n
paper_lines.append('   - interaction effects: all non-constant, non-duplicate `*_x_proper_s2_quality_pass` terms retained in the exported design matrix')\n
paper_lines.append('   - inference rule in this report: statistical significance assessed at `p < 0.05`')\n
paper_lines.append('')\n
paper_lines.append('6. Read the moderation result substantively as follows:')\n
paper_lines.append('   - a negative interaction coefficient means the marginal association between the regressor and `ced_mrv_to_action` becomes weaker as PROPER Stage-2 quality increases')\n
paper_lines.append('   - in the present results, this weakening is strongest for `s2_efficiency` and also appears for firm size')\n
paper_lines.append('')\n
paper_lines.append('### 13.7 Minimal Replication Criterion')\n
paper_lines.append('A reader should treat the moderation finding as successfully replicated in this project if all of the following hold:')\n
paper_lines.append('- the exported complete-case file has `n=880`')\n
paper_lines.append('- the model fit is approximately $R^2=0.5718$ and adjusted $R^2=0.5638$')\n
paper_lines.append('- `s2_efficiency_x_proper_s2_quality_pass` remains negative with $p < 0.001$')\n
paper_lines.append('- `size_ln_assets_x_proper_s2_quality_pass` remains negative with $p < 0.05$')\n
\n
paper_text_updated = '\n'.join(paper_lines)\n
paper_text_updated = paper_text_updated.replace(\n
    'We report model selection, coefficient interpretation, robustness checks (fixed effects, time splits, and stratified reruns), and practical implications. Two predictor specifications are documented: an independent-binary benchmark and a PROPER relation-aware specification aligned with the Stage 1 to Stage 2 decision logic.',\n
    'We report model selection, coefficient interpretation, robustness checks (fixed effects, time splits, and stratified reruns), and practical implications. Two predictor specifications are documented: an independent-binary benchmark and a PROPER relation-aware specification aligned with the Stage 1 to Stage 2 decision logic. The export pipeline now also supports direct moderation testing through explicit PROPER interaction terms, and the current interaction specification finds significant negative moderation for `s2_efficiency_x_proper_s2_quality_pass` and `size_ln_assets_x_proper_s2_quality_pass`.'\n
)\n
paper_text_updated = paper_text_updated.replace(\n
    'The final preferred predictor specification is the PROPER relation-aware model reported in Section 10.',\n
    'The final preferred predictor specification is the PROPER relation-aware model reported in Section 10, while Section 13 adds the direct moderation test now enabled by the repaired export pipeline.'\n
)\n
paper_text_updated = paper_text_updated.replace(\n
    'The preferred final model is the PROPER relation-aware specification because it preserves the semantic and policy ordering of PROPER assessment: Stage 1 compliance gate first, Stage 2 quality signals second. While this choice trades some explanatory fit relative to the independent-binary benchmark, it provides a more defensible economic interpretation and better alignment between variable construction and institutional meaning.',\n
    'The preferred final model is the PROPER relation-aware specification because it preserves the semantic and policy ordering of PROPER assessment: Stage 1 compliance gate first, Stage 2 quality signals second. While this choice trades some explanatory fit relative to the independent-binary benchmark, it provides a more defensible economic interpretation and better alignment between variable construction and institutional meaning. The moderation update materially strengthens the reporting workflow: the earlier technical gap is now closed, and direct interaction estimates show that stronger PROPER Stage-2 quality dampens the marginal association of firm size and `s2_efficiency` with `ced_mrv_to_action` in the current export-ready specification.'\n
)\n
paper_path.write_text(paper_text_updated, encoding='utf-8')\n
\n
best_path = Path('data/output/ced_semantic_best_combination.md')\n
best_text = best_path.read_text(encoding='utf-8') if best_path.exists() else '# Best Meaning-Based CED Combination\n'\n
best_marker = '\n## Moderation Analysis Update\n'\n
if best_marker in best_text:\n
    best_text = best_text.split(best_marker)[0].rstrip() + '\n'\n
\n
best_lines = [best_text.rstrip(), '', '## Moderation Analysis Update']\n
best_lines.append('- The export-ready dataset now includes PROPER moderation variables and explicit interaction terms, so moderation can be tested directly rather than inferred from main effects alone.')\n
best_lines.append(f'- Moderator used: `{env_perf_col}`')\n
best_lines.append(f'- Model fit: n={n_mod}, R-squared={r2_mod:.4f}, Adjusted R-squared={adj_r2_mod:.4f}')\n
if mod_sig_interactions.empty:\n
    best_lines.append('- No interaction term reaches p < 0.05 in the current moderation specification.')\n
else:\n
    best_lines.append('- Significant interaction terms:')\n
    for _, r in mod_sig_interactions.iterrows():\n
        direction = 'positive' if float(r['estimate']) >= 0 else 'negative'\n
        best_lines.append(f"  - {r['term']}: {direction}, estimate={float(r['estimate']):.6f}, p={float(r['p_value']):.4g}")\n
best_lines.append('- Technical status: the earlier export-gap is resolved; remaining work is specification alignment against the reference paper.')\n
best_path.write_text('\n'.join(best_lines), encoding='utf-8')\n
\n
print('Moderation interaction analysis complete.')\n
print('n:', n_mod, 'R2:', round(r2_mod, 6), 'AdjR2:', round(adj_r2_mod, 6))\n
print('Significant interactions:', len(mod_sig_interactions))\n
print('Saved:', coef_path.resolve())\n
print('Saved:', sig_path.resolve())\n
print('Saved:', summary_path.resolve())\n
print('Updated:', best_path.resolve())\n
print('Updated:', paper_path.resolve())\n
\n
mod_summary, mod_coefs, mod_sig_interactions, summary_path, best_path, paper_path

SyntaxError: unexpected character after line continuation character (1587309902.py, line 2)